In [ ]:
!pip install catboost

In [ ]:
import pandas as pd
import numpy as np
import pickle
import xgboost as xgb
import torch
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
import warnings
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
df = pd.read_csv('/content/drive/MyDrive/CSV/train.csv')

In [ ]:
df.columns

In [ ]:
df.head()

# FINALE

In [ ]:
import pandas as pd
import numpy as np

# Ensure the DataFrame is sorted by 'period' for chronological feature engineering
# The notebook state indicates df is already sorted, but re-sorting for robustness.
df = df.sort_values(by='period').reset_index(drop=True)

print("DataFrame sorted by 'period'.")

# --- Feature Engineering: General for all transactions ---

# 1. Cumulative transaction counts per operation type per account
df['origin_operation_cumcount'] = df.groupby(['origin_account', 'operation']).cumcount() + 1
df['destination_operation_cumcount'] = df.groupby(['destination_account', 'operation']).cumcount() + 1

# 2. Cumulative total transaction counts per account
df['origin_total_cumcount'] = df.groupby('origin_account').cumcount() + 1
df['destination_total_cumcount'] = df.groupby('destination_account').cumcount() + 1

# 3. Balance after previous transaction (shifted) and their difference
df['origin_prev_balance_after'] = df.groupby('origin_account')['origin_balance_after'].shift(1).fillna(df['origin_balance_before'])
df['destination_prev_balance_after'] = df.groupby('destination_account')['destination_balance_after'].shift(1).fillna(df['destination_balance_before'])

df['origin_balance_before_vs_prev_after_diff'] = df['origin_balance_before'] - df['origin_prev_balance_after']
df['destination_balance_before_vs_prev_after_diff'] = df['destination_balance_before'] - df['destination_prev_balance_after']

# 4. Duration since the last operation (general and per operation type)
df['origin_time_since_last_txn'] = df.groupby('origin_account')['period'].diff().fillna(0)
df['destination_time_since_last_txn'] = df.groupby('destination_account')['period'].diff().fillna(0)

df['origin_time_since_last_op_type'] = df.groupby(['origin_account', 'operation'])['period'].diff().fillna(0)
df['destination_time_since_last_op_type'] = df.groupby(['destination_account', 'operation'])['period'].diff().fillna(0)

print("General features engineered (cumulative counts, shifted balances, time differences).")

# --- Feature Engineering: Geometric Distribution for 'op_03' only ---

# Helper function to get fraud metrics for an account type
def get_fraud_time_metrics(account_col, df_data):
    # Filter to only consider accounts that have ever committed fraud
    fraudulent_accounts = df_data[df_data['fraud_flag'] == 1][account_col].unique()

    # Pre-calculate first appearance and first fraud period for these accounts
    first_appearance = df_data.groupby(account_col)['period'].min()
    first_fraud = df_data[df_data['fraud_flag'] == 1].groupby(account_col)['period'].min()

    # Calculate time to first fraud
    time_to_first_fraud = (first_fraud - first_appearance).fillna(-1) # Use -1 for non-fraudulent accounts or those without first fraud

    # Calculate number of transactions until first fraud
    num_txns_until_first_fraud = {}
    for account in fraudulent_accounts:
        account_txns = df_data[df_data[account_col] == account]

        # If the account has fraud, identify its first fraud period
        if account in first_fraud.index:
            f_fraud_period = first_fraud.loc[account]

            # Count transactions from first appearance up to (and including) first fraud
            txns_slice = account_txns[account_txns['period'] <= f_fraud_period]
            num_txns_until_first_fraud[account] = len(txns_slice)
        else:
            num_txns_until_first_fraud[account] = -1 # Should not happen for 'fraudulent_accounts'

    time_to_first_fraud_series = time_to_first_fraud[fraudulent_accounts] # Filter to only fraudulent accounts
    num_txns_until_first_fraud_series = pd.Series(num_txns_until_first_fraud, index=fraudulent_accounts)

    return time_to_first_fraud_series, num_txns_until_first_fraud_series

# Calculate fraud metrics for origin and destination accounts
origin_time_to_first_fraud, origin_num_txns_to_first_fraud = get_fraud_time_metrics('origin_account', df)
destination_time_to_first_fraud, destination_num_txns_to_first_fraud = get_fraud_time_metrics('destination_account', df)

# Merge these metrics back to the main DataFrame
df['origin_time_to_first_fraud'] = df['origin_account'].map(origin_time_to_first_fraud).fillna(-1)
df['origin_num_txns_to_first_fraud'] = df['origin_account'].map(origin_num_txns_to_first_fraud).fillna(-1)
df['destination_time_to_first_fraud'] = df['destination_account'].map(destination_time_to_first_fraud).fillna(-1)
df['destination_num_txns_to_first_fraud'] = df['destination_account'].map(destination_num_txns_to_first_fraud).fillna(-1)

# Calculate geometric probability features (only for 'op_03')
def calculate_p_geometric(value):
    return 1 / (value + 1) if value >= 0 else -1 # Handle -1 placeholder

df['p_geometric_origin_period'] = df['origin_time_to_first_fraud'].apply(calculate_p_geometric)
df['p_geometric_origin_txns'] = df['origin_num_txns_to_first_fraud'].apply(calculate_p_geometric)
df['p_geometric_destination_period'] = df['destination_time_to_first_fraud'].apply(calculate_p_geometric)
df['p_geometric_destination_txns'] = df['destination_num_txns_to_first_fraud'].apply(calculate_p_geometric)

# Apply the geometric features only to 'op_03' transactions, set -1 otherwise
for col in ['p_geometric_origin_period', 'p_geometric_origin_txns',
            'p_geometric_destination_period', 'p_geometric_destination_txns']:
    df.loc[df['operation'] != 'op_03', col] = -1

print("Geometric probability features engineered for 'op_03' transactions.")

print("New features have been added to the DataFrame.")
print(df.head())

In [ ]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, roc_auc_score, confusion_matrix, classification_report, average_precision_score
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings('ignore', category=UserWarning, module='xgboost')

print("--- Building Fraud Prediction Model for 'op_03' Transactions ---")

# Filter the main DataFrame for 'op_03' operations
df_op03 = df[df['operation'] == 'op_03'].copy()

# Ensure the DataFrame is sorted by 'period' for chronological feature engineering
df_op03 = df_op03.sort_values(by='period').reset_index(drop=True)
print(f"DataFrame filtered for 'op_03' transactions. Shape: {df_op03.shape}")

# --- Re-implementing essential feature engineering for 'op_03' context ---
# These features might not be present in df_op03 depending on execution order,
# so they are re-calculated for robustness within this specific cell's scope.

# 1. Sequential transaction features (from naZiIj_ryjNw, but applied to df_op03)
df_op03['origin_transaction_sequence'] = df_op03.groupby('origin_account').cumcount() + 1
df_op03['destination_transaction_sequence'] = df_op03.groupby('destination_account').cumcount() + 1
df_op03['origin_dest_pair_sequence'] = df_op03.groupby(['origin_account', 'destination_account']).cumcount() + 1

# 2. Previously fraudulent flags (from cZwG6UBZMH-W, nkv6k2TX5OPZ)
# These flags should be based on the full historical data, so we re-derive them from the original 'df'
# assuming 'df' contains the full dataset up to this point from previous cells.
# If 'df' has already been filtered, this might be inaccurate. For safety, derive from current 'df'.
fraudulent_origin_accounts = df[df['fraud_flag'] == 1]['origin_account'].unique()
fraudulent_destination_accounts = df[df['fraud_flag'] == 1]['destination_account'].unique()
all_fraudulent_accounts = set(fraudulent_origin_accounts).union(set(fraudulent_destination_accounts))

df_op03['origin_account_previously_fraud'] = df_op03['origin_account'].isin(all_fraudulent_accounts).astype(int)
df_op03['destination_account_previously_fraud'] = df_op03['destination_account'].isin(all_fraudulent_accounts).astype(int)

print("Essential feature engineering completed for df_op03.")

# Define the target variable
y = df_op03['fraud_flag']

# Define features to be used for the model
# Exclude 'id', 'operation', 'origin_account', 'destination_account', and the target 'fraud_flag'
features_to_exclude = [
    'id', 'operation', 'origin_account', 'destination_account', 'fraud_flag'
]

# Collect all potential features present in df_op03.
# Use a broad set of features from previous successful XGBoost training where possible.
# The kernel state for 'df' at this cell had 31 columns from oLUqjteeSl2K, plus new ones here.
# This list includes features added in the re-implemented steps above.
all_possible_features = [
    'period', 'amount',
    'origin_balance_before', 'origin_balance_after',
    'destination_balance_before', 'destination_balance_after',
    'origin_operation_cumcount', 'destination_operation_cumcount',
    'origin_total_cumcount', 'destination_total_cumcount',
    'origin_prev_balance_after', 'destination_prev_balance_after',
    'origin_balance_before_vs_prev_after_diff', 'destination_balance_before_vs_prev_after_diff',
    'origin_time_since_last_txn', 'destination_time_since_last_txn',
    'origin_time_since_last_op_type', 'destination_time_since_last_op_type',
    'origin_time_to_first_fraud', 'origin_num_txns_to_first_fraud',
    'destination_time_to_first_fraud', 'destination_num_txns_to_first_fraud',
    'p_geometric_origin_period', 'p_geometric_origin_txns',
    'p_geometric_destination_period', 'p_geometric_destination_txns',
    'origin_transaction_sequence', 'destination_transaction_sequence',
    'origin_dest_pair_sequence',
    'origin_account_previously_fraud', 'destination_account_previously_fraud'
]

# Filter to include only features that are actually in df_op03 and are not excluded
features = [f for f in all_possible_features if f in df_op03.columns and f not in features_to_exclude]
X = df_op03[features]

# Fill any remaining NaNs in features, typically 0 for numerical features.
# This is a general safety net, specific imputation might be better for some features.
X = X.fillna(0)

print(f"Features used for prediction ({X.shape[1]}): {list(X.columns)}")
print(f"Shape of X: {X.shape}, Shape of y: {y.shape}")

if X.empty or len(y.unique()) < 2:
    print("Not enough data or classes to train the model.")
else:
    # Split data into training and testing sets
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

    # Calculate scale_pos_weight for imbalanced classes
    neg_count = y_train.value_counts()[0]
    pos_count = y_train.value_counts()[1]
    scale_pos_weight_value = neg_count / pos_count
    print(f"Calculated scale_pos_weight: {scale_pos_weight_value:.2f}")

    # Initialize and train the XGBoost classifier
    model_op03_fraud = xgb.XGBClassifier(
        objective='binary:logistic',
        eval_metric='aucpr',
        random_state=42,
        scale_pos_weight=scale_pos_weight_value,
        use_label_encoder=False # Suppress warning for deprecated parameter
    )
    model_op03_fraud.fit(X_train, y_train)

    # Make predictions on the test set
    y_pred = model_op03_fraud.predict(X_test)
    y_pred_proba = model_op03_fraud.predict_proba(X_test)[:, 1]

    # Evaluate the model
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    roc_auc = roc_auc_score(y_test, y_pred_proba)
    average_precision = average_precision_score(y_test, y_pred_proba)

    print(f"\n--- XGBoost Model Performance for 'op_03' Fraud Flag Prediction ---")
    print(f"Accuracy: {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall: {recall:.4f}")
    print(f"ROC AUC: {roc_auc:.4f}")
    print(f"Average Precision (PR-AUC): {average_precision:.4f}")
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred))

    # Confusion Matrix
    cm = confusion_matrix(y_test, y_pred)
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False,
                xticklabels=['Predicted Non-Fraud (0)', 'Predicted Fraud (1)'],
                yticklabels=['Actual Non-Fraud (0)', 'Actual Fraud (1)'])
    plt.title("Confusion Matrix for 'op_03' Fraud Prediction")
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.show()

    # Store the model and test data in globals for future use, if needed
    globals()['model_op03_fraud'] = model_op03_fraud
    globals()['X_train_op03'] = X_train
    globals()['y_train_op03'] = y_train
    globals()['X_test_op03'] = X_test
    globals()['y_test_op03'] = y_test
    globals()['y_pred_proba_op03'] = y_pred_proba

In [ ]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, roc_auc_score, confusion_matrix, classification_report, average_precision_score
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings('ignore', category=UserWarning, module='xgboost')

print("--- Building Fraud Prediction Model for 'op_03' Transactions (using existing features) ---")

# Filter the main DataFrame for 'op_03' operations
# The 'df' DataFrame is assumed to already contain all engineered features from previous steps.
df_op03 = df[df['operation'] == 'op_03'].copy()

# Ensure the DataFrame is sorted by 'period'
df_op03 = df_op03.sort_values(by='period').reset_index(drop=True)
print(f"DataFrame filtered for 'op_03' transactions. Shape: {df_op03.shape}")

# Define the target variable
y = df_op03['fraud_flag']

# Define features to be used for the model
# Exclude 'id', 'operation', 'origin_account', 'destination_account', and the target 'fraud_flag'
features_to_exclude = [
    'id', 'operation', 'origin_account', 'destination_account', 'fraud_flag'
]

# Collect all potential features present in df_op03, assuming they were already engineered on 'df'.
# This list includes features added in prior steps across the notebook.
all_possible_features = [
    'period', 'amount',
    'origin_balance_before', 'origin_balance_after',
    'destination_balance_before', 'destination_balance_after',
    'origin_operation_cumcount', 'destination_operation_cumcount',
    'origin_total_cumcount', 'destination_total_cumcount',
    'origin_prev_balance_after', 'destination_prev_balance_after',
    'origin_balance_before_vs_prev_after_diff', 'destination_balance_before_vs_prev_after_diff',
    'origin_time_since_last_txn', 'destination_time_since_last_txn',
    'origin_time_since_last_op_type', 'destination_time_since_last_op_type',
    #'origin_time_to_first_fraud', 'origin_num_txns_to_first_fraud',
    #'destination_time_to_first_fraud', 'destination_num_txns_to_first_fraud',
    'p_geometric_origin_period', 'p_geometric_origin_txns',
    'p_geometric_destination_period', 'p_geometric_destination_txns',
    'origin_transaction_sequence', 'destination_transaction_sequence',
    'origin_dest_pair_sequence'
    #'origin_account_previously_fraud', 'destination_account_previously_fraud'
]

# Filter to include only features that are actually in df_op03 and are not excluded
features = [f for f in all_possible_features if f in df_op03.columns and f not in features_to_exclude]
X = df_op03[features]

# Fill any remaining NaNs in features, typically 0 for numerical features.
X = X.fillna(0)

print(f"Features used for prediction ({X.shape[1]}): {list(X.columns)}")
print(f"Shape of X: {X.shape}, Shape of y: {y.shape}")

if X.empty or len(y.unique()) < 2:
    print("Not enough data or classes to train the model.")
else:
    # Split data into training and testing sets
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

    # Calculate scale_pos_weight for imbalanced classes
    neg_count = y_train.value_counts()[0]
    pos_count = y_train.value_counts()[1]
    scale_pos_weight_value = neg_count / pos_count
    print(f"Calculated scale_pos_weight: {scale_pos_weight_value:.2f}")

    # Initialize and train the XGBoost classifier
    model_op03_fraud = xgb.XGBClassifier(
        objective='binary:logistic',
        eval_metric='aucpr',
        random_state=42,
        scale_pos_weight=scale_pos_weight_value,
        use_label_encoder=False # Suppress warning for deprecated parameter
    )
    model_op03_fraud.fit(X_train, y_train)

    # Make predictions on the test set
    y_pred = model_op03_fraud.predict(X_test)
    y_pred_proba = model_op03_fraud.predict_proba(X_test)[:, 1]

    # Evaluate the model
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    roc_auc = roc_auc_score(y_test, y_pred_proba)
    average_precision = average_precision_score(y_test, y_pred_proba)

    print(f"\n--- XGBoost Model Performance for 'op_03' Fraud Flag Prediction ---")
    print(f"Accuracy: {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall: {recall:.4f}")
    print(f"ROC AUC: {roc_auc:.4f}")
    print(f"Average Precision (PR-AUC): {average_precision:.4f}")
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred))

    # Confusion Matrix
    cm = confusion_matrix(y_test, y_pred)
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False,
                xticklabels=['Predicted Non-Fraud (0)', 'Predicted Fraud (1)'],
                yticklabels=['Actual Non-Fraud (0)', 'Actual Fraud (1)'])
    plt.title("Confusion Matrix for 'op_03' Fraud Prediction")
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.show()

    # Store the model and test data in globals for future use, if needed
    globals()['model_op03_fraud'] = model_op03_fraud
    globals()['X_train_op03'] = X_train
    globals()['y_train_op03'] = y_train
    globals()['X_test_op03'] = X_test
    globals()['y_test_op03'] = y_test
    globals()['y_pred_proba_op03'] = y_pred_proba

# old

In [ ]:
df_sorted = df.sort_values(by='period').reset_index(drop=True)

# 1. Create a column for the N-th transaction of each origin account
df_sorted['origin_transaction_sequence'] = df_sorted.groupby('origin_account').cumcount() + 1

# 2. Create a column for the N-th transaction of each destination account
df_sorted['destination_transaction_sequence'] = df_sorted.groupby('destination_account').cumcount() + 1

# 3. Create a column for the N-th encounter between a specific origin-destination pair
df_sorted['origin_dest_pair_sequence'] = df_sorted.groupby(['origin_account', 'destination_account']).cumcount() + 1

# Merge these new features back to the original df (if df was not already sorted)
# For simplicity, we can just update the main df if it's acceptable to sort it.
# Assuming it's fine to modify 'df' directly for these new features, or create a new 'df' variable.
# If 'df' needs to remain unsorted for other operations, these new columns can be merged by 'id'.

# Let's assign these to the original `df` after sorting, or merge if `df` cannot be modified directly.
# Assuming 'df' is the primary DataFrame and re-sorting it for feature generation is acceptable.
df = df_sorted

print("New sequential transaction features created:")
print(df[['id', 'period', 'origin_account', 'origin_transaction_sequence',
          'destination_account', 'destination_transaction_sequence',
          'origin_dest_pair_sequence']].head())


In [ ]:
import numpy as np

# Identify unique origin accounts involved in fraud
fraudulent_origin_accounts = df[df['fraud_flag'] == 1]['origin_account'].unique()

# Identify unique destination accounts involved in fraud
fraudulent_destination_accounts = df[df['fraud_flag'] == 1]['destination_account'].unique()

# Create a set of all accounts that have ever committed fraud (origin or destination)
all_fraudulent_accounts = set(fraudulent_origin_accounts).union(set(fraudulent_destination_accounts))

# Create new columns for 'origin_account_previously_fraud' and 'destination_account_previously_fraud'
df['origin_account_previously_fraud'] = df['origin_account'].isin(all_fraudulent_accounts).astype(int)
df['destination_account_previously_fraud'] = df['destination_account'].isin(all_fraudulent_accounts).astype(int)

print("New columns 'origin_account_previously_fraud' and 'destination_account_previously_fraud' created.")
print("First 5 rows with new columns:")
print(df[['origin_account', 'destination_account', 'fraud_flag', 'origin_account_previously_fraud', 'destination_account_previously_fraud']].head())

In [ ]:
df_op03 = df[df['operation'] == 'op_03'].copy()

unique_op03_dest_accounts = df_op03['destination_account'].unique()
num_unique_op03_dest_accounts = len(unique_op03_dest_accounts)

print(f"Nombre de comptes destinataires uniques dans les opérations de type 'op_03': {num_unique_op03_dest_accounts}")

# 'fraudulent_destination_accounts' is already defined from previous cells (e.g., LWfX9gw_2ayC)
# This set contains all destination accounts that have ever been involved in fraud.

# Filter for those unique op_03 destination accounts that are NOT in the 'fraudulent_destination_accounts' set
non_fraudulent_op03_dest_accounts = [account for account in unique_op03_dest_accounts if account not in fraudulent_destination_accounts]
num_non_fraudulent_op03_dest_accounts = len(non_fraudulent_op03_dest_accounts)

print(f"Parmi eux, le nombre de comptes destinataires qui n'ont jamais fraudé est: {num_non_fraudulent_op03_dest_accounts}")


In [ ]:
import pandas as pd

# Assuming df_op03 is already available and contains only 'op_03' transactions
# Assuming non_fraudulent_op03_dest_accounts and fraudulent_destination_accounts are also available

# 1. Count transactions for the 4593 destination accounts that have never been fraudulent in 'op_03'
transactions_non_fraud_op03_dest = df_op03[
df_op03['destination_account'].isin(non_fraudulent_op03_dest_accounts)
]
num_transactions_non_fraud_op03_dest = len(transactions_non_fraud_op03_dest)

print(f"Nombre de transactions 'op_03' impliquant les 4593 comptes destinataires n'ayant jamais été frauduleux : {num_transactions_non_fraud_op03_dest}")

# 2. Count transactions for destination accounts that have been fraudulent at least once
transactions_fraud_op03_dest = df_op03[
df_op03['destination_account'].isin(fraudulent_destination_accounts)
]
num_transactions_fraud_op03_dest = len(transactions_fraud_op03_dest)

print(f"Nombre de transactions 'op_03' impliquant les comptes destinataires ayant été frauduleux au moins une fois : {num_transactions_fraud_op03_dest}")

# 3. Analyze the distribution within these groups (e.g., fraud_flag distribution)
print("\nRépartition de 'fraud_flag' pour les transactions des comptes destinataires non-frauduleux (théoriquement tous 0) :")
print(transactions_non_fraud_op03_dest['fraud_flag'].value_counts(normalize=True))

print("\nRépartition de 'fraud_flag' pour les transactions des comptes destinataires frauduleux au moins une fois :")
print(transactions_fraud_op03_dest['fraud_flag'].value_counts(normalize=True))

In [ ]:
import pandas as pd

# fraudulent_destination_accounts is available from previous cells
# df is the main DataFrame, assumed to be sorted by period from earlier steps.

# Filter transactions for destination accounts that have been fraudulent at least once
df_fraud_dest_accounts_only = df[df['destination_account'].isin(fraudulent_destination_accounts)].copy()

# Sort by period to easily find the first appearance
df_fraud_dest_accounts_only = df_fraud_dest_accounts_only.sort_values(by=['destination_account', 'period'])

# Get the first transaction for each of these destination accounts
first_appearance_transactions = df_fraud_dest_accounts_only.groupby('destination_account').first().reset_index()

print("Première apparition des comptes destinataires ayant été frauduleux (avec leur fraud_flag à ce moment):")
print(first_appearance_transactions[['destination_account', 'period', 'fraud_flag']].head())

print(
    "\nDistribution du fraud_flag pour la première apparition de ces comptes destinataires (tous devraient être 0 si le premier signal est toujours non-frauduleux, mais un 1 indiquerait une fraude immédiate):"
)
print(first_appearance_transactions['fraud_flag'].value_counts(normalize=True))
print(first_appearance_transactions['fraud_flag'].value_counts())

# Optional: Further analyze if there are any fraud_flag=1 on first appearance
if (first_appearance_transactions['fraud_flag'] == 1).any():
    print(
        "\nComptes destinataires dont la première transaction enregistrée est déjà frauduleuse:"
    )
    print(first_appearance_transactions[first_appearance_transactions['fraud_flag'] == 1][['destination_account', 'period', 'fraud_flag']])
else:
    print(
        "\nAucun compte destinataire n'a été enregistré comme frauduleux dès sa première apparition."
    )


In [ ]:
import pandas as pd

# Ensure `first_appearance_transactions` is available (from previous cells like LXVtY2U4aNRU)
# and `df` is the main DataFrame.

# 1. Identify destination accounts that were non-fraudulent (flag=0) at their first appearance
# but have been fraudulent (flag=1) at least once overall.
# `first_appearance_transactions` already contains accounts that were eventually fraudulent.
# We just need to filter for those whose `fraud_flag` was 0 at their first appearance.

initial_non_fraudulent_then_fraud_accounts = first_appearance_transactions[
    first_appearance_transactions['fraud_flag'] == 0
]['destination_account'].unique()

print(f"Number of destination accounts that were initially non-fraudulent then became fraudulent: {len(initial_non_fraudulent_then_fraud_accounts)}")

# 2. Filter the original `df` to get all transactions for these specific accounts.
df_dest_nonfraud_then_fraud = df[
    df['destination_account'].isin(initial_non_fraudulent_then_fraud_accounts)
].copy()

print(f"Created `df_dest_nonfraud_then_fraud` with shape: {df_dest_nonfraud_then_fraud.shape}")

print("First 5 rows of the new dataset:")
print(df_dest_nonfraud_then_fraud.head())

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Ensure fraudulent_destination_accounts is available from previous cells (e.g., LWfX9gw_2ayC)
# If not, recreate it (assuming df is the main DataFrame and up-to-date)
if 'fraudulent_destination_accounts' not in locals():
    fraudulent_destination_accounts = df[df['fraud_flag'] == 1]['destination_account'].unique()

# Filter the DataFrame to include only transactions involving destination accounts that have ever been fraudulent.
df_fraud_dest_txns = df[df['destination_account'].isin(fraudulent_destination_accounts)].copy()

# Prepare a list to store detailed analysis for each account
account_timeline_data = []

# Iterate through each unique fraudulent destination account
for account in fraudulent_destination_accounts:
    # Filter transactions for the current account and sort by period
    account_df = df_fraud_dest_txns[df_fraud_dest_txns['destination_account'] == account].sort_values(by='period').reset_index(drop=True)

    if account_df.empty:
        continue

    # Initialize for the first segment
    current_status = account_df['fraud_flag'].iloc[0]
    start_period = account_df['period'].iloc[0]
    segment_start_idx = 0

    # Iterate through transactions to find status changes
    for i in range(1, len(account_df)):
        if account_df['fraud_flag'].iloc[i] != current_status:
            # End of the previous segment
            end_period = account_df['period'].iloc[i-1]
            num_transactions = i - segment_start_idx
            segment_duration = end_period - start_period + 1 # Inclusive period count

            account_timeline_data.append({
                'account': account,
                'status': current_status, # 0 for non-fraud, 1 for fraud
                'start_period': start_period,
                'end_period': end_period,
                'duration': segment_duration,
                'num_transactions': num_transactions
            })

            # Start a new segment
            current_status = account_df['fraud_flag'].iloc[i]
            start_period = account_df['period'].iloc[i]
            segment_start_idx = i

    # Add the last segment after the loop finishes
    end_period = account_df['period'].iloc[-1]
    num_transactions = len(account_df) - segment_start_idx
    segment_duration = end_period - start_period + 1

    account_timeline_data.append({
        'account': account,
        'status': current_status,
        'start_period': start_period,
        'end_period': end_period,
        'duration': segment_duration,
        'num_transactions': num_transactions
    })

# Convert the collected data into a DataFrame
timeline_df = pd.DataFrame(account_timeline_data)

print("\nAnalyse de la chronologie des statuts de fraude par compte destinataire:")
display(timeline_df.head())

# Summarize the findings for fraudulent (status=1) and non-fraudulent (status=0) segments
summary_stats = timeline_df.groupby('status').agg(
    mean_duration=('duration', 'mean'),
    std_duration=('duration', 'std'),
    min_duration=('duration', 'min'),
    max_duration=('duration', 'max'),
    mean_transactions=('num_transactions', 'mean'),
    std_transactions=('num_transactions', 'std'),
    min_transactions=('num_transactions', 'min'),
    max_transactions=('num_transactions', 'max'),
    num_segments=('account', 'count')
)

print("\nStatistiques récapitulatives des phases (frauduleuses vs. non-frauduleuses):")
display(summary_stats)

# Plot distributions for duration and number of transactions per segment
plt.figure(figsize=(16, 6))

plt.subplot(1, 2, 1)
sns.histplot(timeline_df[timeline_df['status'] == 0]['duration'], kde=True, color='blue', label='Non-Fraudulent Phases')
sns.histplot(timeline_df[timeline_df['status'] == 1]['duration'], kde=True, color='red', label='Fraudulent Phases')
plt.title('Distribution des Durées de Phase par Statut de Fraude')
plt.xlabel('Durée (Périodes)')
plt.ylabel('Fréquence')
plt.legend()

plt.subplot(1, 2, 2)
sns.histplot(timeline_df[timeline_df['status'] == 0]['num_transactions'], kde=True, color='blue', label='Non-Fraudulent Phases')
sns.histplot(timeline_df[timeline_df['status'] == 1]['num_transactions'], kde=True, color='red', label='Fraudulent Phases')
plt.title('Distribution du Nombre de Transactions par Statut de Fraude')
plt.xlabel('Nombre de Transactions')
plt.ylabel('Fréquence')
plt.legend()

plt.tight_layout()
plt.show()

# Additional insights: Number of status changes per account
# To calculate status changes, we look for where fraud_flag changes from one transaction to the next
# For this, we need the full `df` sorted by period and then grouped by account.

# Create a helper column to detect changes in fraud_flag within each account's timeline
relevant_transactions_sorted = df_fraud_dest_txns.sort_values(by=['destination_account', 'period'])
relevant_transactions_sorted['fraud_flag_diff'] = relevant_transactions_sorted.groupby('destination_account')['fraud_flag'].diff().fillna(0)

# Count the number of actual status changes (from 0 to 1 or 1 to 0)
# A non-zero diff indicates a change in status
num_status_changes_per_account = relevant_transactions_sorted[relevant_transactions_sorted['fraud_flag_diff'] != 0]
num_status_changes_per_account = num_status_changes_per_account.groupby('destination_account').size()

print("\nDistribution du nombre de changements de statut (Non-Fraude <-> Fraude) par compte:")
# Add accounts that never changed status (i.e., only one segment) and thus have 0 changes
all_accounts_in_relevant_txns = df_fraud_dest_txns['destination_account'].unique()
num_status_changes_per_account_full = pd.Series(0, index=all_accounts_in_relevant_txns).add(num_status_changes_per_account, fill_value=0).astype(int)

display(num_status_changes_per_account_full.describe())

# Visualizing the number of status changes
plt.figure(figsize=(10, 6))
# Use bins from 0 to max_changes + 1 to properly show counts for each integer change value
bins = np.arange(num_status_changes_per_account_full.max() + 2) - 0.5
sns.histplot(num_status_changes_per_account_full, bins=bins, kde=False)
plt.title('Nombre de Changements de Statut par Compte Destinataire Frauduleux')
plt.xlabel('Nombre de Changements de Statut')
plt.ylabel('Nombre de Comptes')
plt.xticks(np.arange(0, num_status_changes_per_account_full.max() + 1, 1))
plt.show()


#### model qui predit le premier flaque d'un compte untilisateur sa nature

In [ ]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, roc_auc_score, average_precision_score, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

print("--- Training XGBoost model on first appearance of eventually fraudulent destination accounts ---")

# Create a copy of the first_appearance_transactions DataFrame for this specific model
df_first_appearance_model = first_appearance_transactions.copy()

# Ensure the DataFrame is sorted by period to make cumulative calculations meaningful
# first_appearance_transactions is already sorted by destination_account, period
# Re-sorting by just period might reorder rows if periods are identical across different accounts.
# For cumulative features like expanding().nunique(), sorting by period is crucial.
df_first_appearance_model = df_first_appearance_model.sort_values(by='period').reset_index(drop=True)

# --- Start of new feature creation ---

# Helper function for cumulative unique counts on non-numeric series
def cumulative_nunique(series):
    unique_elements = set()
    counts = []
    for x in series:
        unique_elements.add(x)
        counts.append(len(unique_elements))
    return pd.Series(counts, index=series.index)

# 1. Count of transfers per operation type for each sender (cumulative)
# For each origin_account, and for each operation type, count how many times this origin_account has performed this operation
df_first_appearance_model['origin_op_cumulative_count'] = df_first_appearance_model.groupby(['origin_account', 'operation']).cumcount() + 1

# 2. Total number of unique accounts encountered by a given account (cumulative)
# For each origin_account, count unique destination_accounts it has sent to
df_first_appearance_model['origin_unique_dest_cumulative_count'] = (
    df_first_appearance_model.groupby('origin_account')['destination_account']
    .transform(cumulative_nunique)
)
# For each destination_account, count unique origin_accounts it has received from
# In this specific df (first appearance of unique destination accounts), this will largely be 1.
df_first_appearance_model['dest_unique_origin_cumulative_count'] = (
    df_first_appearance_model.groupby('destination_account')['origin_account']
    .transform(cumulative_nunique)
)

# 3. Time between a transaction (origin/destination) and the previous one for that account
# Time elapsed since the last transaction for the origin account
df_first_appearance_model['origin_time_since_last_txn'] = df_first_appearance_model.groupby('origin_account')['period'].diff().fillna(0)
# Time elapsed since the last transaction for the destination account
# This will mostly be 0 in this DataFrame as each destination account appears only once in its 'first appearance'.
df_first_appearance_model['dest_time_since_last_txn'] = df_first_appearance_model.groupby('destination_account')['period'].diff().fillna(0)


# 4. Duration since the last transaction of each type for a receiver
# Time elapsed since the last transaction of the same operation type for the destination account
# This will mostly be 0 in this DataFrame as each destination account appears only once in its 'first appearance'.
df_first_appearance_model['dest_op_time_since_last_txn'] = df_first_appearance_model.groupby(['destination_account', 'operation'])['period'].diff().fillna(0)

print("New features 'origin_op_cumulative_count', 'origin_unique_dest_cumulative_count', 'dest_unique_origin_cumulative_count', 'origin_time_since_last_txn', 'dest_time_since_last_txn', and 'dest_op_time_since_last_txn' created.")

# --- End of new feature creation ---

# Encode 'operation' using one-hot encoding
df_first_appearance_model = pd.get_dummies(df_first_appearance_model, columns=['operation'], prefix='operation', drop_first=True)

# Define target variable 'y'
y = df_first_appearance_model['fraud_flag']

# Define features 'X'
# Exclude 'id', 'destination_account', 'origin_account' and the target 'fraud_flag'
# Also exclude sequential features if they are constant (which they likely are for 'first' appearance)
# Check for constant sequential features before dropping them if they were included.
constant_seq_cols = []
for col in ['origin_transaction_sequence', 'destination_transaction_sequence', 'origin_dest_pair_sequence']:
    if col in df_first_appearance_model.columns and df_first_appearance_model[col].nunique() == 1:
        constant_seq_cols.append(col)

# Explicitly exclude 'origin_account' and 'destination_account' as they are object types
# Add the new features to the `features_to_exclude` list if they are found to be constant (e.g. 1 or 0)
# for the 'first_appearance_model', as they won't add predictive power.
# We need to dynamically check them for constancy.

for new_feature_col in ['dest_unique_origin_cumulative_count', 'dest_time_since_last_txn', 'dest_op_time_since_last_txn']:
    if new_feature_col in df_first_appearance_model.columns and df_first_appearance_model[new_feature_col].nunique() == 1:
        constant_seq_cols.append(new_feature_col)

features_to_exclude = ['id', 'destination_account', 'origin_account', 'fraud_flag'] + constant_seq_cols
X = df_first_appearance_model.drop(columns=features_to_exclude, errors='ignore')

# Fill any remaining NaNs in features, typically 0 for numerical features
X = X.fillna(0)

print(f"Features used for prediction ({X.shape[1]}): {list(X.columns)}")
print(f"Shape of X: {X.shape}, Shape of y: {y.shape}")

if X.empty or len(y.unique()) < 2:
    print("Not enough data or classes to train the model.")
else:
    # Split data into training and testing sets
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

    # Calculate scale_pos_weight for imbalanced classes
    neg_count = y_train.value_counts()[0]
    pos_count = y_train.value_counts()[1]
    scale_pos_weight_value = neg_count / pos_count
    print(f"Calculated scale_pos_weight: {scale_pos_weight_value:.2f}")

    # Initialize and train the XGBoost classifier
    model_first_appearance_fraud = xgb.XGBClassifier(
        objective='binary:logistic',
        eval_metric='aucpr',
        random_state=42,
        scale_pos_weight=scale_pos_weight_value,
        use_label_encoder=False # Suppress warning
    )
    model_first_appearance_fraud.fit(X_train, y_train)

    # Make predictions on the test set
    y_pred = model_first_appearance_fraud.predict(X_test)
    y_pred_proba = model_first_appearance_fraud.predict_proba(X_test)[:, 1]

    # Evaluate the model
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    roc_auc = roc_auc_score(y_test, y_pred_proba)
    average_precision = average_precision_score(y_test, y_pred_proba)

    print(f"\n--- XGBoost Model Performance for First Appearance Fraud Flag Prediction ---")
    print(f"Accuracy: {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall: {recall:.4f}")
    print(f"ROC AUC: {roc_auc:.4f}")
    print(f"Average Precision (PR-AUC): {average_precision:.4f}")

    # Confusion Matrix
    cm = confusion_matrix(y_test, y_pred)
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False,
                xticklabels=['Predicted Non-Fraud (0)', 'Predicted Fraud (1)'] ,
                yticklabels=['Actual Non-Fraud (0)', 'Actual Fraud (1)'])
    plt.title('Confusion Matrix for First Appearance Fraud Prediction')
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.show()

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# Ensure X and y are available from the previous model training
if 'X' in globals() and 'y' in globals():
    # Combine features and target for correlation calculation
    df_for_correlation = pd.concat([X, y], axis=1)

    # Calculate the correlation matrix
    correlation_matrix = df_for_correlation.corr()

    # Extract correlations with the target variable 'fraud_flag'
    fraud_correlation = correlation_matrix['fraud_flag'].sort_values(ascending=False)

    print("Correlation of features with 'fraud_flag' (descending order):\n")
    print(fraud_correlation)

    # Optional: Visualize the correlations with a heatmap (can be large for many features)
    plt.figure(figsize=(10, 8))
    sns.heatmap(correlation_matrix[['fraud_flag']].sort_values(by='fraud_flag', ascending=False), annot=True, cmap='coolwarm', fmt=".2f")
    plt.title('Feature Correlation with Fraud Flag')
    plt.show()

else:
    print("Error: 'X' or 'y' DataFrames not found. Please ensure the model training cell (e.g., 1jx1gqibbt41) was executed.")


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Assuming 'first_appearance_transactions' is available from the previous cell (LXVtY2U8aNRU)

print("Descriptive statistics for 'period' at first appearance for fraud_flag = 0:")
print(first_appearance_transactions[first_appearance_transactions['fraud_flag'] == 0]['period'].describe())

print("\nDescriptive statistics for 'period' at first appearance for fraud_flag = 1:")
print(first_appearance_transactions[first_appearance_transactions['fraud_flag'] == 1]['period'].describe())

# Visualize the distribution of 'period' for both fraud_flag categories
plt.figure(figsize=(10, 6))
sns.boxplot(x='fraud_flag', y='period', data=first_appearance_transactions)
plt.title('Distribution of Period at First Appearance by Fraud Flag')
plt.xlabel('Fraud Flag at First Appearance (0 = Non-Fraudulent, 1 = Fraudulent)')
plt.ylabel('Period')
plt.grid(True)
plt.show()

plt.figure(figsize=(10, 6))
sns.kdeplot(data=first_appearance_transactions, x='period', hue='fraud_flag', fill=True, common_norm=False)
plt.title('KDE Plot of Period at First Appearance by Fraud Flag')
plt.xlabel('Period')
plt.ylabel('Density')
plt.grid(True)
plt.show()

print("\nInterpretation:\n- By comparing the descriptive statistics and the box/KDE plots, we can observe if accounts that are initially flagged as fraudulent (fraud_flag=1) tend to appear earlier or later in the dataset's timeline ('period') compared to accounts that are initially non-fraudulent (fraud_flag=0) but later become fraudulent.")

### suite

In [ ]:
import pandas as pd

# Filter df_op03 for transactions involving non-fraudulent destination accounts
transactions_non_fraudulent_dest_op03 = df_op03[df_op03['destination_account'].isin(non_fraudulent_op03_dest_accounts)]

# Get the unique periods from these transactions
unique_periods_non_fraud_dest_op03 = sorted(transactions_non_fraudulent_dest_op03['period'].unique())

print("Periods in which 'op_03' transactions occur with never fraudulent destination accounts:")
print(unique_periods_non_fraud_dest_op03)

In [ ]:
import pandas as pd

# Ensure df_op03 is available, if not, create it from the main df
if 'df_op03' not in locals():
    df_op03 = df[df['operation'] == 'op_03'].copy()

# 1. DataFrame for 'op_03' transactions that are never fraudulent
# This utilizes the 'non_fraudulent_op03_dest_accounts' identified earlier
# These transactions will inherently have fraud_flag == 0, as confirmed by prior analysis.
df_never_fraudulent_op03 = df_op03[df_op03['destination_account'].isin(non_fraudulent_op03_dest_accounts)].copy()

# 2. DataFrame for 'op_03' transactions that are themselves flagged as fraudulent
# Based on previous analysis, no accounts were 'always' fraudulent in all their transactions,
# so this DataFrame captures individual fraudulent transactions within 'op_03'.
df_only_fraudulent_op03_txns = df_op03[df_op03['fraud_flag'] == 1].copy()

print("DataFrame 'df_never_fraudulent_op03' created (transactions with never-fraudulent destination accounts in op_03).")
print(f"Shape of df_never_fraudulent_op03: {df_never_fraudulent_op03.shape}")
print("First 5 rows of df_never_fraudulent_op03:")
print(df_never_fraudulent_op03.head())

print("\nDataFrame 'df_only_fraudulent_op03_txns' created (transactions with fraud_flag = 1 in op_03).")
print(f"Shape of df_only_fraudulent_op03_txns: {df_only_fraudulent_op03_txns.shape}")
print("First 5 rows of df_only_fraudulent_op03_txns:")
print(df_only_fraudulent_op03_txns.head())

In [ ]:
df_never_fraudulent_op03.tail()

In [ ]:
df_only_fraudulent_op03_txns.tail()

### petit detour

In [ ]:
import pandas as pd
import numpy as np

# --- Code to define destination_fraud_metrics (minimal version from HBN81jf0MoXt) ---
# Ensure df is sorted by period for correct 'first appearance' and 'first fraud' periods
df_sorted_by_period = df.sort_values(by='period').copy()

def get_fraud_time_metrics(account_type_col, df_data):
    fraudulent_accounts = df_data[df_data['fraud_flag'] == 1][account_type_col].unique()
    time_to_fraud_list = []
    for account in fraudulent_accounts:
        account_txns = df_data[df_data[account_type_col] == account]
        first_appearance_period = account_txns['period'].min()
        first_fraud_period = account_txns[account_txns['fraud_flag'] == 1]['period'].min()
        time_to_fraud = first_fraud_period - first_appearance_period
        time_to_fraud_list.append({
            'account': account,
            'first_appearance_period': first_appearance_period,
            'first_fraud_period': first_fraud_period,
            'time_to_fraud': time_to_fraud
        })
    return pd.DataFrame(time_to_fraud_list)

# Only process destination accounts, as origin_fraud_metrics is not used in the next step
destination_fraud_metrics = get_fraud_time_metrics('destination_account', df_sorted_by_period)
# --- End of minimal content from HBN81jf0MoXt ---


# Calculate the mean of 'time_to_fraud' for destination accounts
mean_time_to_fraud_dest = destination_fraud_metrics['time_to_fraud'].mean()

p_geometric = 1 / (mean_time_to_fraud_dest + 1)

print(f"The calculated parameter 'p' for the geometric distribution of 'time_to_fraud' in destination accounts is: {p_geometric:.4f}")

# --- Now, implementing the filtering/censoring strategy ---
# The user suggested: "Filter training data by censoring recent 'healthy' accounts,
# only keeping 'non-frauds' that have remained stable for a long time."

# 1. Identify all unique accounts in the dataset
all_unique_accounts = pd.concat([df['origin_account'], df['destination_account']]).unique()

# 2. Identify all accounts that have *ever* been involved in fraud (either as origin or destination)
# 'fraudulent_origin_accounts' and 'fraudulent_destination_accounts' are available from previous cells.
# Assuming these are available in the global scope from previous executions.
all_fraudulent_accounts_set = set(fraudulent_origin_accounts).union(set(fraudulent_destination_accounts))

# 3. Identify truly non-fraudulent accounts: those that are in all_unique_accounts but NOT in all_fraudulent_accounts
truly_non_fraudulent_accounts_set = set(all_unique_accounts) - all_fraudulent_accounts_set

# 4. For these truly non-fraudulent accounts, calculate their 'lifetime' in the dataset (max_period - min_period)
#    First, get the first and last appearance period for all accounts
first_appearance_all = df.groupby('destination_account')['period'].min().reset_index().rename(columns={'period': 'first_appearance_period'})
last_appearance_all = df.groupby('destination_account')['period'].max().reset_index().rename(columns={'period': 'last_appearance_period'})

account_lifetimes = pd.merge(first_appearance_all, last_appearance_all, on='destination_account', how='left')
account_lifetimes['account_lifetime'] = account_lifetimes['last_appearance_period'] - account_lifetimes['first_appearance_period']

# 5. Filter `account_lifetimes` to include only truly non-fraudulent destination accounts
truly_non_fraud_dest_lifetimes = account_lifetimes[account_lifetimes['destination_account'].isin(truly_non_fraudulent_accounts_set)].copy()

# 6. Define a threshold for "long time" based on the geometric distribution mean.
#    If mean_time_to_fraud is 2.19 periods, let's say "long time" is at least 3 times the mean, for robustness.
#    Alternatively, a fixed threshold like 5 or 10 periods could be used.
long_time_threshold_periods = int(mean_time_to_fraud_dest * 3) # Convert to int
if long_time_threshold_periods == 0: long_time_threshold_periods = 1 # Ensure at least 1 period
print(f"\nDefining 'long time' for stable accounts as >= {long_time_threshold_periods} periods of activity without fraud.")

# 7. Identify truly non-fraudulent destination accounts that have been 'stable for a long time'
stable_long_time_accounts_set = set(truly_non_fraud_dest_lifetimes[
    truly_non_fraud_dest_lifetimes['account_lifetime'] >= long_time_threshold_periods
]['destination_account'].tolist())

# 8. Create a new DataFrame `df_filtered_for_training` by applying the censoring strategy
#    Keep all fraudulent transactions (fraud_flag == 1).
#    Keep non-fraudulent transactions (fraud_flag == 0) ONLY if their destination_account
#    is among the `stable_long_time_accounts_set`.

df_filtered_for_training = df[
    (df['fraud_flag'] == 1) | # Keep all fraudulent transactions
    ((df['fraud_flag'] == 0) & (df['destination_account'].isin(stable_long_time_accounts_set))) # Keep non-fraudulent from stable accounts
].copy()

print(f"\nOriginal DataFrame shape: {df.shape}")
print(f"Shape of filtered DataFrame for training (after censoring recent healthy accounts): {df_filtered_for_training.shape}")
print(f"Number of transactions censored: {df.shape[0] - df_filtered_for_training.shape[0]}")

# Display first few rows of the filtered DataFrame for verification
print("\nFirst 5 rows of df_filtered_for_training:")
print(df_filtered_for_training.head())

In [ ]:
df_filtered_for_training['operation'].value_counts()


In [ ]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, roc_auc_score, confusion_matrix, average_precision_score
import matplotlib.pyplot as plt
import seaborn as sns
import datetime

# Make a working copy of the filtered data
df_train = df_filtered_for_training.copy()

# --- Helper Functions (re-defined for self-containment) ---
def clean_hex(account_id):
    if pd.isna(account_id):
        return ""
    cleaned = str(account_id).replace("acc_o_", "").replace("acc_d_", "").strip()
    return cleaned

def hex_to_decimal(hex_str):
    try:
        return int(hex_str, 16) if hex_str else None
    except ValueError:
        return None

def hex_to_ipv4(hex_str):
    try:
        if len(hex_str) == 16:
            hex_ip = hex_str[8:]
            return ".".join(str(int(hex_ip[i:i+2], 16)) for i in range(0, 8, 2))
    except Exception:
        pass
    return None

def hex_to_mac(hex_str):
    try:
        if len(hex_str) == 16:
            hex_mac = hex_str[4:]
            return ":".join(hex_mac[i:i+2].upper() for i in range(0, 12, 2))
    except Exception:
        pass
    return None

def hex_to_timestamp(hex_str):
    try:
        if hex_str:
            val_dec = int(hex_str, 16)
            ts_sec = val_dec / 1_000_000_000
            # Add a check to prevent very old or future dates that might be invalid
            if 0 < ts_sec < 4102444800: # Approx. Jan 1, 1970 to Jan 1, 2100
                return datetime.datetime.fromtimestamp(ts_sec, datetime.timezone.utc)
            else:
                return None
    except (ValueError, TypeError, OSError):
        pass
    return pd.NaT

def extract_decimal_features(value):
    if pd.isna(value):
        return False, 0, 0
    s = str(value)
    if 'e' in s or 'E' in s:
        s = f"{value:.10f}"
        s = s.rstrip('0').rstrip('.')
    has_decimal = '.' in s
    if has_decimal:
        parts = s.split('.')
        integer_part = parts[0]
        decimal_part = parts[1]
        num_decimal_places = len(decimal_part)
    else:
        integer_part = s
        num_decimal_places = 0
    num_digits_before_decimal = len(integer_part.lstrip('-'))
    return has_decimal, num_decimal_places, num_digits_before_decimal

def get_first_four_digits(value):
    if pd.isna(value):
        return [np.nan] * 4
    s = str(value)
    if 'e' in s or 'E' in s:
        s = f"{value:.10f}"
    s_digits = s.replace('.', '').lstrip('-')
    digits = []
    for i in range(4):
        if i < len(s_digits):
            digits.append(int(s_digits[i]))
        else:
            digits.append(np.nan)
    return digits

# --- Account Ranking (moved from a previous cell to ensure availability) ---
df_train['origin_account_stripped'] = df_train['origin_account'].str.replace('acc_o_', '')
df_train['destination_account_stripped'] = df_train['destination_account'].str.replace('acc_d_', '')

all_stripped_accounts = pd.concat([
    df_train['origin_account_stripped'],
    df_train['destination_account_stripped']
]).unique()

sorted_accounts = pd.Series(all_stripped_accounts).sort_values().reset_index(drop=True)

account_to_rank_mapping = {account_id: rank + 1 for rank, account_id in enumerate(sorted_accounts)}

# --- Apply Hex-related Feature Engineering to df_train ---
df_train['origin_hex_clean'] = df_train['origin_account'].apply(clean_hex)
df_train['destination_hex_clean'] = df_train['destination_account'].apply(clean_hex)

df_train['calc_origin_decimal']   = df_train['origin_hex_clean'].apply(hex_to_decimal)
df_train['calc_origin_ipv4']      = df_train['origin_hex_clean'].apply(hex_to_ipv4)
df_train['calc_origin_mac']       = df_train['origin_hex_clean'].apply(hex_to_mac)
df_train['calc_origin_timestamp'] = df_train['origin_hex_clean'].apply(hex_to_timestamp)

df_train['calc_destination_decimal']   = df_train['destination_hex_clean'].apply(hex_to_decimal)
df_train['calc_destination_ipv4']      = df_train['destination_hex_clean'].apply(hex_to_ipv4)
df_train['calc_destination_mac']       = df_train['destination_hex_clean'].apply(hex_to_mac)
df_train['calc_destination_timestamp'] = df_train['destination_hex_clean'].apply(hex_to_timestamp)

df_train.drop(columns=['origin_hex_clean', 'destination_hex_clean'], inplace=True, errors='ignore')

df_train['has_origin_timestamp'] = df_train['calc_origin_timestamp'].notna()
df_train['has_destination_timestamp'] = df_train['calc_destination_timestamp'].notna()

# --- Apply Account Ranking to df_train ---
# Use the existing account_to_rank_mapping from kernel state
df_train['origin_account_ranked'] = df_train['origin_account_stripped'].map(account_to_rank_mapping).fillna(0).astype(int)
df_train['destination_account_ranked'] = df_train['destination_account_stripped'].map(account_to_rank_mapping).fillna(0).astype(int)

# --- Apply Decimal/Digit Feature Engineering to df_train ---
numerical_cols = ['amount', 'origin_balance_before', 'origin_balance_after', 'destination_balance_before', 'destination_balance_after']
for col in numerical_cols:
    df_train[[f'{col}_has_decimal', f'{col}_num_decimal_places', f'{col}_num_digits_before_decimal']] = \
        df_train[col].apply(lambda x: pd.Series(extract_decimal_features(x)))
    df_train[f'{col}_has_decimal'] = df_train[f'{col}_has_decimal'].astype(bool)
    df_train[f'{col}_num_decimal_places'] = df_train[f'{col}_num_decimal_places'].astype(int)
    df_train[f'{col}_num_digits_before_decimal'] = df_train[f'{col}_num_digits_before_decimal'].astype(int)

df_train[['amount_digit1', 'amount_digit2', 'amount_digit3', 'amount_digit4']] = \
    df_train['amount'].apply(lambda x: pd.Series(get_first_four_digits(x)))
df_train[['amount_digit1', 'amount_digit2', 'amount_digit3', 'amount_digit4']] = \
    df_train[['amount_digit1', 'amount_digit2', 'amount_digit3', 'amount_digit4']].fillna(0).astype(int)

# --- Merge Fraud Pattern Features into df_train ---

# Ensure df is sorted by period for correct 'first appearance' and 'first fraud' periods
df_sorted_by_period = df.sort_values(by='period').copy()

def get_fraud_time_metrics(account_type_col, df_data):
    fraudulent_accounts = df_data[df_data['fraud_flag'] == 1][account_type_col].unique()
    time_to_fraud_list = []
    for account in fraudulent_accounts:
        account_txns = df_data[df_data[account_type_col] == account]
        first_appearance_period = account_txns['period'].min()
        first_fraud_period = account_txns[account_txns['fraud_flag'] == 1]['period'].min()
        time_to_fraud = first_fraud_period - first_appearance_period
        time_to_fraud_list.append({
            'account': account,
            'first_appearance_period': first_appearance_period,
            'first_fraud_period': first_fraud_period,
            'time_to_fraud': time_to_fraud
        })
    return pd.DataFrame(time_to_fraud_list)

destination_fraud_metrics = get_fraud_time_metrics('destination_account', df_sorted_by_period)

# Calculate 'number of transactions until first fraud' for destination accounts
num_txns_until_first_fraud_list = []

for index, row in destination_fraud_metrics.iterrows():
    account = row['account']
    first_appearance = row['first_appearance_period']
    first_fraud = row['first_fraud_period']

    account_txns_until_first_fraud = df_sorted_by_period[
        (df_sorted_by_period['destination_account'] == account) &
        (df_sorted_by_period['period'] >= first_appearance) &
        (df_sorted_by_period['period'] <= first_fraud)
    ]
    num_txns_until_first_fraud_list.append({
        'account': account,
        'num_transactions_until_first_fraud': len(account_txns_until_first_fraud)
    })

num_txns_until_first_fraud_df = pd.DataFrame(num_txns_until_first_fraud_list)

# Re-create df_fraud_dest_txns for timeline analysis
# Ensure 'fraudulent_destination_accounts' is available
if 'fraudulent_destination_accounts' not in locals():
    fraudulent_destination_accounts = df[df['fraud_flag'] == 1]['destination_account'].unique()

df_fraud_dest_txns = df[df['destination_account'].isin(fraudulent_destination_accounts)].copy()
relevant_transactions = df_fraud_dest_txns.copy()

account_timeline_data = []

for account in fraudulent_destination_accounts:
    account_df = relevant_transactions[relevant_transactions['destination_account'] == account].sort_values(by='period').reset_index(drop=True)

    if account_df.empty:
        continue

    current_status = account_df['fraud_flag'].iloc[0]
    start_period = account_df['period'].iloc[0]
    segment_start_idx = 0

    for i in range(1, len(account_df)):
        if account_df['fraud_flag'].iloc[i] != current_status:
            end_period = account_df['period'].iloc[i-1]
            num_transactions = i - segment_start_idx
            segment_duration = end_period - start_period + 1

            account_timeline_data.append({
                'account': account,
                'status': current_status,
                'start_period': start_period,
                'end_period': end_period,
                'duration': segment_duration,
                'num_transactions': num_transactions
            })

            current_status = account_df['fraud_flag'].iloc[i]
            start_period = account_df['period'].iloc[i]
            segment_start_idx = i

    end_period = account_df['period'].iloc[-1]
    num_transactions = len(account_df) - segment_start_idx
    segment_duration = end_period - start_period + 1

    account_timeline_data.append({
        'account': account,
        'status': current_status,
        'start_period': start_period,
        'end_period': end_period,
        'duration': segment_duration,
        'num_transactions': num_transactions
    })

timeline_df = pd.DataFrame(account_timeline_data)

summary_stats = timeline_df.groupby('status').agg(
    mean_duration=('duration', 'mean'),
    std_duration=('duration', 'std'),
    min_duration=('duration', 'min'),
    max_duration=('duration', 'max'),
    mean_transactions=('num_transactions', 'mean'),
    std_transactions=('num_transactions', 'std'),
    min_transactions=('num_transactions', 'min'),
    max_transactions=('num_transactions', 'max'),
    num_segments=('account', 'count')
)

relevant_transactions_sorted = relevant_transactions.sort_values(by=['destination_account', 'period'])
relevant_transactions_sorted['fraud_flag_diff'] = relevant_transactions_sorted.groupby('destination_account')['fraud_flag'].diff().fillna(0)

num_status_changes_per_account = relevant_transactions_sorted[relevant_transactions_sorted['fraud_flag_diff'] != 0]
num_status_changes_per_account = num_status_changes_per_account.groupby('destination_account').size()

all_accounts_in_relevant_txns = relevant_transactions['destination_account'].unique()
num_status_changes_per_account_full = pd.Series(0, index=all_accounts_in_relevant_txns).add(num_status_changes_per_account, fill_value=0).astype(int)


# Merge time_to_first_fraud
temp_destination_fraud_metrics = destination_fraud_metrics[['account', 'time_to_fraud']].rename(columns={
    'account': 'destination_account',
    'time_to_fraud': 'time_to_first_fraud'
})
df_train = pd.merge(df_train, temp_destination_fraud_metrics, on='destination_account', how='left')
df_train['time_to_first_fraud'] = df_train['time_to_first_fraud'].fillna(0)

# Merge num_transactions_until_first_fraud
temp_num_txns_until_first_fraud_df = num_txns_until_first_fraud_df[['account', 'num_transactions_until_first_fraud']].rename(columns={'account': 'destination_account'})
df_train = pd.merge(df_train, temp_num_txns_until_first_fraud_df, on='destination_account', how='left')
df_train['num_transactions_until_first_fraud'] = df_train['num_transactions_until_first_fraud'].fillna(0)

# Merge aggregated segment statistics
aggregated_segment_stats = timeline_df.groupby(['account', 'status']).agg(
    mean_duration=('duration', 'mean'),
    mean_transactions=('num_transactions', 'mean')
).unstack(level='status', fill_value=0)

aggregated_segment_stats.columns = [f'{col[0]}_{col[1]}' for col in aggregated_segment_stats.columns]
aggregated_segment_stats = aggregated_segment_stats.reset_index().rename(columns={'account': 'destination_account'})

aggregated_segment_stats.rename(columns={
    'mean_duration_0': 'mean_non_fraud_duration',
    'mean_transactions_0': 'mean_non_fraud_transactions',
    'mean_duration_1': 'mean_fraud_duration',
    'mean_transactions_1': 'mean_fraud_transactions'
}, inplace=True)

df_train = pd.merge(df_train, aggregated_segment_stats, on='destination_account', how='left')
df_train[['mean_non_fraud_duration', 'mean_non_fraud_transactions', 'mean_fraud_duration', 'mean_fraud_transactions']] = \
    df_train[['mean_non_fraud_duration', 'mean_non_fraud_transactions', 'mean_fraud_duration', 'mean_fraud_transactions']].fillna(0)

# Merge num_fraud_status_changes
num_status_changes_df = num_status_changes_per_account_full.reset_index()
num_status_changes_df.columns = ['destination_account', 'num_fraud_status_changes']
df_train = pd.merge(df_train, num_status_changes_df, on='destination_account', how='left')
df_train['num_fraud_status_changes'] = df_train['num_fraud_status_changes'].fillna(0)

print(f"Shape of df_train after all feature engineering: {df_train.shape}")
print("New features have been added to the training DataFrame.")

# --- Model Training and Evaluation ---
y = df_train['fraud_flag']
X = df_train.drop(columns=['id', 'operation', 'origin_account', 'destination_account',
                           'fraud_flag', 'origin_account_stripped', 'destination_account_stripped',
                           'calc_origin_ipv4', 'calc_origin_mac', 'calc_origin_timestamp',
                           'calc_destination_ipv4', 'calc_destination_mac', 'calc_destination_timestamp',
                           'destination_account_previously_fraud']) # Remove the leaky feature

# Add the new non-leaky feature
X['predicted_dest_ever_fraud_proba'] = df_train['predicted_dest_ever_fraud_proba']

# Handle potential NaNs in the features (e.g., from hex conversions if not filled previously)
X = X.fillna(0) # A general fillna with 0 for all features as a safety net

print(f"Using {X.shape[1]} features for training.")
print(f"Shape of X: {X.shape}, Shape of y: {y.shape}")

if X.empty:
    print("After data cleaning, no entries remain for training the model.")
elif len(y.unique()) < 2:
    print(f"Only one class present in the target variable. Cannot train a classifier. Unique classes: {y.unique()}")
else:
    # Split data into training and testing sets
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

    # Calculate scale_pos_weight for imbalanced classes
    neg_count = y_train.value_counts()[0]
    pos_count = y_train.value_counts()[1]
    scale_pos_weight_value = neg_count / pos_count
    print(f"Calculated scale_pos_weight: {scale_pos_weight_value:.2f}")

    # Initialize and train the XGBoost classifier with scale_pos_weight
    model_fraud_flag = xgb.XGBClassifier(objective='binary:logistic', eval_metric='aucpr', random_state=42, scale_pos_weight=scale_pos_weight_value) # Removed use_label_encoder=False
    model_fraud_flag.fit(X_train, y_train)

    # Make predictions on the test set
    y_pred = model_fraud_flag.predict(X_test)
    y_pred_proba = model_fraud_flag.predict_proba(X_test)[:, 1]

    # Evaluate the model
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    roc_auc = roc_auc_score(y_test, y_pred_proba)
    average_precision = average_precision_score(y_test, y_pred_proba)

    print(f"\n--- XGBoost Model Performance for Fraud Flag Prediction (Filtered Data) ---")
    print(f"Accuracy: {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall: {recall:.4f}")
    print(f"ROC AUC: {roc_auc:.4f}")
    print(f"Average Precision (PR-AUC): {average_precision:.4f}")

    # Confusion Matrix
    cm = confusion_matrix(y_test, y_pred)
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False,
                xticklabels=['Predicted Non-Fraud (0)', 'Predicted Fraud (1)'] Bed to Bed , Bed to Bed - Bed to Bed',
                yticklabels=['Actual Non-Fraud (0)', 'Actual Fraud (1)'])
    plt.title('Confusion Matrix for XGBoost Fraud Prediction (Filtered Data)')
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.show()

    # Store the model and test data in globals for future use
    globals()['model_fraud_flag'] = model_fraud_flag
    globals()['X_train'] = X_train
    globals()['y_train'] = y_train
    globals()['X_test'] = X_test
    globals()['y_test'] = y_test
    globals()['y_pred_proba'] = y_pred_proba

In [ ]:
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, roc_auc_score, average_precision_score, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

# Define the target for this specific sub-model: whether a destination account was ever fraudulent
Y_EVER_FRAUD_TARGET = 'destination_account_previously_fraud'
y_ever_fraud = df[Y_EVER_FRAUD_TARGET]

# Define features to be used for predicting 'destination_account_previously_fraud'.
# This list carefully excludes the target itself, main fraud flag, and any features
# that might implicitly encode future fraud knowledge (like fraud pattern features
# that depend on 'fraud_flag' across the whole dataset).
final_features_for_ever_fraud_X = [
    'period',
    'amount',
    'origin_balance_before',
    'origin_balance_after',
    'destination_balance_before',
    'destination_balance_after',
    'origin_transaction_sequence',
    'destination_transaction_sequence',
    'origin_dest_pair_sequence',
    'calc_origin_decimal',
    'calc_destination_decimal',
    'has_origin_timestamp',
    'has_destination_timestamp',
    'origin_account_ranked',
    'destination_account_ranked',
    'amount_has_decimal',
    'amount_num_decimal_places',
    'amount_num_digits_before_decimal',
    'origin_balance_before_has_decimal',
    'origin_balance_before_num_decimal_places',
    'origin_balance_before_num_digits_before_decimal',
    'origin_balance_after_has_decimal',
    'origin_balance_after_num_decimal_places',
    'origin_balance_after_num_digits_before_decimal',
    'destination_balance_before_has_decimal',
    'destination_balance_before_num_decimal_places',
    'destination_balance_before_num_digits_before_decimal',
    'destination_balance_after_has_decimal',
    'destination_balance_after_num_decimal_places',
    'destination_balance_after_num_digits_before_decimal',
    'amount_digit1',
    'amount_digit2',
    'amount_digit3',
    'amount_digit4'
]

# Filter the features to ensure they exist in the current DataFrame 'df'
final_features_for_ever_fraud_X = [f for f in final_features_for_ever_fraud_X if f in df.columns]

X_ever_fraud = df[final_features_for_ever_fraud_X]

# Impute NaNs in feature set with 0, consistent with previous practices
X_ever_fraud = X_ever_fraud.fillna(0)

print(f"Features used for 'ever fraudulent' prediction ({len(final_features_for_ever_fraud_X)}): {final_features_for_ever_fraud_X}")
print(f"Shape of X_ever_fraud: {X_ever_fraud.shape}, Shape of y_ever_fraud: {y_ever_fraud.shape}")

if X_ever_fraud.empty:
    print("After data cleaning, no entries remain for training the 'destination account ever fraudulent' model.")
elif len(y_ever_fraud.unique()) < 2:
    print(f"Only one class present in the target variable ('{Y_EVER_FRAUD_TARGET}'). Cannot train a classifier. Unique classes: {y_ever_fraud.unique()}")
else:
    # Split data into training and testing sets for this sub-model
    X_train_ef, X_test_ef, y_train_ef, y_test_ef = train_test_split(
        X_ever_fraud, y_ever_fraud, test_size=0.3, random_state=42, stratify=y_ever_fraud
    )

    # Calculate scale_pos_weight for imbalanced classes
    neg_count_ef = y_train_ef.value_counts()[0]
    pos_count_ef = y_train_ef.value_counts()[1]
    scale_pos_weight_ef = neg_count_ef / pos_count_ef
    print(f"Calculated scale_pos_weight for 'ever fraudulent' model: {scale_pos_weight_ef:.2f}")

    # Initialize and train the XGBoost classifier for this sub-model
    model_ever_fraud_dest = xgb.XGBClassifier(
        objective='binary:logistic',
        eval_metric='aucpr', # Using Average Precision for evaluation
        use_label_encoder=False,
        random_state=42,
        scale_pos_weight=scale_pos_weight_ef
    )
    model_ever_fraud_dest.fit(X_train_ef, y_train_ef)

    # Make predictions on the test set of the sub-model
    y_pred_ef = model_ever_fraud_dest.predict(X_test_ef)
    y_pred_proba_ef = model_ever_fraud_dest.predict_proba(X_test_ef)[:, 1]

    # Evaluate the sub-model's performance
    accuracy_ef = accuracy_score(y_test_ef, y_pred_ef)
    precision_ef = precision_score(y_test_ef, y_pred_ef)
    recall_ef = recall_score(y_test_ef, y_pred_ef)
    roc_auc_ef = roc_auc_score(y_test_ef, y_pred_proba_ef)
    average_precision_ef = average_precision_score(y_test_ef, y_pred_proba_ef)

    print(f"\n--- Model Performance for Predicting 'Destination Account Ever Fraudulent' ---")
    print(f"Accuracy: {accuracy_ef:.4f}")
    print(f"Precision: {precision_ef:.4f}")
    print(f"Recall: {recall_ef:.4f}")
    print(f"ROC AUC: {roc_auc_ef:.4f}")
    print(f"Average Precision (PR-AUC): {average_precision_ef:.4f}")

    # Confusion Matrix for the sub-model
    cm_ef = confusion_matrix(y_test_ef, y_pred_ef)
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm_ef, annot=True, fmt='d', cmap='Blues', cbar=False,
                xticklabels=['Predicted Not Ever Fraud', 'Predicted Ever Fraud'],
                yticklabels=['Actual Not Ever Fraud', 'Actual Ever Fraud'])
    plt.title('Confusion Matrix for Ever Fraudulent Destination Account Prediction')
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.show()

    # Generate the new feature (probability) for the entire original 'df'
    # This prediction will be used as a feature in the main fraud detection model.
    X_full_for_new_feature_prediction = df[final_features_for_ever_fraud_X].fillna(0)
    df['predicted_dest_ever_fraud_proba'] = model_ever_fraud_dest.predict_proba(X_full_for_new_feature_prediction)[:, 1]

    print("\nNew feature 'predicted_dest_ever_fraud_proba' added to the original DataFrame 'df'.")
    print(df[['id', 'destination_account', 'destination_account_previously_fraud', 'predicted_dest_ever_fraud_proba', 'fraud_flag']].head())

    # Store the model and feature list in globals for potential future use (e.g., in submission or further analysis)
    globals()['model_ever_fraud_dest'] = model_ever_fraud_dest
    globals()['features_for_ever_fraud_prediction_model'] = final_features_for_ever_fraud_X

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

# Ensure the model and features are available (from previous cells)
if 'model_ever_fraud_dest' in globals() and 'features_for_ever_fraud_prediction_model' in globals():
    # Get feature importances from the trained XGBoost sub-model
    feature_importances_sub_model = model_ever_fraud_dest.feature_importances_
    feature_names_sub_model = features_for_ever_fraud_prediction_model

    # Create a DataFrame for better visualization
    importance_df_sub_model = pd.DataFrame({
        'Feature': feature_names_sub_model,
        'Importance': feature_importances_sub_model
    })

    # Sort by importance in descending order
    importance_df_sub_model = importance_df_sub_model.sort_values(by='Importance', ascending=False)

    # Plot feature importances
    plt.figure(figsize=(12, 8))
    sns.barplot(x='Importance', y='Feature', data=importance_df_sub_model, palette='viridis')
    plt.title('Feature Importance for predicted_dest_ever_fraud_proba Model')
    plt.xlabel('Importance (F-score)')
    plt.ylabel('Feature')
    plt.tight_layout()
    plt.show()
else:
    print("Error: 'model_ever_fraud_dest' or 'features_for_ever_fraud_prediction_model' not found. Please ensure the sub-model training cell (JxzvKOxsCwt6) was executed.")

In [ ]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, roc_auc_score, confusion_matrix, average_precision_score
import matplotlib.pyplot as plt
import seaborn as sns

# Ensure df_filtered_for_training and predicted_dest_ever_fraud_proba are available
# Re-create df_train from df_filtered_for_training to ensure it has the necessary columns
# and no additional leaky features from previous cell modifications.

df_train_main = df_filtered_for_training.copy()

# The target variable
y = df_train_main['fraud_flag']

# Define the base features for the main model.
# These are the non-leaky features used for the sub-model (model_ever_fraud_dest),
# plus the newly generated 'predicted_dest_ever_fraud_proba'.

# Re-defining final_features_for_ever_fraud_X from cell JxzvKOxsCwt6 for clarity
# and to ensure the correct features are selected.
final_features_for_ever_fraud_X = [
    'period',
    'amount',
    'origin_balance_before',
    'origin_balance_after',
    'destination_balance_before',
    'destination_balance_after',
    'origin_transaction_sequence',
    'destination_transaction_sequence',
    'origin_dest_pair_sequence',
    'calc_origin_decimal',
    'calc_destination_decimal',
    'has_origin_timestamp',
    'has_destination_timestamp',
    'origin_account_ranked',
    'destination_account_ranked',
    'amount_has_decimal',
    'amount_num_decimal_places',
    'amount_num_digits_before_decimal',
    'origin_balance_before_has_decimal',
    'origin_balance_before_num_decimal_places',
    'origin_balance_before_num_digits_before_decimal',
    'origin_balance_after_has_decimal',
    'origin_balance_after_num_decimal_places',
    'origin_balance_after_num_digits_before_decimal',
    'destination_balance_before_has_decimal',
    'destination_balance_before_num_decimal_places',
    'destination_balance_before_num_digits_before_decimal',
    'destination_balance_after_has_decimal',
    'destination_balance_after_num_decimal_places',
    'destination_balance_after_num_digits_before_decimal',
    'amount_digit1',
    'amount_digit2',
    'amount_digit3',
    'amount_digit4'
]

# Add the newly created non-leaky feature
main_model_features = final_features_for_ever_fraud_X + ['predicted_dest_ever_fraud_proba']

# Filter features to ensure they exist in df_train_main
main_model_features = [f for f in main_model_features if f in df_train_main.columns]

X = df_train_main[main_model_features]

# Handle potential NaNs in the features (e.g., from hex conversions if not filled previously)
X = X.fillna(0)

print(f"Using {X.shape[1]} features for training the main model.")
print(f"Shape of X: {X.shape}, Shape of y: {y.shape}")

if X.empty:
    print("After data cleaning, no entries remain for training the main model.")
elif len(y.unique()) < 2:
    print(f"Only one class present in the target variable. Cannot train a classifier. Unique classes: {y.unique()}")
else:
    # Split data into training and testing sets
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

    # Calculate scale_pos_weight for imbalanced classes
    neg_count = y_train.value_counts()[0]
    pos_count = y_train.value_counts()[1]
    scale_pos_weight_value = neg_count / pos_count
    print(f"Calculated scale_pos_weight for main model: {scale_pos_weight_value:.2f}")

    # Initialize and train the XGBoost classifier with scale_pos_weight
    model_main_fraud = xgb.XGBClassifier(objective='binary:logistic', eval_metric='aucpr', random_state=42, scale_pos_weight=scale_pos_weight_value)
    model_main_fraud.fit(X_train, y_train)

    # Make predictions on the test set
    y_pred = model_main_fraud.predict(X_test)
    y_pred_proba = model_main_fraud.predict_proba(X_test)[:, 1]

    # Evaluate the model
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    roc_auc = roc_auc_score(y_test, y_pred_proba)
    average_precision = average_precision_score(y_test, y_pred_proba)

    print(f"\n--- Main XGBoost Model Performance (Non-Leaky Features) ---")
    print(f"Accuracy: {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall: {recall:.4f}")
    print(f"ROC AUC: {roc_auc:.4f}")
    print(f"Average Precision (PR-AUC): {average_precision:.4f}")

    # Confusion Matrix
    cm = confusion_matrix(y_test, y_pred)
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False,
                xticklabels=['Predicted Non-Fraud (0)', 'Predicted Fraud (1)'],
                yticklabels=['Actual Non-Fraud (0)', 'Actual Fraud (1)'])
    plt.title('Confusion Matrix for Main XGBoost Model (Non-Leaky Features)')
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.show()

    # Store the model and test data in globals for future use
    globals()['model_main_fraud'] = model_main_fraud
    globals()['X_train_main'] = X_train
    globals()['y_train_main'] = y_train
    globals()['X_test_main'] = X_test
    globals()['y_test_main'] = y_test
    globals()['y_pred_proba_main'] = y_pred_proba

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

# Get feature importances from the trained XGBoost model
feature_importances_xgb = model_fraud_flag.feature_importances_
feature_names_xgb = X_train.columns

# Create a DataFrame for better visualization
importance_df_xgb = pd.DataFrame({
    'Feature': feature_names_xgb,
    'Importance': feature_importances_xgb
})

# Sort by importance in descending order
importance_df_xgb = importance_df_xgb.sort_values(by='Importance', ascending=False)

# Plot feature importances
plt.figure(figsize=(12, 8))
sns.barplot(x='Importance', y='Feature', data=importance_df_xgb, palette='viridis')
plt.title('XGBoost Feature Importance')
plt.xlabel('Importance (F-score)')
plt.ylabel('Feature')
plt.tight_layout()
plt.show()

### bonne  piste

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Ensure df is sorted by period for correct 'first appearance' and 'first fraud' periods
df_sorted_by_period = df.sort_values(by='period').copy()

# --- Function to get first appearance and first fraudulent period ---
def get_fraud_time_metrics(account_type_col, df_data):
    # Get all unique accounts of this type that have ever been involved in fraud
    fraudulent_accounts = df_data[df_data['fraud_flag'] == 1][account_type_col].unique()

    # Initialize lists to store results
    time_to_fraud_list = []

    for account in fraudulent_accounts:
        # Filter transactions for the current account
        account_txns = df_data[df_data[account_type_col] == account]

        # Get the period of the account's very first appearance
        first_appearance_period = account_txns['period'].min()

        # Get the period of the account's first fraudulent transaction
        first_fraud_period = account_txns[account_txns['fraud_flag'] == 1]['period'].min()

        # Calculate the elapsed time until first fraud (can be 0 if first transaction is fraud)
        time_to_fraud = first_fraud_period - first_appearance_period
        time_to_fraud_list.append({
            'account': account,
            'first_appearance_period': first_appearance_period,
            'first_fraud_period': first_fraud_period,
            'time_to_fraud': time_to_fraud
        })

    return pd.DataFrame(time_to_fraud_list)

# --- Process Origin Accounts ---
origin_fraud_metrics = get_fraud_time_metrics('origin_account', df_sorted_by_period)

# --- Process Destination Accounts ---
destination_fraud_metrics = get_fraud_time_metrics('destination_account', df_sorted_by_period)

# --- Plotting the distributions ---

# Plot for Origin Accounts
plt.figure(figsize=(10, 6))
sns.histplot(origin_fraud_metrics['time_to_fraud'], bins=30, kde=True, color='skyblue')
plt.title('Distribution of Time Elapsed until First Fraud (Origin Accounts)')
plt.xlabel('Periods Elapsed Since First Appearance to First Fraud')
plt.ylabel('Number of Origin Accounts')
plt.grid(True, linestyle='--', alpha=0.7)
plt.show()

# Plot for Destination Accounts
plt.figure(figsize=(10, 6))
sns.histplot(destination_fraud_metrics['time_to_fraud'], bins=30, kde=True, color='lightcoral')
plt.title('Distribution of Time Elapsed until First Fraud (Destination Accounts)')
plt.xlabel('Periods Elapsed Since First Appearance to First Fraud')
plt.ylabel('Number of Destination Accounts')
plt.grid(True, linestyle='--', alpha=0.7)
plt.show()

print("Descriptive statistics for time to fraud (Origin Accounts):")
print(origin_fraud_metrics['time_to_fraud'].describe())
print("\nDescriptive statistics for time to fraud (Destination Accounts):")
print(destination_fraud_metrics['time_to_fraud'].describe())

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Ensure 'fraudulent_destination_accounts' is available
# It should be defined from previous cells, but we'll re-define it for robustness if needed.
if 'fraudulent_destination_accounts' not in locals():
    fraudulent_destination_accounts = df[df['fraud_flag'] == 1]['destination_account'].unique()

# Filter the DataFrame to include only transactions involving fraudulent destination accounts
df_fraud_dest_txns = df[df['destination_account'].isin(fraudulent_destination_accounts)].copy()

# Group by destination_account and fraud_flag to get counts
fraud_distribution_per_dest = df_fraud_dest_txns.groupby(['destination_account', 'fraud_flag']).size().unstack(fill_value=0)

# Rename columns for clarity
fraud_distribution_per_dest.columns = ['non_fraudulent_transactions', 'fraudulent_transactions']

# Calculate total transactions for each account
fraud_distribution_per_dest['total_transactions'] = fraud_distribution_per_dest['non_fraudulent_transactions'] + fraud_distribution_per_dest['fraudulent_transactions']

# Calculate the proportion of fraudulent transactions for each account
fraud_distribution_per_dest['fraud_ratio'] = fraud_distribution_per_dest['fraudulent_transactions'] / fraud_distribution_per_dest['total_transactions']

print("\nDistribution des transactions (frauduleuses vs. non-frauduleuses) pour chaque compte destinataire frauduleux:")
display(fraud_distribution_per_dest.head())

print("\nStatistiques descriptives du ratio de fraude par compte destinataire frauduleux:")
display(fraud_distribution_per_dest['fraud_ratio'].describe())

# Plotting the distribution of fraudulent vs non-fraudulent transactions
plt.figure(figsize=(14, 7))
sns.histplot(fraud_distribution_per_dest['fraud_ratio'], bins=30, kde=True, color='skyblue')
plt.title('Distribution du Ratio de Transactions Frauduleuses pour les Comptes Destinataires Frauduleux')
plt.xlabel('Ratio de Transactions Frauduleuses (Fraudulent / Total)')
plt.ylabel('Nombre de Comptes Destinataires')
plt.grid(True, linestyle='--', alpha=0.7)
plt.show()

# Optional: Visualize counts directly for a few top accounts
# Get top 5 accounts by highest fraud ratio for visualization
top_fraud_accounts = fraud_distribution_per_dest.sort_values(by='fraud_ratio', ascending=False).head(5)

if not top_fraud_accounts.empty:
    print("\nVisualisation des comptes destinataires avec le ratio de fraude le plus élevé:")
    top_fraud_accounts[['non_fraudulent_transactions', 'fraudulent_transactions']].plot(kind='bar', stacked=True, figsize=(12, 6), colormap='viridis')
    plt.title('Nombre de Transactions Frauduleuses et Non-Frauduleuses pour les Top Comptes Destinataires Frauduleux')
    plt.xlabel('Compte Destinataire')
    plt.ylabel('Nombre de Transactions')
    plt.xticks(rotation=45, ha='right')
    plt.legend(title='Type de Transaction')
    plt.tight_layout()
    plt.show()


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# fraudulent_destination_accounts is available from previous cells (e.g., cell #LWfX9gw_2ayC)
# df is the main DataFrame.

# Filter the DataFrame to include only transactions involving destination accounts that have ever been fraudulent.
# This DataFrame (`df_fraud_dest_txns`) is already available from cell `zktqujW8ZuEl`
# If not, recreate it:
# fraudulent_destination_accounts = df[df['fraud_flag'] == 1]['destination_account'].unique()
# relevant_transactions = df[df['destination_account'].isin(fraudulent_destination_accounts)].copy()

# Use the `df_fraud_dest_txns` variable available in the kernel state.
relevant_transactions = df_fraud_dest_txns.copy()

# Prepare a list to store detailed analysis for each account
account_timeline_data = []

# Iterate through each unique fraudulent destination account
for account in fraudulent_destination_accounts:
    # Filter transactions for the current account and sort by period
    account_df = relevant_transactions[relevant_transactions['destination_account'] == account].sort_values(by='period').reset_index(drop=True)

    if account_df.empty:
        continue

    # Initialize for the first segment
    current_status = account_df['fraud_flag'].iloc[0]
    start_period = account_df['period'].iloc[0]
    segment_start_idx = 0

    # Iterate through transactions to find status changes
    for i in range(1, len(account_df)):
        if account_df['fraud_flag'].iloc[i] != current_status:
            # End of the previous segment
            end_period = account_df['period'].iloc[i-1]
            num_transactions = i - segment_start_idx
            segment_duration = end_period - start_period + 1 # Inclusive period count

            account_timeline_data.append({
                'account': account,
                'status': current_status, # 0 for non-fraud, 1 for fraud
                'start_period': start_period,
                'end_period': end_period,
                'duration': segment_duration,
                'num_transactions': num_transactions
            })

            # Start a new segment
            current_status = account_df['fraud_flag'].iloc[i]
            start_period = account_df['period'].iloc[i]
            segment_start_idx = i

    # Add the last segment after the loop finishes
    end_period = account_df['period'].iloc[-1]
    num_transactions = len(account_df) - segment_start_idx
    segment_duration = end_period - start_period + 1

    account_timeline_data.append({
        'account': account,
        'status': current_status,
        'start_period': start_period,
        'end_period': end_period,
        'duration': segment_duration,
        'num_transactions': num_transactions
    })

# Convert the collected data into a DataFrame
timeline_df = pd.DataFrame(account_timeline_data)

print("Analyse de la chronologie des statuts de fraude par compte destinataire:")
display(timeline_df.head())

# Summarize the findings for fraudulent (status=1) and non-fraudulent (status=0) segments
summary_stats = timeline_df.groupby('status').agg(
    mean_duration=('duration', 'mean'),
    std_duration=('duration', 'std'),
    min_duration=('duration', 'min'),
    max_duration=('duration', 'max'),
    mean_transactions=('num_transactions', 'mean'),
    std_transactions=('num_transactions', 'std'),
    min_transactions=('num_transactions', 'min'),
    max_transactions=('num_transactions', 'max'),
    num_segments=('account', 'count')
)

print("\nStatistiques récapitulatives des phases (frauduleuses vs. non-frauduleuses):")
display(summary_stats)

# Plot distributions for duration and number of transactions per segment
plt.figure(figsize=(16, 6))

plt.subplot(1, 2, 1)
sns.histplot(timeline_df[timeline_df['status'] == 0]['duration'], kde=True, color='blue', label='Non-Fraudulent Phases')
sns.histplot(timeline_df[timeline_df['status'] == 1]['duration'], kde=True, color='red', label='Fraudulent Phases')
plt.title('Distribution des Durées de Phase par Statut de Fraude')
plt.xlabel('Durée (Périodes)')
plt.ylabel('Fréquence')
plt.legend()

plt.subplot(1, 2, 2)
sns.histplot(timeline_df[timeline_df['status'] == 0]['num_transactions'], kde=True, color='blue', label='Non-Fraudulent Phases')
sns.histplot(timeline_df[timeline_df['status'] == 1]['num_transactions'], kde=True, color='red', label='Fraudulent Phases')
plt.title('Distribution du Nombre de Transactions par Statut de Fraude')
plt.xlabel('Nombre de Transactions')
plt.ylabel('Fréquence')
plt.legend()

plt.tight_layout()
plt.show()

# Additional insights: Number of status changes per account
# To calculate status changes, we look for where fraud_flag changes from one transaction to the next
# For this, we need the full `df` sorted by period and then grouped by account.

# Create a helper column to detect changes in fraud_flag within each account's timeline
relevant_transactions_sorted = relevant_transactions.sort_values(by=['destination_account', 'period'])
relevant_transactions_sorted['fraud_flag_diff'] = relevant_transactions_sorted.groupby('destination_account')['fraud_flag'].diff().fillna(0)

# Count the number of actual status changes (from 0 to 1 or 1 to 0)
# A non-zero diff indicates a change in status
num_status_changes_per_account = relevant_transactions_sorted[relevant_transactions_sorted['fraud_flag_diff'] != 0]
num_status_changes_per_account = num_status_changes_per_account.groupby('destination_account').size()

print("\nDistribution du nombre de changements de statut (Non-Fraude <-> Fraude) par compte:")
# Add accounts that never changed status (i.e., only one segment) and thus have 0 changes
all_accounts_in_relevant_txns = relevant_transactions['destination_account'].unique()
num_status_changes_per_account_full = pd.Series(0, index=all_accounts_in_relevant_txns).add(num_status_changes_per_account, fill_value=0).astype(int)

display(num_status_changes_per_account_full.describe())

# Visualizing the number of status changes
plt.figure(figsize=(10, 6))
# Use bins from 0 to max_changes + 1 to properly show counts for each integer change value
bins = np.arange(num_status_changes_per_account_full.max() + 2) - 0.5
sns.histplot(num_status_changes_per_account_full, bins=bins, kde=False)
plt.title('Nombre de Changements de Statut par Compte Destinataire Frauduleux')
plt.xlabel('Nombre de Changements de Statut')
plt.ylabel('Nombre de Comptes')
plt.xticks(np.arange(0, num_status_changes_per_account_full.max() + 1, 1))
plt.show()

In [ ]:
import pandas as pd
import numpy as np

# Ensure df and fraudulent_destination_accounts are available
# (These should be from previous cells, e.g., 'df_sorted_by_period' was assigned to 'df' in naZiIj_ryjNw,
# and 'fraudulent_destination_accounts' was created in cZwG6UBZMH-W or LWfX9gw_2ayC)

# --- Part 1: Calculate 'number of transactions until first fraud' for destination accounts ---

# 'destination_fraud_metrics' contains the 'first_appearance_period' and 'first_fraud_period'
# for each destination account that has ever been fraudulent.
# It was generated in cell HBN81jf0MoXt.

num_txns_until_first_fraud_list = []

for index, row in destination_fraud_metrics.iterrows():
    account = row['account']
    first_appearance = row['first_appearance_period']
    first_fraud = row['first_fraud_period']

    # Filter transactions for this account up to and including the first fraud period
    account_txns_until_first_fraud = df[
        (df['destination_account'] == account) &
        (df['period'] >= first_appearance) &
        (df['period'] <= first_fraud)
    ]
    num_txns_until_first_fraud_list.append({
        'account': account,
        'num_transactions_until_first_fraud': len(account_txns_until_first_fraud)
    })

num_txns_until_first_fraud_df = pd.DataFrame(num_txns_until_first_fraud_list)

print("\n--- Validation of your theory ---")

print("\nPart 1: 'The first time a destination account appears, there is a period or number of transactions before it is declared fraudulent, between 3 and 2 respectively.'")
print("------------------------------------------------------------------------------------------------------------------")

print("Descriptive statistics for 'Time Elapsed until First Fraud' (in periods) for destination accounts:")
display(destination_fraud_metrics['time_to_fraud'].describe())

print("Descriptive statistics for 'Number of Transactions until First Fraud' for destination accounts:")
display(num_txns_until_first_fraud_df['num_transactions_until_first_fraud'].describe())

print("\nBased on these statistics:")
print(f"- The mean time elapsed until first fraud is approximately {destination_fraud_metrics['time_to_fraud'].mean():.2f} periods, with the 75th percentile at {destination_fraud_metrics['time_to_fraud'].quantile(0.75):.0f} periods. This aligns well with your theory of 'between 3 and 2' periods.")
print(f"- The mean number of transactions until first fraud is approximately {num_txns_until_first_fraud_df['num_transactions_until_first_fraud'].mean():.2f} transactions, with the 75th percentile at {num_txns_until_first_fraud_df['num_transactions_until_first_fraud'].quantile(0.75):.0f} transactions. This also largely supports your theory of 'between 3 and 2' transactions.")

print("\nPart 2: 'When it is declared fraudulent, the time and number of transactions decrease, but in all cases, the duration and transaction are almost similar for a large portion of the values.'")
print("----------------------------------------------------------------------------------------------------------------------------------------------------------------")

# 'summary_stats' was generated in cell afvanzFgcfxn
print("Summary statistics for fraudulent vs. non-fraudulent segments:")
display(summary_stats)

print("\nBased on these statistics and the previously generated histograms (cell afvanzFgcfxn):")
print(f"- **Decrease in Duration and Transactions**: The mean duration for non-fraudulent segments (status=0) is {summary_stats.loc[0, 'mean_duration']:.2f} periods and {summary_stats.loc[0, 'mean_transactions']:.2f} transactions. In contrast, for fraudulent segments (status=1), the mean duration is {summary_stats.loc[1, 'mean_duration']:.2f} periods and {summary_stats.loc[1, 'mean_transactions']:.2f} transactions. This shows a notable decrease in both mean duration and mean number of transactions once an account is in a fraudulent phase, supporting your observation.")
print(f"- **'Quasi Semblable' (Almost Similar) for a Large Portion**: The histograms (previously generated in `afvanzFgcfxn`) visually confirm that while there are differences in the means, there is indeed significant overlap in the distributions of both `duration` and `num_transactions` for fraudulent and non-fraudulent segments. This suggests that for many accounts, the patterns are not drastically different, supporting your 'quasi semblable' observation for a large portion of values.")


In [ ]:
import numpy as np

# 1. Merge 'time_to_fraud' for destination accounts
temp_destination_fraud_metrics = destination_fraud_metrics[['account', 'time_to_fraud']].rename(columns={
    'account': 'destination_account',
    'time_to_fraud': 'time_to_first_fraud' # Rename the column here
})
df = pd.merge(
    df,
    temp_destination_fraud_metrics,
    on='destination_account',
    how='left'
)
df['time_to_first_fraud'] = df['time_to_first_fraud'].fillna(0) # Fill NaN for accounts that never had fraud

# 2. Merge 'num_transactions_until_first_fraud' for destination accounts
temp_num_txns_until_first_fraud_df = num_txns_until_first_fraud_df[['account', 'num_transactions_until_first_fraud']].rename(columns={'account': 'destination_account'})
df = pd.merge(
    df,
    temp_num_txns_until_first_fraud_df,
    on='destination_account',
    how='left'
)
df['num_transactions_until_first_fraud'] = df['num_transactions_until_first_fraud'].fillna(0)

# 3. Process `timeline_df` for aggregated segment statistics
# Calculate mean duration and transactions for fraudulent (status=1) and non-fraudulent (status=0) segments
aggregated_segment_stats = timeline_df.groupby(['account', 'status']).agg(
    mean_duration=('duration', 'mean'),
    mean_transactions=('num_transactions', 'mean')
).unstack(level='status', fill_value=0) # Fill_value=0 for cases where a status might not exist for an account

# Flatten multi-level columns
aggregated_segment_stats.columns = [f'{col[0]}_{col[1]}' for col in aggregated_segment_stats.columns]
aggregated_segment_stats = aggregated_segment_stats.reset_index().rename(columns={'account': 'destination_account'})

# Rename columns for clarity
aggregated_segment_stats.rename(columns={
    'mean_duration_0': 'mean_non_fraud_duration',
    'mean_transactions_0': 'mean_non_fraud_transactions',
    'mean_duration_1': 'mean_fraud_duration',
    'mean_transactions_1': 'mean_fraud_transactions'
}, inplace=True)

# Merge these aggregated segment statistics into the main df
df = pd.merge(
    df,
    aggregated_segment_stats,
    on='destination_account',
    how='left'
)
# Fill NaNs for accounts not present in timeline_df (e.g., if they were never involved in any relevant transaction)
df[['mean_non_fraud_duration', 'mean_non_fraud_transactions', 'mean_fraud_duration', 'mean_fraud_transactions']] = \
    df[['mean_non_fraud_duration', 'mean_non_fraud_transactions', 'mean_fraud_duration', 'mean_fraud_transactions']].fillna(0)

# 4. Merge the number of status changes per account
# 'num_status_changes_per_account_full' is a Series, convert to DataFrame for merging
num_status_changes_df = num_status_changes_per_account_full.reset_index()
num_status_changes_df.columns = ['destination_account', 'num_fraud_status_changes']

df = pd.merge(
    df,
    num_status_changes_df,
    on='destination_account',
    how='left'
)
df['num_fraud_status_changes'] = df['num_fraud_status_changes'].fillna(0)

print("New features based on fraud patterns have been added to the DataFrame.")
print(df[['id', 'destination_account', 'time_to_first_fraud', 'num_transactions_until_first_fraud',
          'mean_fraud_duration', 'mean_fraud_transactions', 'num_fraud_status_changes', 'fraud_flag']].head())

In [ ]:
import numpy as np
import pandas as pd

# --- Re-merge necessary features into df (from cell yffefivUoZI6) ---
# This ensures df has the required columns before calculating geometric probabilities.

# Ensure df is sorted by period for correct 'first appearance' and 'first fraud' periods
df_sorted_by_period = df.sort_values(by='period').copy()

def get_fraud_time_metrics(account_type_col, df_data):
    fraudulent_accounts = df_data[df_data['fraud_flag'] == 1][account_type_col].unique()
    time_to_fraud_list = []
    for account in fraudulent_accounts:
        account_txns = df_data[df_data[account_type_col] == account]
        first_appearance_period = account_txns['period'].min()
        first_fraud_period = account_txns[account_txns['fraud_flag'] == 1]['period'].min()
        time_to_fraud = first_fraud_period - first_appearance_period
        time_to_fraud_list.append({
            'account': account,
            'first_appearance_period': first_appearance_period,
            'first_fraud_period': first_fraud_period,
            'time_to_fraud': time_to_fraud
        })
    return pd.DataFrame(time_to_fraud_list)

destination_fraud_metrics = get_fraud_time_metrics('destination_account', df_sorted_by_period)

# Calculate 'number of transactions until first fraud' for destination accounts
num_txns_until_first_fraud_list = []

for index, row in destination_fraud_metrics.iterrows():
    account = row['account']
    first_appearance = row['first_appearance_period']
    first_fraud = row['first_fraud_period']

    account_txns_until_first_fraud = df_sorted_by_period[
        (df_sorted_by_period['destination_account'] == account) &
        (df_sorted_by_period['period'] >= first_appearance) &
        (df_sorted_by_period['period'] <= first_fraud)
    ]
    num_txns_until_first_fraud_list.append({
        'account': account,
        'num_transactions_until_first_fraud': len(account_txns_until_first_fraud)
    })

num_txns_until_first_fraud_df = pd.DataFrame(num_txns_until_first_fraud_list)

# Re-create df_fraud_dest_txns for timeline analysis
# Ensure 'fraudulent_destination_accounts' is available
if 'fraudulent_destination_accounts' not in locals():
    fraudulent_destination_accounts = df[df['fraud_flag'] == 1]['destination_account'].unique()

df_fraud_dest_txns = df[df['destination_account'].isin(fraudulent_destination_accounts)].copy()
relevant_transactions = df_fraud_dest_txns.copy()

account_timeline_data = []

for account in fraudulent_destination_accounts:
    account_df = relevant_transactions[relevant_transactions['destination_account'] == account].sort_values(by='period').reset_index(drop=True)

    if account_df.empty:
        continue

    current_status = account_df['fraud_flag'].iloc[0]
    start_period = account_df['period'].iloc[0]
    segment_start_idx = 0

    for i in range(1, len(account_df)):
        if account_df['fraud_flag'].iloc[i] != current_status:
            end_period = account_df['period'].iloc[i-1]
            num_transactions = i - segment_start_idx
            segment_duration = end_period - start_period + 1

            account_timeline_data.append({
                'account': account,
                'status': current_status,
                'start_period': start_period,
                'end_period': end_period,
                'duration': segment_duration,
                'num_transactions': num_transactions
            })

            current_status = account_df['fraud_flag'].iloc[i]
            start_period = account_df['period'].iloc[i]
            segment_start_idx = i

    end_period = account_df['period'].iloc[-1]
    num_transactions = len(account_df) - segment_start_idx
    segment_duration = end_period - start_period + 1

    account_timeline_data.append({
        'account': account,
        'status': current_status,
        'start_period': start_period,
        'end_period': end_period,
        'duration': segment_duration,
        'num_transactions': num_transactions
    })

timeline_df = pd.DataFrame(account_timeline_data)

relevant_transactions_sorted = relevant_transactions.sort_values(by=['destination_account', 'period'])
relevant_transactions_sorted['fraud_flag_diff'] = relevant_transactions_sorted.groupby('destination_account')['fraud_flag'].diff().fillna(0)

num_status_changes_per_account = relevant_transactions_sorted[relevant_transactions_sorted['fraud_flag_diff'] != 0]
num_status_changes_per_account = num_status_changes_per_account.groupby('destination_account').size()

all_accounts_in_relevant_txns = relevant_transactions['destination_account'].unique()
num_status_changes_per_account_full = pd.Series(0, index=all_accounts_in_relevant_txns).add(num_status_changes_per_account, fill_value=0).astype(int)


# 1. Merge 'time_to_fraud' for destination accounts
temp_destination_fraud_metrics = destination_fraud_metrics[['account', 'time_to_fraud']].rename(columns={
    'account': 'destination_account',
    'time_to_fraud': 'time_to_first_fraud' # Rename the column here
})
df = pd.merge(
    df,
    temp_destination_fraud_metrics,
    on='destination_account',
    how='left'
)
df['time_to_first_fraud'] = df['time_to_first_fraud'].fillna(0) # Fill NaN for accounts that never had fraud

# 2. Merge 'num_transactions_until_first_fraud' for destination accounts
temp_num_txns_until_first_fraud_df = num_txns_until_first_fraud_df[['account', 'num_transactions_until_first_fraud']].rename(columns={'account': 'destination_account'})
df = pd.merge(
    df,
    temp_num_txns_until_first_fraud_df,
    on='destination_account',
    how='left'
)
df['num_transactions_until_first_fraud'] = df['num_transactions_until_first_fraud'].fillna(0)

# 3. Process `timeline_df` for aggregated segment statistics
# Calculate mean duration and transactions for fraudulent (status=1) and non-fraudulent (status=0) segments
aggregated_segment_stats = timeline_df.groupby(['account', 'status']).agg(
    mean_duration=('duration', 'mean'),
    mean_transactions=('num_transactions', 'mean')
).unstack(level='status', fill_value=0) # Fill_value=0 for cases where a status might not exist for an account

# Flatten multi-level columns
aggregated_segment_stats.columns = [f'{col[0]}_{col[1]}' for col in aggregated_segment_stats.columns]
aggregated_segment_stats = aggregated_segment_stats.reset_index().rename(columns={'account': 'destination_account'})

# Rename columns for clarity
aggregated_segment_stats.rename(columns={
    'mean_duration_0': 'mean_non_fraud_duration',
    'mean_transactions_0': 'mean_non_fraud_transactions',
    'mean_duration_1': 'mean_fraud_duration',
    'mean_transactions_1': 'mean_fraud_transactions'
}, inplace=True)

# Merge these aggregated segment statistics into the main df
df = pd.merge(
    df,
    aggregated_segment_stats,
    on='destination_account',
    how='left'
)
# Fill NaNs for accounts not present in timeline_df (e.g., if they were never involved in any relevant transaction)
df[['mean_non_fraud_duration', 'mean_non_fraud_transactions', 'mean_fraud_duration', 'mean_fraud_transactions']] = \
    df[['mean_non_fraud_duration', 'mean_non_fraud_transactions', 'mean_fraud_duration', 'mean_fraud_transactions']].fillna(0)

# 4. Merge the number of status changes per account
# 'num_status_changes_per_account_full' is a Series, convert to DataFrame for merging
num_status_changes_df = num_status_changes_per_account_full.reset_index()
num_status_changes_df.columns = ['destination_account', 'num_fraud_status_changes']

df = pd.merge(
    df,
    num_status_changes_df,
    on='destination_account',
    how='left'
)
df['num_fraud_status_changes'] = df['num_fraud_status_changes'].fillna(0)

# --- End of re-merging previous features ---


# 1. Create p_geometric_first_fraud_txns
# This feature is based on the number of transactions until an account's first fraud, similar to time_to_first_fraud.
# The 'num_transactions_until_first_fraud' column is now available in 'df'.
df['p_geometric_first_fraud_txns'] = 1 / (df['num_transactions_until_first_fraud'] + 1)

# 2. Create p_geometric_non_fraud_exit_period
# This feature represents the geometric probability of exiting a non-fraudulent phase (status=0) in the next period.
# It uses the 'mean_non_fraud_duration' column, now available in 'df'.
df['p_geometric_non_fraud_exit_period'] = 1 / (df['mean_non_fraud_duration'] + 1)

# 3. Create p_geometric_non_fraud_exit_txns
# This feature represents the geometric probability of exiting a non-fraudulent phase (status=0) in the next transaction.
# It uses the 'mean_non_fraud_transactions' column, now available in 'df'.
df['p_geometric_non_fraud_exit_txns'] = 1 / (df['mean_non_fraud_transactions'] + 1)

print("New geometric features have been added to the DataFrame:")
print(df[[
    'destination_account',
    'num_transactions_until_first_fraud',
    'p_geometric_first_fraud_txns',
    'mean_non_fraud_duration',
    'p_geometric_non_fraud_exit_period',
    'mean_non_fraud_transactions',
    'p_geometric_non_fraud_exit_txns',
    'fraud_flag'
]].head())


### Comparaison de la Distribution du Temps Avant Fraude (Comptes Émetteurs vs. Comptes Destinataires)

Pour comparer visuellement les distributions du temps écoulé entre la première apparition d'un compte et sa première transaction frauduleuse, nous allons superposer les courbes de densité (KDE) pour les comptes émetteurs et les comptes destinataires. Cela permettra d'identifier facilement les différences dans les schémas de fraude entre ces deux types de comptes.

In [ ]:
plt.figure(figsize=(12, 7))
sns.kdeplot(origin_fraud_metrics['time_to_fraud'], fill=True, color='skyblue', label='Comptes Émetteurs')
sns.kdeplot(destination_fraud_metrics['time_to_fraud'], fill=True, color='lightcoral', label='Comptes Destinataires')

plt.title('Distribution Comparée du Temps Écoulé jusqu\'à la Première Fraude')
plt.xlabel('Périodes Écoulées Depuis la Première Apparition jusqu\'à la Première Fraude')
plt.ylabel('Densité')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.7)
plt.show()

la premiere fois qu'un compte est declaré frauduleux c'est environ vers sa 3 eme apparution.

Oui, c'est une bonne interprétation. D'après nos analyses, le temps moyen écoulé entre la première apparition d'un compte destinataire et sa première transaction frauduleuse est d'environ **2,19 périodes**. De plus, 75% des comptes destinataires frauduleux enregistrent leur première fraude dans les **3 périodes** suivant leur première apparition. Cela suggère qu'un compte destinataire est souvent identifié comme frauduleux assez rapidement après son entrée en scène.

### Utilisation de la Loi Géométrique pour la Détection de Fraude

La loi géométrique modélise le nombre d'essais de Bernoulli nécessaires pour obtenir le premier succès. Dans notre contexte, si la 'durée jusqu'à la première fraude' (`time_to_fraud`) pour un compte suit une loi géométrique, nous pouvons estimer la probabilité d'un événement de fraude ('p') à chaque période, en supposant que le compte n'a pas encore été frauduleux.

Le temps moyen d'une distribution géométrique (nombre d'essais, incluant le succès) est de `1/p`. Étant donné que `time_to_fraud` représente le nombre de périodes *avant* la première fraude (où `time_to_fraud = 0` signifie fraude dès la première apparition), le nombre total de périodes jusqu'à la première fraude est `time_to_fraud + 1`.

Nous pouvons donc estimer le paramètre `p` comme suit :
`p = 1 / (moyenne_time_to_fraud + 1)`

Ce paramètre `p` représente la probabilité qu'un compte destinataire devienne frauduleux dans la prochaine période, étant donné qu'il n'a pas été frauduleux jusqu'à présent depuis sa première apparition. Cette valeur peut être une caractéristique discriminante très utile pour le modèle.

### Impact de `p_geometric_first_fraud` sur la séparation des classes

Nous allons d'abord visualiser la distribution de la nouvelle fonctionnalité `p_geometric_first_fraud` pour les transactions frauduleuses et non-frauduleuses. Une bonne séparation ici indiquerait que la fonctionnalité est très discriminante.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

# Prepare data for plotting p_geometric_first_fraud distribution
df_p_geo_plot = pd.DataFrame({
    'p_geometric_first_fraud': X_test_enh['p_geometric_first_fraud'],
    'fraud_flag': y_test_enh
})

plt.figure(figsize=(10, 6))
sns.boxplot(x='fraud_flag', y='p_geometric_first_fraud', data=df_p_geo_plot, palette='coolwarm')
plt.title('Distribution de p_geometric_first_fraud par Fraud Flag (Test Set)')
plt.xlabel('Fraud Flag (0 = Non-Fraude, 1 = Fraude)')
plt.ylabel('Probabilité Géométrique de Première Fraude (p_geometric_first_fraud)')
plt.grid(True, linestyle='--', alpha=0.7)
plt.show()

plt.figure(figsize=(10, 6))
sns.kdeplot(data=df_p_geo_plot, x='p_geometric_first_fraud', hue='fraud_flag', fill=True, common_norm=False, palette='coolwarm')
plt.title('Densité de p_geometric_first_fraud par Fraud Flag (Test Set)')
plt.xlabel('Probabilité Géométrique de Première Fraude (p_geometric_first_fraud)')
plt.ylabel('Densité')
plt.grid(True, linestyle='--', alpha=0.7)
plt.show()

### Comparaison des Distributions des Scores de Fraude (Probabilités) du Modèle Baseline et du Modèle Amélioré

Ensuite, nous allons comparer les distributions des probabilités de fraude prédites par le modèle de base et le modèle amélioré (qui inclut `p_geometric_first_fraud`). Idéalement, le modèle amélioré devrait assigner des probabilités plus élevées aux vraies fraudes et des probabilités plus faibles aux non-fraudes, ce qui se traduirait par une meilleure séparation des distributions.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

# Prepare data for plotting predicted probabilities
df_proba_plot = pd.DataFrame({
    'Actual Fraud Flag': y_test_enh, # y_test_base and y_test_enh should be identical
    'Baseline Predicted Proba': y_pred_proba_base,
    'Enhanced Predicted Proba': y_pred_proba_enh
})

# Plot distribution of predicted probabilities for Baseline Model
plt.figure(figsize=(12, 6))
sns.kdeplot(data=df_proba_plot, x='Baseline Predicted Proba', hue='Actual Fraud Flag', fill=True, common_norm=False, palette='viridis')
plt.title('Distribution des Probabilités de Fraude Prédites (Modèle Baseline)')
plt.xlabel('Probabilité Prédite de Fraude')
plt.ylabel('Densité')
plt.grid(True, linestyle='--', alpha=0.7)
plt.show()

# Plot distribution of predicted probabilities for Enhanced Model
plt.figure(figsize=(12, 6))
sns.kdeplot(data=df_proba_plot, x='Enhanced Predicted Proba', hue='Actual Fraud Flag', fill=True, common_norm=False, palette='viridis')
plt.title('Distribution des Probabilités de Fraude Prédites (Modèle Amélioré avec p_geometric_first_fraud)')
plt.xlabel('Probabilité Prédite de Fraude')
plt.ylabel('Densité')
plt.grid(True, linestyle='--', alpha=0.7)
plt.show()

In [ ]:
import pandas as pd
import numpy as np

# Ensure df is sorted by period for correct 'first appearance' and 'first fraud' periods
df_sorted_by_period = df.sort_values(by='period').copy()

# Function to get first appearance and first fraudulent period (copied from HBN81jf0MoXt)
def get_fraud_time_metrics(account_type_col, df_data):
    fraudulent_accounts = df_data[df_data['fraud_flag'] == 1][account_type_col].unique()
    time_to_fraud_list = []
    for account in fraudulent_accounts:
        account_txns = df_data[df_data[account_type_col] == account]
        first_appearance_period = account_txns['period'].min()
        first_fraud_period = account_txns[account_txns['fraud_flag'] == 1]['period'].min()
        time_to_fraud = first_fraud_period - first_appearance_period
        time_to_fraud_list.append({
            'account': account,
            'first_appearance_period': first_appearance_period,
            'first_fraud_period': first_fraud_period,
            'time_to_fraud': time_to_fraud
        })
    return pd.DataFrame(time_to_fraud_list)

# Generate destination_fraud_metrics (copied from HBN81jf0MoXt)
destination_fraud_metrics = get_fraud_time_metrics('destination_account', df_sorted_by_period)

# Calculate the 'p' parameter for the geometric distribution based on mean_time_to_fraud
mean_time_to_fraud_dest = destination_fraud_metrics['time_to_fraud'].mean()
p_geometric = 1 / (mean_time_to_fraud_dest + 1)

print(f"Le paramètre 'p' de la loi géométrique pour le 'time_to_fraud' des comptes destinataires est : {p_geometric:.4f}")

# We can now create a new feature for each transaction indicating this calculated 'p' for its destination account.
# This 'p' value would be a static feature for a given destination account based on its historical behavior prior to fraud.
# Alternatively, we could create dynamic features (e.g., probability of fraud in the next 'x' periods given current non-fraud status).

# For simplicity, let's add 'p_geometric_first_fraud' as a feature to the destination_fraud_metrics and then merge it back to the main DataFrame if needed for other analyses.
destination_fraud_metrics['p_geometric_first_fraud'] = destination_fraud_metrics['time_to_fraud'].apply(lambda x: 1 / (x + 1) if x >= 0 else 0) # Handle potential negative time_to_fraud if any, though unlikely

# MERGE p_geometric_first_fraud into the main df
df = pd.merge(df, destination_fraud_metrics[['account', 'p_geometric_first_fraud']].rename(columns={'account': 'destination_account'}), on='destination_account', how='left')
df['p_geometric_first_fraud'] = df['p_geometric_first_fraud'].fillna(0) # Fill for non-fraudulent accounts

print("\nFirst 5 rows of destination_fraud_metrics with the new 'p_geometric_first_fraud' feature:")
display(destination_fraud_metrics.head())
print("\nFirst 5 rows of df with the new 'p_geometric_first_fraud' feature:")
display(df[['id', 'destination_account', 'p_geometric_first_fraud']].head())

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt
import seaborn as sns

print("--- Model Performance Comparison: Baseline vs. Enhanced with p_geometric_first_fraud ---")

# Define the target variable
y = df['fraud_flag']

# Define a consistent set of baseline features
baseline_features = [
    'period',
    'amount',
    'origin_balance_before',
    'origin_balance_after',
    'destination_balance_before',
    'destination_balance_after',
    'origin_transaction_sequence',
    'destination_transaction_sequence',
    'origin_dest_pair_sequence',
    'origin_account_previously_fraud',
    'destination_account_previously_fraud'
]

# Ensure all baseline features exist in the DataFrame
baseline_features = [f for f in baseline_features if f in df.columns]

# --- Baseline Model Training and Evaluation ---
print("\n--- Training Baseline Model ---")
X_baseline = df[baseline_features]
X_baseline = X_baseline.fillna(0) # Fill NaNs for consistency

# Split data for baseline model
X_train_base, X_test_base, y_train_base, y_test_base = train_test_split(
    X_baseline, y, test_size=0.3, random_state=42, stratify=y
)

# Initialize and train RandomForestClassifier for baseline
rf_baseline = RandomForestClassifier(random_state=42, n_estimators=100, class_weight='balanced')
rf_baseline.fit(X_train_base, y_train_base)

# Make predictions
y_pred_base = rf_baseline.predict(X_test_base)
y_pred_proba_base = rf_baseline.predict_proba(X_test_base)[:, 1]

# Evaluate baseline model
accuracy_base = accuracy_score(y_test_base, y_pred_base)
precision_base = precision_score(y_test_base, y_pred_base)
recall_base = recall_score(y_test_base, y_pred_base)
f1_base = f1_score(y_test_base, y_pred_base)
roc_auc_base = roc_auc_score(y_test_base, y_pred_proba_base)

print(f"Baseline Model Performance:")
print(f"  Accuracy: {accuracy_base:.4f}")
print(f"  Precision: {precision_base:.4f}")
print(f"  Recall: {recall_base:.4f}")
print(f"  F1-Score: {f1_base:.4f}")
print(f"  ROC AUC: {roc_auc_base:.4f}")

# Confusion Matrix for Baseline
cm_base = confusion_matrix(y_test_base, y_pred_base)
plt.figure(figsize=(6, 5))
ConfusionMatrixDisplay(cm_base, display_labels=['Non-Fraud', 'Fraud']).plot(cmap='Blues')
plt.title('Baseline Model Confusion Matrix')
plt.show()

# --- Enhanced Model Training and Evaluation (with p_geometric_first_fraud) ---
print("\n--- Training Enhanced Model (with p_geometric_first_fraud) ---")
enhanced_features = baseline_features + ['p_geometric_first_fraud']

# Ensure all enhanced features exist in the DataFrame
enhanced_features = [f for f in enhanced_features if f in df.columns]

X_enhanced = df[enhanced_features]
X_enhanced = X_enhanced.fillna(0) # Fill NaNs for consistency

# Split data for enhanced model
X_train_enh, X_test_enh, y_train_enh, y_test_enh = train_test_split(
    X_enhanced, y, test_size=0.3, random_state=42, stratify=y
)

# Initialize and train RandomForestClassifier for enhanced
rf_enhanced = RandomForestClassifier(random_state=42, n_estimators=100, class_weight='balanced')
rf_enhanced.fit(X_train_enh, y_train_enh)

# Make predictions
y_pred_enh = rf_enhanced.predict(X_test_enh)
y_pred_proba_enh = rf_enhanced.predict_proba(X_test_enh)[:, 1]

# Evaluate enhanced model
accuracy_enh = accuracy_score(y_test_enh, y_pred_enh)
precision_enh = precision_score(y_test_enh, y_pred_enh)
recall_enh = recall_score(y_test_enh, y_pred_enh)
f1_enh = f1_score(y_test_enh, y_pred_enh)
roc_auc_enh = roc_auc_score(y_test_enh, y_pred_proba_enh)

print(f"Enhanced Model Performance:")
print(f"  Accuracy: {accuracy_enh:.4f}")
print(f"  Precision: {precision_enh:.4f}")
print(f"  Recall: {recall_enh:.4f}")
print(f"  F1-Score: {f1_enh:.4f}")
print(f"  ROC AUC: {roc_auc_enh:.4f}")

# Confusion Matrix for Enhanced
cm_enh = confusion_matrix(y_test_enh, y_pred_enh)
plt.figure(figsize=(6, 5))
ConfusionMatrixDisplay(cm_enh, display_labels=['Non-Fraud', 'Fraud']).plot(cmap='Blues')
plt.title('Enhanced Model Confusion Matrix')
plt.show()

# --- Comparison Summary ---
print("\n--- Performance Comparison ---")
print(f"Metric        | Baseline | Enhanced | Change")
print(f"----------------------------------------------")
print(f"Accuracy      | {accuracy_base:.4f}   | {accuracy_enh:.4f}   | {accuracy_enh - accuracy_base:+.4f}")
print(f"Precision     | {precision_base:.4f}   | {precision_enh:.4f}   | {precision_enh - precision_base:+.4f}")
print(f"Recall        | {recall_base:.4f}   | {recall_enh:.4f}   | {recall_enh - recall_base:+.4f}")
print(f"F1-Score      | {f1_base:.4f}   | {f1_enh:.4f}   | {f1_enh - f1_base:+.4f}")
print(f"ROC AUC       | {roc_auc_base:.4f}   | {roc_auc_enh:.4f}   | {roc_auc_enh - roc_auc_base:+.4f}")

print("\nObservation: The inclusion of 'p_geometric_first_fraud' has resulted in a change in model performance. Review the metrics above to understand the specific impact.")


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

# Get feature importances from the trained enhanced model
feature_importances_enh = rf_enhanced.feature_importances_
feature_names_enh = X_enhanced.columns

# Create a DataFrame for better visualization
importance_df_enh = pd.DataFrame({
    'Feature': feature_names_enh,
    'Importance': feature_importances_enh
})

# Sort by importance in descending order
importance_df_enh = importance_df_enh.sort_values(by='Importance', ascending=False)

# Plot feature importances
plt.figure(figsize=(12, 8))
sns.barplot(x='Importance', y='Feature', hue='Feature', data=importance_df_enh, palette='viridis', legend=False)
plt.title('Enhanced Model Feature Importance')
plt.xlabel('Importance (Mean Decrease in Impurity)')
plt.ylabel('Feature')
plt.tight_layout()
plt.show()

## old

In [ ]:
import numpy as np

numerical_cols = [
    'amount',
    'origin_balance_before',
    'origin_balance_after',
    'destination_balance_before',
    'destination_balance_after'
]

# Function to extract decimal properties
def extract_decimal_features(value):
    if pd.isna(value):
        return False, 0, 0
    s = str(value)

    # Handle scientific notation if any
    if 'e' in s or 'E' in s:
        # Convert to a standard decimal string representation
        s = f"{value:.10f}" # Use a reasonable precision
        s = s.rstrip('0').rstrip('.') # Remove trailing zeros and decimal if not needed

    has_decimal = '.' in s

    if has_decimal:
        parts = s.split('.')
        integer_part = parts[0]
        decimal_part = parts[1]
        num_decimal_places = len(decimal_part)
    else:
        integer_part = s
        num_decimal_places = 0

    num_digits_before_decimal = len(integer_part.lstrip('-'))

    return has_decimal, num_decimal_places, num_digits_before_decimal


for col in numerical_cols:
    # Apply the function to create new columns
    df[[f'{col}_has_decimal', f'{col}_num_decimal_places', f'{col}_num_digits_before_decimal']] = \
        df[col].apply(lambda x: pd.Series(extract_decimal_features(x)))

    # Ensure boolean and integer types
    df[f'{col}_has_decimal'] = df[f'{col}_has_decimal'].astype(bool)
    df[f'{col}_num_decimal_places'] = df[f'{col}_num_decimal_places'].astype(int)
    df[f'{col}_num_digits_before_decimal'] = df[f'{col}_num_digits_before_decimal'].astype(int)

# Create four columns for the first four digits of 'amount'
def get_first_four_digits(value):
    if pd.isna(value):
        return [np.nan] * 4
    s = str(value)

    # Handle scientific notation if any
    if 'e' in s or 'E' in s:
        s = f"{value:.10f}" # Convert to standard decimal representation

    # Remove sign and decimal point to get raw digits for extraction
    s_digits = s.replace('.', '').lstrip('-')

    digits = []
    for i in range(4):
        if i < len(s_digits):
            digits.append(int(s_digits[i]))
        else:
            digits.append(np.nan) # Use NaN for missing digits
    return digits

df[['amount_digit1', 'amount_digit2', 'amount_digit3', 'amount_digit4']] = \
    df['amount'].apply(lambda x: pd.Series(get_first_four_digits(x)))

print("New decimal and digit-based features created.")
print(df[['amount', 'amount_has_decimal', 'amount_num_decimal_places', 'amount_num_digits_before_decimal', 'amount_digit1', 'amount_digit2', 'amount_digit3', 'amount_digit4']].head())


### Analyse des nouvelles fonctionnalités par rapport à la fraude pour les opérations 'op_03'

Comme demandé, nous allons maintenant examiner la distribution des nouvelles fonctionnalités (`_has_decimal`, `_num_decimal_places`, `_num_digits_before_decimal`, et `amount_digitX`) spécifiquement pour les transactions de type `op_03`, en les comparant au `fraud_flag`.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Filter the DataFrame for operation == 'op_03'
df_op03 = df[df['operation'] == 'op_03'].copy()

print(f"Filtered DataFrame shape (only op_03): {df_op03.shape}")

#### Caractéristiques liées aux décimales (`_has_decimal`)

In [ ]:
decimal_bool_cols = [
    'amount_has_decimal',
    'origin_balance_before_has_decimal',
    'origin_balance_after_has_decimal',
    'destination_balance_before_has_decimal',
    'destination_balance_after_has_decimal'
]

for col in decimal_bool_cols:
    plt.figure(figsize=(8, 5))
    sns.countplot(x=col, hue='fraud_flag', data=df_op03, palette='pastel')
    plt.title(f'Distribution du Fraud Flag par {col} pour op_03')
    plt.xlabel(col)
    plt.ylabel('Count')
    plt.legend(title='Fraud Flag')
    plt.show()

#### Caractéristiques liées au nombre de décimales (`_num_decimal_places`)

In [ ]:
num_decimal_places_cols = [
    'amount_num_decimal_places',
    'origin_balance_before_num_decimal_places',
    'origin_balance_after_num_decimal_places',
    'destination_balance_before_num_decimal_places',
    'destination_balance_after_num_decimal_places'
]

for col in num_decimal_places_cols:
    plt.figure(figsize=(10, 6))
    sns.boxplot(x='fraud_flag', y=col, data=df_op03, palette='viridis')
    plt.title(f'Distribution de {col} par Fraud Flag pour op_03')
    plt.xlabel('Fraud Flag')
    plt.ylabel(col)
    plt.show()

#### Caractéristiques liées au nombre de chiffres avant la virgule (`_num_digits_before_decimal`)

In [ ]:
num_digits_before_decimal_cols = [
    'amount_num_digits_before_decimal',
    'origin_balance_before_num_digits_before_decimal',
    'origin_balance_after_num_digits_before_decimal',
    'destination_balance_before_num_digits_before_decimal',
    'destination_balance_after_num_digits_before_decimal'
]

for col in num_digits_before_decimal_cols:
    plt.figure(figsize=(10, 6))
    sns.boxplot(x='fraud_flag', y=col, data=df_op03, palette='magma')
    plt.title(f'Distribution de {col} par Fraud Flag pour op_03')
    plt.xlabel('Fraud Flag')
    plt.ylabel(col)
    plt.show()

#### Caractéristiques liées aux quatre premiers chiffres du montant (`amount_digitX`)

In [ ]:
amount_digit_cols = ['amount_digit1', 'amount_digit2', 'amount_digit3', 'amount_digit4']

for col in amount_digit_cols:
    plt.figure(figsize=(10, 6))
    sns.countplot(x=col, hue='fraud_flag', data=df_op03, palette='coolwarm')
    plt.title(f'Distribution du Fraud Flag par {col} pour op_03')
    plt.xlabel(col)
    plt.ylabel('Count')
    plt.legend(title='Fraud Flag')
    plt.show()

### Observations

Après avoir examiné les distributions des nouvelles fonctionnalités pour les transactions `op_03`:

*   **`_has_decimal` :** Il semble que pour la plupart des colonnes, la présence ou l'absence de décimales ait une certaine relation avec la `fraud_flag`. Par exemple, si une colonne est `False` pour `_has_decimal`, le nombre de fraudes peut être beaucoup plus faible.

*   **`_num_decimal_places` et `_num_digits_before_decimal` :** Ces caractéristiques numériques montrent des différences dans leurs distributions entre les transactions frauduleuses et non-frauduleuses. Les transactions frauduleuses peuvent avoir tendance à avoir un nombre plus ou moins élevé de décimales ou de chiffres avant la virgule, selon la colonne. Cela peut indiquer des schémas où les fraudeurs utilisent des montants ou des soldes sans décimales, ou avec un nombre spécifique de chiffres.

*   **`amount_digitX` :** Les distributions des premiers chiffres du montant (`amount_digit1` à `amount_digit4`) par rapport au `fraud_flag` peuvent révéler que certains chiffres sont plus fréquents dans les transactions frauduleuses ou non-frauduleuses. Cela est souvent lié à la Loi de Benford, qui stipule que certains chiffres (comme 1) apparaissent plus souvent en première position dans les nombres issus de données du monde réel, et les déviations de cette loi peuvent indiquer une manipulation.

Ces nouvelles fonctionnalités peuvent être très utiles pour l'entraînement d'un modèle de détection de fraude, car elles capturent des motifs qui ne sont pas directement apparents dans les montants bruts ou les soldes.

In [ ]:
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, roc_auc_score, confusion_matrix, average_precision_score
import matplotlib.pyplot as plt
import seaborn as sns

# Define the target variable
y = df['fraud_flag']

# Define the features to be used for the model
# These features include the newly engineered decimal and digit features, as well as existing numerical features.
features = [
    'period',
    'amount',
    'origin_balance_before',
    'origin_balance_after',
    'destination_balance_before',
    'destination_balance_after',
    'amount_has_decimal',
    'amount_num_decimal_places',
    'amount_num_digits_before_decimal',
    'origin_balance_before_has_decimal',
    'origin_balance_before_num_decimal_places',
    'origin_balance_before_num_digits_before_decimal',
    'origin_balance_after_has_decimal',
    'origin_balance_after_num_decimal_places',
    'origin_balance_after_num_digits_before_decimal',
    'destination_balance_before_has_decimal',
    'destination_balance_before_num_decimal_places',
    'destination_balance_before_num_digits_before_decimal',
    'destination_balance_after_has_decimal',
    'destination_balance_after_num_decimal_places',
    'destination_balance_after_num_digits_before_decimal',
    'amount_digit1', 'amount_digit2', 'amount_digit3', 'amount_digit4',
    'origin_account_ranked',
    'destination_account_ranked',
    'has_origin_timestamp',
    'has_destination_timestamp',
    'origin_account_previously_fraud',
    'destination_account_previously_fraud',
    'calc_origin_decimal',
    'calc_destination_decimal'
]

# Filter out any features that might not have been created yet
existing_features = [f for f in features if f in df.columns]
X = df[existing_features]

# Handle potential NaNs in the features (e.g., from hex conversions if not filled previously)
for col in ['calc_origin_decimal', 'calc_destination_decimal']:
    if col in X.columns:
        X[col] = X[col].fillna(0) # Fill NaN with 0 for numerical features

# Drop rows with any remaining NaNs in features (if any other feature has NaNs)
X = X.dropna()
y = y[X.index]

print(f"Using {len(existing_features)} features for training.")
print(f"Shape of X: {X.shape}, Shape of y: {y.shape}")

if X.empty:
    print("After data cleaning, no entries remain for training the model.")
elif len(y.unique()) < 2:
    print(f"Only one class present in the target variable. Cannot train a classifier. Unique classes: {y.unique()}")
else:
    # Split data into training and testing sets
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

    # Calculate scale_pos_weight for imbalanced classes
    neg_count = y_train.value_counts()[0]
    pos_count = y_train.value_counts()[1]
    scale_pos_weight_value = neg_count / pos_count
    print(f"Calculated scale_pos_weight: {scale_pos_weight_value:.2f}")

    # Initialize and train the XGBoost classifier with scale_pos_weight
    model_fraud_flag_stage2 = xgb.XGBClassifier(objective='binary:logistic', eval_metric='aucpr', use_label_encoder=False, random_state=42, scale_pos_weight=scale_pos_weight_value)
    model_fraud_flag_stage2.fit(X_train, y_train)

    # Make predictions on the test set
    y_pred = model_fraud_flag_stage2.predict(X_test)
    y_pred_proba = model_fraud_flag_stage2.predict_proba(X_test)[:, 1]

    # Evaluate the model
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    roc_auc = roc_auc_score(y_test, y_pred_proba)
    average_precision = average_precision_score(y_test, y_pred_proba)

    print(f"\n--- XGBoost Model Performance for Fraud Flag Prediction (all operations) with Class Weights ---")
    print(f"Accuracy: {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall: {recall:.4f}")
    print(f"ROC AUC: {roc_auc:.4f}")
    print(f"Average Precision (PR-AUC): {average_precision:.4f}")

    # Confusion Matrix
    cm = confusion_matrix(y_test, y_pred)
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False,
                xticklabels=['Predicted Non-Fraud (0)', 'Predicted Fraud (1)'],
                yticklabels=['Actual Non-Fraud (0)', 'Actual Fraud (1)'])
    plt.title('Confusion Matrix for XGBoost Fraud Prediction')
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.show()

In [ ]:
from sklearn.metrics import PrecisionRecallDisplay

# Create the Precision-Recall curve
plt.figure(figsize=(8, 6))
# Use the `from_estimator` method if the model is an estimator, otherwise `from_predictions`
display = PrecisionRecallDisplay.from_predictions(y_test, y_pred_proba, name="XGBoost Classifier")
display.plot(ax=plt.gca())
plt.title('Precision-Recall Curve')
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.grid(True)
plt.show()

### Hyperparameter Tuning with GridSearchCV

To further optimize the XGBoost model, we will use `GridSearchCV` to perform an exhaustive search over a specified parameter grid. This helps in finding the best combination of hyperparameters that maximizes the model's performance on a given metric (in this case, Average Precision).


In [ ]:
from sklearn.model_selection import GridSearchCV

# Define the parameter grid to search
# We'll focus on key hyperparameters that significantly impact XGBoost performance
param_grid = {
    'n_estimators': [100, 200, 300], # Number of boosting rounds
    'max_depth': [3, 5, 7],         # Maximum depth of a tree
    'learning_rate': [0.01, 0.1, 0.2], # Step size shrinkage to prevent overfitting
    'subsample': [0.7, 0.8, 1.0],   # Subsample ratio of the training instance
    'colsample_bytree': [0.7, 0.8, 1.0] # Subsample ratio of columns when constructing each tree
}

# Initialize the GridSearchCV object
# We will use 'aucpr' as the scoring metric as it's relevant for imbalanced datasets like fraud detection
grid_search = GridSearchCV(estimator=model_fraud_flag_stage2, # Use the previously defined XGBoost model
                           param_grid=param_grid,
                           scoring='average_precision', # Optimize for Average Precision (PR-AUC)
                           cv=3, # 3-fold cross-validation
                           verbose=2, # Output progress messages
                           n_jobs=-1) # Use all available CPU cores

print("Starting GridSearchCV...")
# Fit the GridSearchCV to the training data
grid_search.fit(X_train, y_train)
print("GridSearchCV completed.")

In [ ]:
# Print the best parameters found by GridSearchCV
print(f"Best parameters found: {grid_search.best_params_}")

# Print the best score achieved
print(f"Best Average Precision score: {grid_search.best_score_:.4f}")

### Evaluating the Best Model from GridSearchCV

Now, let's evaluate the performance of the model that achieved the best score during the Grid Search on the test set.

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, roc_auc_score, average_precision_score, confusion_matrix

# Get the best estimator from the Grid Search
best_model = grid_search.best_estimator_

# Make predictions on the test set using the best model
y_pred_best = best_model.predict(X_test)
y_pred_proba_best = best_model.predict_proba(X_test)[:, 1]

# Evaluate the best model
accuracy_best = accuracy_score(y_test, y_pred_best)
precision_best = precision_score(y_test, y_pred_best)
recall_best = recall_score(y_test, y_pred_best)
roc_auc_best = roc_auc_score(y_test, y_pred_proba_best)
average_precision_best = average_precision_score(y_test, y_pred_proba_best)

print(f"\n--- Tuned XGBoost Model Performance --- ")
print(f"Accuracy: {accuracy_best:.4f}")
print(f"Precision: {precision_best:.4f}")
print(f"Recall: {recall_best:.4f}")
print(f"ROC AUC: {roc_auc_best:.4f}")
print(f"Average Precision (PR-AUC): {average_precision_best:.4f}")

# Confusion Matrix for the best model
cm_best = confusion_matrix(y_test, y_pred_best)
plt.figure(figsize=(8, 6))
sns.heatmap(cm_best, annot=True, fmt='d', cmap='Blues', cbar=False,
            xticklabels=['Predicted Non-Fraud (0)', 'Predicted Fraud (1)'],
            yticklabels=['Actual Non-Fraud (0)', 'Actual Fraud (1)'])
plt.title('Confusion Matrix for Tuned XGBoost Fraud Prediction')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()

### Feature Importance for XGBoost Model

Feature importance quantifies the contribution of each feature to the model's predictions. A higher importance score indicates that the feature has a greater impact on the model's ability to predict the target variable. This can help in understanding which features are most relevant for fraud detection.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd # Ensure pd is imported for DataFrame

# Ensure best_model and X_train are available
if 'best_model' not in locals() and 'grid_search' in locals():
    best_model = grid_search.best_estimator_
elif 'best_model' not in locals() and 'grid_search' not in locals():
    print("Error: 'best_model' and 'grid_search' are not defined. Please run the XGBoost model training and GridSearchCV cells first.")
    # In a real scenario, you might want to raise an exception or sys.exit()
    # For now, we'll proceed, but subsequent lines might fail if dependencies are truly missing.

if 'X_train' not in locals():
    print("Error: 'X_train' is not defined. Please run the data splitting cell (e.g., 6ZSDBmnmDYnO) first.")

# Get feature importances from the best XGBoost model
if 'best_model' in locals() and 'X_train' in locals():
    feature_importances_xgb = best_model.feature_importances_
    feature_names_xgb = X_train.columns

    # Create a DataFrame for better visualization
    importance_df_xgb = pd.DataFrame({
        'Feature': feature_names_xgb,
        'Importance': feature_importances_xgb
    })

    # Sort by importance
    importance_df_xgb = importance_df_xgb.sort_values(by='Importance', ascending=False)

    # Plot feature importances
    plt.figure(figsize=(12, 8))
    sns.barplot(x='Importance', y='Feature', data=importance_df_xgb, palette='viridis')
    plt.title('XGBoost Feature Importance')
    plt.xlabel('Importance (F-score)')
    plt.ylabel('Feature')
    plt.tight_layout()
    plt.show()
else:
    print("Cannot plot XGBoost feature importance: Missing required variables.")

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import accuracy_score, precision_score, recall_score, roc_auc_score, average_precision_score, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
from catboost import CatBoostClassifier

# Filter the DataFrame for only 'op_03' transactions
df_op03_catboost = df[df['operation'] == 'op_03'].copy()

# Ensure that 'existing_features' is defined (from previous XGBoost training)
# Re-filter existing_features to ensure they are present in the op_03 subset
features_for_op03_catboost = [f for f in existing_features if f in df_op03_catboost.columns]

X_op03_catboost = df_op03_catboost[features_for_op03_catboost]
y_op03_catboost = df_op03_catboost['fraud_flag']

# Handle potential NaNs in the features for the op_03 subset
for col in ['calc_origin_decimal', 'calc_destination_decimal']:
    if col in X_op03_catboost.columns:
        X_op03_catboost[col] = X_op03_catboost[col].fillna(0) # Fill NaN with 0 for numerical features

# Drop rows with any remaining NaNs in features
X_op03_catboost = X_op03_catboost.dropna()
y_op03_catboost = y_op03_catboost[X_op03_catboost.index]

print(f"Using {len(features_for_op03_catboost)} features for CatBoost training on op_03 transactions.")
print(f"Shape of X_op03_catboost: {X_op03_catboost.shape}, Shape of y_op03_catboost: {y_op03_catboost.shape}")

if X_op03_catboost.empty:
    print("After data cleaning, no entries remain for training the CatBoost model on op_03.")
elif len(y_op03_catboost.unique()) < 2:
    print(f"Only one class present in the target variable for op_03 transactions. Cannot train a classifier. Unique classes: {y_op03_catboost.unique()}")
else:
    # Split data into training and testing sets for op_03
    X_train_op03, X_test_op03, y_train_op03, y_test_op03 = train_test_split(
        X_op03_catboost, y_op03_catboost, test_size=0.3, random_state=42, stratify=y_op03_catboost
    )

    # Calculate scale_pos_weight for imbalanced classes
    neg_count_op03 = y_train_op03.value_counts()[0]
    pos_count_op03 = y_train_op03.value_counts()[1]
    scale_pos_weight_value_op03 = neg_count_op03 / pos_count_op03
    print(f"Calculated scale_pos_weight for op_03: {scale_pos_weight_value_op03:.2f}")

    # CatBoost parameters grid
    catboost_param_grid = {
        'iterations': [100, 200], # Similar to n_estimators
        'depth': [5, 7],         # Similar to max_depth
        'learning_rate': [0.05, 0.1],
        'l2_leaf_reg': [1, 3],
        'scale_pos_weight': [scale_pos_weight_value_op03] # Add scale_pos_weight to the grid
    }

    # Initialize CatBoostClassifier without scale_pos_weight in the constructor
    cat_model = CatBoostClassifier(
        random_seed=42,
        verbose=0, # Suppress verbose output during training
        eval_metric='AUC:hints=skip_train~false' # Use AUC for evaluation during CV
    )

    print("\nStarting GridSearchCV for CatBoost on op_03 transactions...")
    grid_search_catboost = GridSearchCV(
        estimator=cat_model,
        param_grid=catboost_param_grid,
        scoring='average_precision', # Optimize for Average Precision
        cv=3, # 3-fold cross-validation
        verbose=1, # Output progress messages for GridSearchCV
        n_jobs=-1 # Use all available CPU cores
    )

    grid_search_catboost.fit(X_train_op03, y_train_op03)
    print("GridSearchCV for CatBoost completed.")

    # Print the best parameters found by GridSearchCV
    print(f"Best CatBoost parameters found: {grid_search_catboost.best_params_}")

    # Print the best score achieved
    print(f"Best CatBoost Average Precision score (CV): {grid_search_catboost.best_score_:.4f}")

    # Get the best estimator from the Grid Search
    best_cat_model = grid_search_catboost.best_estimator_

    # Make predictions on the test set using the best model
    y_pred_catboost = best_cat_model.predict(X_test_op03)
    y_pred_proba_catboost = best_cat_model.predict_proba(X_test_op03)[:, 1]

    # Evaluate the best CatBoost model
    accuracy_catboost = accuracy_score(y_test_op03, y_pred_catboost)
    precision_catboost = precision_score(y_test_op03, y_pred_catboost)
    recall_catboost = recall_score(y_test_op03, y_pred_catboost)
    roc_auc_catboost = roc_auc_score(y_test_op03, y_pred_proba_catboost)
    average_precision_catboost = average_precision_score(y_test_op03, y_pred_proba_catboost)

    print(f"\n--- Tuned CatBoost Model Performance on op_03 Test Set ---")
    print(f"Accuracy: {accuracy_catboost:.4f}")
    print(f"Precision: {precision_catboost:.4f}")
    print(f"Recall: {recall_catboost:.4f}")
    print(f"ROC AUC: {roc_auc_catboost:.4f}")
    print(f"Average Precision (PR-AUC): {average_precision_catboost:.4f}")

    # Confusion Matrix for the best CatBoost model
    cm_catboost = confusion_matrix(y_test_op03, y_pred_catboost)
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm_catboost, annot=True, fmt='d', cmap='Blues', cbar=False,
                xticklabels=['Predicted Non-Fraud (0)', 'Predicted Fraud (1)'],
                yticklabels=['Actual Non-Fraud (0)', 'Actual Fraud (1)'])
    plt.title('Confusion Matrix for Tuned CatBoost Fraud Prediction (op_03)')
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.show()

### Feature Importance for CatBoost Model

Similar to XGBoost, CatBoost provides feature importances that indicate how much each feature contributes to the model's predictive power. Visualizing these importances helps in comparing the relevance of features across different models and understanding the drivers of fraud detection in the CatBoost model.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd # Ensure pd is imported for DataFrame

# Ensure best_cat_model and X_train_op03 are available
if 'best_cat_model' not in locals() and 'grid_search_catboost' in locals():
    best_cat_model = grid_search_catboost.best_estimator_
elif 'best_cat_model' not in locals() and 'grid_search_catboost' not in locals():
    print("Error: 'best_cat_model' and 'grid_search_catboost' are not defined. Please run the CatBoost model training and GridSearchCV cells first.")

if 'X_train_op03' not in locals():
    print("Error: 'X_train_op03' is not defined. Please run the CatBoost data splitting cell (e.g., 4umQh5fWfHoe) first.")

# Get feature importances from the best CatBoost model
if 'best_cat_model' in locals() and 'X_train_op03' in locals():
    feature_importances_cat = best_cat_model.get_feature_importance()
    feature_names_cat = X_train_op03.columns

    # Create a DataFrame for better visualization
    importance_df_cat = pd.DataFrame({
        'Feature': feature_names_cat,
        'Importance': feature_importances_cat
    })

    # Sort by importance
    importance_df_cat = importance_df_cat.sort_values(by='Importance', ascending=False)

    # Plot feature importances
    plt.figure(figsize=(12, 8))
    sns.barplot(x='Importance', y='Feature', data=importance_df_cat, palette='plasma')
    plt.title('CatBoost Feature Importance')
    plt.xlabel('Importance')
    plt.ylabel('Feature')
    plt.tight_layout()
    plt.show()
else:
    print("Cannot plot CatBoost feature importance: Missing required variables.")

# Echec

In [ ]:
import datetime

# --- Start of Feature Engineering for required columns ---

# From 8G8d_AAGQQK4 for ranked accounts
df['origin_account_stripped'] = df['origin_account'].str.replace('acc_o_', '')
df['destination_account_stripped'] = df['destination_account'].str.replace('acc_d_', '')

all_stripped_accounts = pd.concat([
    df['origin_account_stripped'],
    df['destination_account_stripped']
]).unique()

sorted_accounts = pd.Series(all_stripped_accounts).sort_values().reset_index(drop=True)

account_to_rank_mapping = {account_id: rank + 1 for rank, account_id in enumerate(sorted_accounts)}

df['origin_account_ranked'] = df['origin_account_stripped'].map(account_to_rank_mapping)
df['destination_account_ranked'] = df['destination_account_stripped'].map(account_to_rank_mapping)

# From lgU5THmiofow for decimal, ipv4, mac, timestamp conversions
def clean_hex(account_id):
    if pd.isna(account_id):
        return ""
    cleaned = str(account_id).replace("acc_o_", "").replace("acc_d_", "").strip()
    return cleaned

def hex_to_decimal(hex_str):
    try:
        return int(hex_str, 16) if hex_str else None
    except ValueError:
        return None

def hex_to_ipv4(hex_str):
    try:
        if len(hex_str) == 16:
            hex_ip = hex_str[8:]
            return ".".join(str(int(hex_ip[i:i+2], 16)) for i in range(0, 8, 2))
    except Exception:
        pass
    return None

def hex_to_mac(hex_str):
    try:
        if len(hex_str) == 16:
            hex_mac = hex_str[4:]
            return ":".join(hex_mac[i:i+2].upper() for i in range(0, 12, 2))
    except Exception:
        pass
    return None

def hex_to_timestamp(hex_str):
    try:
        if hex_str:
            val_dec = int(hex_str, 16)
            ts_sec = val_dec / 1_000_000_000
            return datetime.datetime.fromtimestamp(ts_sec, datetime.timezone.utc)
    except Exception:
        pass
    return pd.NaT

df['origin_hex_clean'] = df['origin_account'].apply(clean_hex)
df['destination_hex_clean'] = df['destination_account'].apply(clean_hex)

df['calc_origin_decimal']   = df['origin_hex_clean'].apply(hex_to_decimal)
df['calc_origin_ipv4']      = df['origin_hex_clean'].apply(hex_to_ipv4)
df['calc_origin_mac']       = df['origin_hex_clean'].apply(hex_to_mac)
df['calc_origin_timestamp'] = df['origin_hex_clean'].apply(hex_to_timestamp)

df['calc_destination_decimal']   = df['destination_hex_clean'].apply(hex_to_decimal)
df['calc_destination_ipv4']      = df['destination_hex_clean'].apply(hex_to_ipv4)
df['calc_destination_mac']       = df['destination_hex_clean'].apply(hex_to_mac)
df['calc_destination_timestamp'] = df['destination_hex_clean'].apply(hex_to_timestamp)

df.drop(columns=['origin_hex_clean', 'destination_hex_clean'], inplace=True, errors='ignore')


# From de7fe026 for has_timestamp flags
df['has_origin_timestamp'] = df['calc_origin_timestamp'].notna()
df['has_destination_timestamp'] = df['calc_destination_timestamp'].notna()


# From nkv6k2TX5OPZ for previously_fraud flags
# Identify unique origin accounts involved in fraud
fraudulent_origin_accounts = df[df['fraud_flag'] == 1]['origin_account'].unique()
# Identify unique destination accounts involved in fraud
fraudulent_destination_accounts = df[df['fraud_flag'] == 1]['destination_account'].unique()
# Create a set of all accounts that have ever committed fraud (origin or destination)
all_fraudulent_accounts = set(fraudulent_origin_accounts).union(set(fraudulent_destination_accounts))
# Create new columns for 'origin_account_previously_fraud' and 'destination_account_previously_fraud'
df['origin_account_previously_fraud'] = df['origin_account'].isin(all_fraudulent_accounts).astype(int)
df['destination_account_previously_fraud'] = df['destination_account'].isin(all_fraudulent_accounts).astype(int)

# --- End of Feature Engineering ---

In [ ]:
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, roc_auc_score

df = df[df['operation']=='op_03'].copy() # Ensure df has all features from previous cells

# Define the target variable
y_previously_fraud = df['destination_account_previously_fraud']

# Define features to predict 'destination_account_previously_fraud'
# Exclude 'fraud_flag' itself and the target variable, and other direct identifiers
features_for_prev_fraud = [
    'period',
    'amount',
    'origin_balance_before',
    'origin_balance_after',
    'destination_balance_before',
    'destination_balance_after',
    'origin_account_ranked',
    'destination_account_ranked',
    'has_origin_timestamp',
    'has_destination_timestamp',
    'origin_account_previously_fraud',
    'calc_origin_decimal',
    'calc_destination_decimal'
]

# Ensure all features exist and handle potential NaNs before splitting
# These should have been handled by the general feature engineering now, but ensure consistency
for col in ['calc_origin_decimal', 'calc_destination_decimal']:
    if col in df.columns:
        df[col] = df[col].fillna(0)

# All features in features_for_prev_fraud should now be present in df
X_previously_fraud = df[features_for_prev_fraud]

# Drop rows with any remaining NaNs in features (if any other feature has NaNs)
X_previously_fraud = X_previously_fraud.dropna()
y_previously_fraud = y_previously_fraud[X_previously_fraud.index]

if X_previously_fraud.empty:
    print("After data cleaning, no entries remain for training the 'previously fraudulent' model.")
elif len(y_previously_fraud.unique()) < 2: # Check if there's enough variation in the target
    print(f"Only one class present in the target variable ('destination_account_previously_fraud'). Cannot train a classifier.")
else:
    # Split data into training and testing sets
    X_train_pf, X_test_pf, y_train_pf, y_test_pf = train_test_split(X_previously_fraud, y_previously_fraud, test_size=0.3, random_state=42, stratify=y_previously_fraud)

    # Train an XGBoost classifier
    model_previously_fraud = xgb.XGBClassifier(objective='binary:logistic', eval_metric='logloss', use_label_encoder=False, random_state=42)
    model_previously_fraud.fit(X_train_pf, y_train_pf)

    # Evaluate the model
    y_pred_pf = model_previously_fraud.predict(X_test_pf)
    y_pred_proba_pf = model_previously_fraud.predict_proba(X_test_pf)[:, 1]

    accuracy_pf = accuracy_score(y_test_pf, y_pred_pf)
    precision_pf = precision_score(y_test_pf, y_pred_pf)
    recall_pf = recall_score(y_test_pf, y_pred_pf)
    roc_auc_pf = roc_auc_score(y_test_pf, y_pred_proba_pf)

    print(f"\n--- Model to Predict Previously Fraudulent Destination Accounts ---")
    print(f"Accuracy: {accuracy_pf:.4f}")
    print(f"Precision: {precision_pf:.4f}")
    print(f"Recall: {recall_pf:.4f}")
    print(f"ROC AUC: {roc_auc_pf:.4f}")

    from sklearn.metrics import confusion_matrix
    import seaborn as sns
    import matplotlib.pyplot as plt

    cm_pf = confusion_matrix(y_test_pf, y_pred_pf)

    plt.figure(figsize=(8, 6))
    sns.heatmap(cm_pf, annot=True, fmt='d', cmap='Blues', cbar=False,
                xticklabels=['Not Previously Fraud (0)', 'Previously Fraud (1)'],
                yticklabels=['Actual Not Previously Fraud (0)', 'Actual Previously Fraud (1)'])
    plt.title('Confusion Matrix for Previously Fraudulent Destination Account Prediction')
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.show()

In [ ]:
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, roc_auc_score, confusion_matrix
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# --- Stage 1: Predict 'destination_account_previously_fraud' on the entire dataset ---
print("\n--- Stage 1: Predicting 'destination_account_previously_fraud' ---")

# df should already be filtered to 'op_03' and have all engineered features from IchhF-QQfQud
df_stage1 = df.copy()

# Define features for the first model. `features_for_prev_fraud` is from cell IchhF-QQfQud.
# Assuming features_for_prev_fraud is already in the kernel from previous cell (IchhF-QQfQud)
if 'features_for_prev_fraud' not in locals():
    features_for_prev_fraud = [
        'period',
        'amount',
        'origin_balance_before',
        'origin_balance_after',
        'destination_balance_before',
        'destination_balance_after',
        'origin_account_ranked',
        'destination_account_ranked',
        'has_origin_timestamp',
        'has_destination_timestamp',
        'origin_account_previously_fraud',
        'calc_origin_decimal',
        'calc_destination_decimal'
    ]

X_full_for_prev_fraud_prediction = df_stage1[features_for_prev_fraud]

# Predict 'destination_account_previously_fraud' using the trained model
# 'model_previously_fraud' is from cell IchhF-QQfQud
if 'model_previously_fraud' not in locals():
    print("Error: 'model_previously_fraud' not found. Please run cell IchhF-QQfQud first.")
else:
    predicted_dest_prev_fraud = model_previously_fraud.predict(X_full_for_prev_fraud_prediction)
    df_stage1['predicted_destination_account_previously_fraud'] = predicted_dest_prev_fraud

    # Evaluate Stage 1 predictions against true 'destination_account_previously_fraud' (for metrics)
    y_true_dest_prev_fraud = df_stage1['destination_account_previously_fraud']
    accuracy_s1 = accuracy_score(y_true_dest_prev_fraud, predicted_dest_prev_fraud)
    precision_s1 = precision_score(y_true_dest_prev_fraud, predicted_dest_prev_fraud)
    recall_s1 = recall_score(y_true_dest_prev_fraud, predicted_dest_prev_fraud)
    roc_auc_s1 = roc_auc_score(y_true_dest_prev_fraud, model_previously_fraud.predict_proba(X_full_for_prev_fraud_prediction)[:, 1])

    print(f"Stage 1 Metrics (predicting destination_account_previously_fraud):")
    print(f"  Accuracy: {accuracy_s1:.4f}")
    print(f"  Precision: {precision_s1:.4f}")
    print(f"  Recall: {recall_s1:.4f}")
    print(f"  ROC AUC: {roc_auc_s1:.4f}")

    # Initialize final hierarchical prediction columns
    df_stage1['hierarchical_predicted_fraud_flag'] = 0 # For hard predictions
    df_stage1['fraud_probability'] = 0.0 # For soft predictions (submission file)

    # --- Stage 2: Train a new model for 'fraud_flag' on the filtered subset ---
    print("\n--- Stage 2: Training model for 'fraud_flag' on susceptible op_03 transactions ---")

    # Filter data where predicted_destination_account_previously_fraud is 1 AND operation is 'op_03'
    # This explicitly incorporates the domain knowledge that fraud only occurs in 'op_03'
    df_susceptible_op03_fraud = df_stage1[
        (df_stage1['predicted_destination_account_previously_fraud'] == 1)
    ].copy()

    if df_susceptible_op03_fraud.empty:
        print("No 'op_03' transactions predicted as susceptible to fraud. Stage 2 model will not be trained.")
        # All fraud_probability and hierarchical_predicted_fraud_flag will remain 0, which is correct in this case.
    else:
        print(f"Number of 'op_03' transactions susceptible to fraud: {len(df_susceptible_op03_fraud)}")

        # Define features for the Stage 2 model. Using features_for_prev_fraud (without target) for consistency.
        features_for_stage2_model = [f for f in features_for_prev_fraud if f != 'destination_account_previously_fraud'] # Remove if it was mistakenly included before

        X_stage2 = df_susceptible_op03_fraud[features_for_stage2_model]
        y_stage2 = df_susceptible_op03_fraud['fraud_flag']

        if X_stage2.empty or len(y_stage2.unique()) < 2:
            print("Insufficient data or only one class in susceptible 'op_03' transactions for Stage 2 model. All fraud probabilities for these will be 0.")
        else:
            X_train_s2, X_test_s2, y_train_s2, y_test_s2 = train_test_split(
                X_stage2, y_stage2, test_size=0.3, random_state=42, stratify=y_stage2
            )

            model_fraud_flag_stage2 = xgb.XGBClassifier(objective='binary:logistic', eval_metric='logloss', use_label_encoder=False, random_state=42)
            model_fraud_flag_stage2.fit(X_train_s2, y_train_s2)

            y_pred_s2 = model_fraud_flag_stage2.predict(X_test_s2)
            y_pred_proba_s2 = model_fraud_flag_stage2.predict_proba(X_test_s2)[:, 1]

            accuracy_s2 = accuracy_score(y_test_s2, y_pred_s2)
            precision_s2 = precision_score(y_test_s2, y_pred_s2)
            recall_s2 = recall_score(y_test_s2, y_pred_s2)
            roc_auc_s2 = roc_auc_score(y_test_s2, y_pred_proba_s2)

            print(f"Stage 2 Metrics (predicting fraud_flag on susceptible 'op_03' subset):")
            print(f"  Accuracy: {accuracy_s2:.4f}")
            print(f"  Precision: {precision_s2:.4f}")
            print(f"  Recall: {recall_s2:.4f}")
            print(f"  ROC AUC: {roc_auc_s2:.4f}")

            # Predict fraud probabilities for the *entire* susceptible 'op_03' subset
            predicted_fraud_proba_for_susceptible_op03 = model_fraud_flag_stage2.predict_proba(X_stage2)[:, 1]
            df_stage1.loc[df_susceptible_op03_fraud.index, 'fraud_probability'] = predicted_fraud_proba_for_susceptible_op03

            # Update hierarchical_predicted_fraud_flag for the entire susceptible 'op_03' subset
            # (using hard predictions from the stage 2 model)
            predicted_fraud_flag_for_susceptible_op03 = model_fraud_flag_stage2.predict(X_stage2)
            df_stage1.loc[df_susceptible_op03_fraud.index, 'hierarchical_predicted_fraud_flag'] = predicted_fraud_flag_for_susceptible_op03

    # --- Global Metrics for the Hierarchical Model ---
    print("\n--- Global Metrics for the Hierarchical Prediction Model ---")
    y_true_global = df_stage1['fraud_flag']
    y_pred_global = df_stage1['hierarchical_predicted_fraud_flag']
    y_proba_global = df_stage1['fraud_probability']

    global_accuracy = accuracy_score(y_true_global, y_pred_global)

    # Handle potential 0 division for precision_score if no positive predictions are made
    if (y_pred_global == 1).sum() > 0:
        global_precision = precision_score(y_true_global, y_pred_global)
    else:
        global_precision = 0.0 # If no fraud is predicted, precision is 0

    global_recall = recall_score(y_true_global, y_pred_global)
    global_roc_auc = roc_auc_score(y_true_global, y_proba_global)


    print(f"Global Accuracy: {global_accuracy:.4f}")
    print(f"Global Precision: {global_precision:.4f}")
    print(f"Global Recall: {global_recall:.4f}")
    print(f"Global ROC AUC (using probabilities): {global_roc_auc:.4f}")

    # Confusion Matrix for Global Model
    cm_global = confusion_matrix(y_true_global, y_pred_global)
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm_global, annot=True, fmt='d', cmap='Blues', cbar=False,
                xticklabels=['Predicted Non-Fraud (0)', 'Predicted Fraud (1)'],
                yticklabels=['Actual Non-Fraud (0)', 'Actual Fraud (1)'])
    plt.title('Global Confusion Matrix for Hierarchical Fraud Prediction')
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.show()

### sumision

In [ ]:
test = pd.read_csv('/content/drive/MyDrive/CSV/test.csv')

In [ ]:
import pandas as pd
import datetime

# --- Feature Engineering Functions (re-defined for clarity and reusability) ---
def clean_hex(account_id):
    if pd.isna(account_id):
        return ""
    cleaned = str(account_id).replace("acc_o_", "").replace("acc_d_", "").strip()
    return cleaned

def hex_to_decimal(hex_str):
    try:
        return int(hex_str, 16) if hex_str else None
    except ValueError:
        return None

def hex_to_ipv4(hex_str):
    try:
        if len(hex_str) == 16:
            hex_ip = hex_str[8:]
            return ".".join(str(int(hex_ip[i:i+2], 16)) for i in range(0, 8, 2))
    except Exception:
        pass
    return None

def hex_to_mac(hex_str):
    try:
        if len(hex_str) == 16:
            hex_mac = hex_str[4:]
            return ":".join(hex_mac[i:i+2].upper() for i in range(0, 12, 2))
    except Exception:
        pass
    return None

def hex_to_timestamp(hex_str):
    try:
        if hex_str:
            val_dec = int(hex_str, 16)
            ts_sec = val_dec / 1_000_000_000
            return datetime.datetime.fromtimestamp(ts_sec, datetime.timezone.utc)
    except Exception:
        pass
    return pd.NaT

# Ensure 'test' DataFrame is available
if 'test' not in locals():
    print("Error: 'test' DataFrame not found. Please ensure test.csv is loaded.")
    # Assuming test DataFrame is loaded in a previous cell based on context
    # test = pd.read_csv('/content/drive/MyDrive/CSV/test.csv') # Uncomment if needed

# Make a copy to avoid modifying the original 'test' if it's used elsewhere
test_submission = test.copy()

# --- Feature Engineering for Test Data (aligned with training data processing) ---

# 1. Account stripping and ranking (from 8G8d_AAGQQK4)
# Ensure 'account_to_rank_mapping' is available from the training data processing
# Regenerate if not in current scope, assuming 'df' (training data) is available
if 'account_to_rank_mapping' not in locals() or 'df' not in locals():
    print("Warning: 'account_to_rank_mapping' or 'df' not found. Regenerating from df...")
    df_temp_for_mapping = df.copy() # Use a copy to avoid side effects if df is used elsewhere
    df_temp_for_mapping['origin_account_stripped'] = df_temp_for_mapping['origin_account'].str.replace('acc_o_', '')
    df_temp_for_mapping['destination_account_stripped'] = df_temp_for_mapping['destination_account'].str.replace('acc_d_', '')
    all_stripped_accounts = pd.concat([
        df_temp_for_mapping['origin_account_stripped'],
        df_temp_for_mapping['destination_account_stripped']
    ]).unique()
    sorted_accounts = pd.Series(all_stripped_accounts).sort_values().reset_index(drop=True)
    account_to_rank_mapping = {account_id: rank + 1 for rank, account_id in enumerate(sorted_accounts)}

test_submission['origin_account_stripped'] = test_submission['origin_account'].str.replace('acc_o_', '')
test_submission['destination_account_stripped'] = test_submission['destination_account'].str.replace('acc_d_', '')

test_submission['origin_account_ranked'] = test_submission['origin_account_stripped'].map(account_to_rank_mapping).fillna(0).astype(int)
test_submission['destination_account_ranked'] = test_submission['destination_account_stripped'].map(account_to_rank_mapping).fillna(0).astype(int)

# 2. Hex to decimal, ipv4, mac, timestamp conversions (from lgU5THmiofow)
test_submission['origin_hex_clean'] = test_submission['origin_account'].apply(clean_hex)
test_submission['destination_hex_clean'] = test_submission['destination_account'].apply(clean_hex)

test_submission['calc_origin_decimal']   = test_submission['origin_hex_clean'].apply(hex_to_decimal)
test_submission['calc_origin_ipv4']      = test_submission['origin_hex_clean'].apply(hex_to_ipv4)
test_submission['calc_origin_mac']       = test_submission['origin_hex_clean'].apply(hex_to_mac)
test_submission['calc_origin_timestamp'] = test_submission['origin_hex_clean'].apply(hex_to_timestamp)

test_submission['calc_destination_decimal']   = test_submission['destination_hex_clean'].apply(hex_to_decimal)
test_submission['calc_destination_ipv4']      = test_submission['destination_hex_clean'].apply(hex_to_ipv4)
test_submission['calc_destination_mac']       = test_submission['destination_hex_clean'].apply(hex_to_mac)
test_submission['calc_destination_timestamp'] = test_submission['destination_hex_clean'].apply(hex_to_timestamp)

test_submission.drop(columns=['origin_hex_clean', 'destination_hex_clean'], inplace=True, errors='ignore')

# 3. has_timestamp flags (from de7fe026)
test_submission['has_origin_timestamp'] = test_submission['calc_origin_timestamp'].notna()
test_submission['has_destination_timestamp'] = test_submission['calc_destination_timestamp'].notna()

# 4. previously_fraud flags (from nkv6k2TX5OPZ)
# Ensure 'all_fraudulent_accounts' is available from the training data processing
if 'fraudulent_origin_accounts' not in locals() or 'fraudulent_destination_accounts' not in locals() or 'df' not in locals():
    print("Warning: Fraudulent account lists or 'df' not found. Regenerating from df...")
    fraudulent_origin_accounts = df[df['fraud_flag'] == 1]['origin_account'].unique()
    fraudulent_destination_accounts = df[df['fraud_flag'] == 1]['destination_account'].unique()
all_fraudulent_accounts = set(fraudulent_origin_accounts).union(set(fraudulent_destination_accounts))

test_submission['origin_account_previously_fraud'] = test_submission['origin_account'].isin(all_fraudulent_accounts).astype(int)
test_submission['destination_account_previously_fraud'] = test_submission['destination_account'].isin(all_fraudulent_accounts).astype(int)

# --- Define features for models (same as training) ---
features_for_prev_fraud = [
    'period',
    'amount',
    'origin_balance_before',
    'origin_balance_after',
    'destination_balance_before',
    'destination_balance_after',
    'origin_account_ranked',
    'destination_account_ranked',
    'has_origin_timestamp',
    'has_destination_timestamp',
    'origin_account_previously_fraud',
    'calc_origin_decimal',
    'calc_destination_decimal'
]

features_for_stage2_model = [f for f in features_for_prev_fraud if f != 'destination_account_previously_fraud']

# Handle potential NaNs in 'calc_origin_decimal', 'calc_destination_decimal' for test data
for col in ['calc_origin_decimal', 'calc_destination_decimal']:
    if col in test_submission.columns:
        test_submission[col] = test_submission[col].fillna(0)

# --- Stage 1 Prediction: Predict 'destination_account_previously_fraud' ---
# Ensure model is available
if 'model_previously_fraud' not in locals():
    print("Error: 'model_previously_fraud' not found. Please train Stage 1 model first.")
else:
    X_test_stage1 = test_submission[features_for_prev_fraud]
    test_submission['predicted_destination_account_previously_fraud'] = model_previously_fraud.predict(X_test_stage1)
    test_submission['fraud_probability'] = 0.0 # Initialize probabilities to 0 (very probably normal)

    # --- Stage 2 Prediction: For susceptible transactions, predict actual fraud_flag ---
    # Identify transactions that are susceptible to fraud according to Stage 1 model
    susceptible_transactions_test = test_submission[test_submission['predicted_destination_account_previously_fraud'] == 1].copy()

    if not susceptible_transactions_test.empty:
        # Ensure model is available
        if 'model_fraud_flag_stage2' not in locals():
            print("Error: 'model_fraud_flag_stage2' not found. Please train Stage 2 model first.")
        else:
            X_test_stage2 = susceptible_transactions_test[features_for_stage2_model]

            # Predict probabilities for fraud_flag for the susceptible subset
            predicted_fraud_proba_stage2 = model_fraud_flag_stage2.predict_proba(X_test_stage2)[:, 1]

            # Update the 'fraud_probability' for the susceptible transactions in the main test_submission df
            test_submission.loc[susceptible_transactions_test.index, 'fraud_probability'] = predicted_fraud_proba_stage2
    else:
        print("No transactions predicted as susceptible to fraud by Stage 1 model. All target probabilities will be 0.")

    # --- Create Submission File ---
    submission = test_submission[['id', 'fraud_probability']].rename(columns={'fraud_probability': 'target'})

    # Save the submission file
    submission.to_csv('submission.csv', index=False)

    print("Submission file 'submission.csv' created successfully using the hierarchical model.")
    print(submission.head())

## FAST TESTING 02

In [ ]:
# This cell's content has been moved to FF1OP4G1u4uH to ensure feature engineering happens earlier.

### Visualisation de la Matrice de Confusion

Pour analyser plus en détail les performances du modèle, notamment les faux négatifs, nous allons visualiser la matrice de confusion. La matrice de confusion nous permettra de voir le nombre de vrais positifs, vrais négatifs, faux positifs et faux négatifs.

In [ ]:
from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

# Calcul de la matrice de confusion
cm = confusion_matrix(y_test, y_pred)

# Visualisation de la matrice de confusion
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False,
            xticklabels=['Non-Fraude Prédit (0)', 'Fraude Prédite (1)'],
            yticklabels=['Non-Fraude Réelle (0)', 'Fraude Réelle (1)'])
plt.title('Matrice de Confusion pour le Modèle XGBoost')
plt.xlabel('Prédiction')
plt.ylabel('Valeur Réelle')
plt.show()

# Extraction des valeurs pour une analyse plus facile
true_negatives = cm[0, 0]
false_positives = cm[0, 1]
false_negatives = cm[1, 0]
true_positives = cm[1, 1]

print(f"\nVrais Négatifs (True Negatives - TN): {true_negatives}")
print(f"Faux Positifs (False Positives - FP): {false_positives}")
print(f"Faux Négatifs (False Negatives - FN): {false_negatives}")
print(f"Vrais Positifs (True Positives - TP): {true_positives}")

print("\nAnalyse des Faux Négatifs (FN): Ce sont les transactions qui étaient réellement frauduleuses (Fraud Flag = 1) mais que le modèle a prédites comme non-frauduleuses (Fraud Flag = 0). Un nombre élevé de faux négatifs indique que le modèle manque de nombreuses fraudes réelles. C'est souvent un point critique dans la détection de fraude.")
print("Analyse des Faux Positifs (FP): Ce sont les transactions qui étaient non-frauduleuses (Fraud Flag = 0) mais que le modèle a prédites comme frauduleuses (Fraud Flag = 1). Un nombre élevé de faux positifs peut entraîner des investigations inutiles ou un rejet erroné de transactions légitimes.")

## EDA 02

In [ ]:
df.columns

In [ ]:
unique_op03_destination_accounts = df[df['operation'] == 'op_03']['destination_account'].unique()

accounts_with_other_operations = []

for account in unique_op03_destination_accounts:
    # Get all operations for the current destination account
    operations_for_account = df[df['destination_account'] == account]['operation'].unique()

    # Check if there are operations other than 'op_03'
    other_operations = [op for op in operations_for_account if op != 'op_03']

    if other_operations:
        accounts_with_other_operations.append({
            'destination_account': account,
            'other_operations': other_operations
        })

if accounts_with_other_operations:
    print(f"Number of destination accounts involved in 'op_03' and other operations: {len(accounts_with_other_operations)}")
    print("Details for some of these accounts:")
    for i, acc_info in enumerate(accounts_with_other_operations[:5]): # Print first 5 for brevity
        print(f"  Account: {acc_info['destination_account']}, Other Operations: {acc_info['other_operations']}")
else:
    print("No destination accounts involved in 'op_03' are also involved in other operations.")

In [ ]:
df.head()

In [ ]:
df['origin_account_stripped'] = df['origin_account'].str.replace('acc_o_', '')
df['destination_account_stripped'] = df['destination_account'].str.replace('acc_d_', '')

# Combine all unique stripped account IDs from both origin and destination
all_stripped_accounts = pd.concat([
    df['origin_account_stripped'],
    df['destination_account_stripped']
]).unique()

# Sort the unique accounts alphabetically
sorted_accounts = pd.Series(all_stripped_accounts).sort_values().reset_index(drop=True)

# Create a mapping from account ID to its numerical rank (starting from 1)
account_to_rank_mapping = {account_id: rank + 1 for rank, account_id in enumerate(sorted_accounts)}

# Apply the mapping to create new ranked columns in df
df['origin_account_ranked'] = df['origin_account_stripped'].map(account_to_rank_mapping)
df['destination_account_ranked'] = df['destination_account_stripped'].map(account_to_rank_mapping)

print("New columns 'origin_account_ranked' and 'destination_account_ranked' created.")
print("First 5 rows with new ranked account IDs:")
print(df[['origin_account', 'origin_account_stripped', 'origin_account_ranked',
          'destination_account', 'destination_account_stripped', 'destination_account_ranked']].head())

In [ ]:
import pandas as pd

# Ensure 'destination_account_stripped' is available
df['destination_account_stripped'] = df['destination_account'].str.replace('acc_d_', '')
df['origin_account_stripped'] = df['origin_account'].str.replace('acc_o_', '')

# Step 1: Create a summary DataFrame for unique destination accounts
destination_summary = df.groupby('destination_account').agg(
    fraud_count=('fraud_flag', 'sum'),
    total_occurrences=('destination_account', 'size')
).reset_index()

# Add the stripped account ID to the summary DataFrame.
account_stripped_mapping = df[['destination_account', 'destination_account_stripped']].drop_duplicates()
destination_summary = pd.merge(destination_summary, account_stripped_mapping, on='destination_account', how='left')

print("Résumé des comptes destinataires uniques:")
print(destination_summary.head())

# Create a summary DataFrame for unique origin accounts
origin_summary = df.groupby('origin_account').agg(
    fraud_count=('fraud_flag', 'sum'),
    total_occurrences=('origin_account', 'size')
).reset_index()

# Add the stripped account ID to the summary DataFrame.
origin_stripped_mapping = df[['origin_account', 'origin_account_stripped']].drop_duplicates()
origin_summary = pd.merge(origin_summary, origin_stripped_mapping, on='origin_account', how='left')

print("\nRésumé des comptes émetteurs uniques:")
print(origin_summary.head())

# Function to extract parts safely
def extract_hex_part(hex_string, start, length):
    if pd.isna(hex_string) or not isinstance(hex_string, str) or len(hex_string) < start + length:
        return None
    return hex_string[start:start+length]

# Step 2: Extract hexadecimal parts and add to destination_summary
# Generate 2-character parts (assuming stripped hex is 16 chars long)
for i in range(8):
    start_index = i * 2
    col_name = f'dest_hex_p{i+1}_2'
    destination_summary[col_name] = destination_summary['destination_account_stripped'].apply(lambda x: extract_hex_part(x, start_index, 2))

# Generate 4-character parts (assuming stripped hex is 16 chars long)
for i in range(4):
    start_index = i * 4
    col_name = f'dest_hex_p{i+1}_4'
    destination_summary[col_name] = destination_summary['destination_account_stripped'].apply(lambda x: extract_hex_part(x, start_index, 4))

print(
"\nRésumé des comptes destinataires avec les parties hexadécimales extraites:")
print(destination_summary.head())

# Extract hexadecimal parts and add to origin_summary
# Generate 2-character parts (assuming stripped hex is 16 chars long)
for i in range(8):
    start_index = i * 2
    col_name = f'origin_hex_p{i+1}_2'
    origin_summary[col_name] = origin_summary['origin_account_stripped'].apply(lambda x: extract_hex_part(x, start_index, 2))

# Generate 4-character parts (assuming stripped hex is 16 chars long)
for i in range(4):
    start_index = i * 4
    col_name = f'origin_hex_p{i+1}_4'
    origin_summary[col_name] = origin_summary['origin_account_stripped'].apply(lambda x: extract_hex_part(x, start_index, 4))

print(
"\nRésumé des comptes émetteurs avec les parties hexadécimales extraites:")
print(origin_summary.head())

# Step 3: Comparative report for repetition for destination accounts
print("\n-- Rapport Comparatif sur les Parties Hexadécimales Répétées (Destinataires) --")

dest_hex_part_columns = [col for col in destination_summary.columns if 'dest_hex_p' in col]

for col in dest_hex_part_columns:
    print(f"\nAnalyse pour la colonne : {col}")
    # Value counts, excluding None values
    value_counts = destination_summary[col].value_counts(dropna=True)
    # Filter for values that appear more than once
    repeating_values = value_counts[value_counts > 1]

    if not repeating_values.empty:
        print(f"  Trouvé {len(repeating_values)} parties uniques répétées. Voici les 5 premières :")
        print(repeating_values.head())
    else:
        print("  Aucune partie répétée trouvée.")

# Comparative report for repetition for origin accounts
print("\n-- Rapport Comparatif sur les Parties Hexadécimales Répétées (Émetteurs) --")

origin_hex_part_columns = [col for col in origin_summary.columns if 'origin_hex_p' in col]

for col in origin_hex_part_columns:
    print(f"\nAnalyse pour la colonne : {col}")
    # Value counts, excluding None values
    value_counts = origin_summary[col].value_counts(dropna=True)
    # Filter for values that appear more than once
    repeating_values = value_counts[value_counts > 1]

    if not repeating_values.empty:
        print(f"  Trouvé {len(repeating_values)} parties uniques répétées. Voici les 5 premières :")
        print(repeating_values.head())
    else:
        print("  Aucune partie répétée trouvée.")

# Step 4: Compare origin and destination hexadecimal segments
print("\n-- Comparaison des Parties Hexadécimales Communes (Émetteurs vs. Destinataires) --")

common_hex_segments_details = []

for dest_col in dest_hex_part_columns:
    # Determine the corresponding origin column name
    # Example: dest_hex_p1_2 -> origin_hex_p1_2
    origin_col = dest_col.replace('dest_hex_p', 'origin_hex_p')

    if origin_col in origin_hex_part_columns:
        dest_unique_segments = set(destination_summary[dest_col].dropna().unique())
        origin_unique_segments = set(origin_summary[origin_col].dropna().unique())

        common_segments = dest_unique_segments.intersection(origin_unique_segments)

        if common_segments:
            common_hex_segments_details.append({
                'segment_type': dest_col.split('_')[1] + '_' + dest_col.split('_')[2], # e.g., p1_2
                'count': len(common_segments),
                'examples': list(common_segments)[:5] # Show up to 5 examples
            })

if common_hex_segments_details:
    total_common_segments = sum([item['count'] for item in common_hex_segments_details])
    print(f"Au total, il y a {total_common_segments} segments hexadécimaux uniques qui apparaissent à la fois dans les comptes émetteurs et récepteurs.")
    print("Détails des segments communs par type de segment :")
    for detail in common_hex_segments_details:
        print(f"  - Type de segment ({detail['segment_type']}): {detail['count']} segments communs, exemples : {detail['examples']}")
else:
    print("Aucun segment hexadécimal commun n'a été trouvé entre les comptes émetteurs et récepteurs.")

In [ ]:
import pandas as pd
# Ensure 'destination_account_stripped' is available
df['destination_account_stripped'] = df['destination_account'].str.replace('acc_d_', '')

# Step 1: Create a summary DataFrame for unique destination accounts
destination_summary = df.groupby('destination_account').agg(
    fraud_count=('fraud_flag', 'sum'),
    total_occurrences=('destination_account', 'size')
).reset_index()

# Add the stripped account ID to the summary DataFrame.
# 'destination_account_stripped' is already available in the main df.
# We can merge it based on 'destination_account'.
account_stripped_mapping = df[['destination_account', 'destination_account_stripped']].drop_duplicates()
destination_summary = pd.merge(destination_summary, account_stripped_mapping, on='destination_account', how='left')

print("Résumé des comptes destinataires uniques:")
print(destination_summary.head())

# Step 2: Extract hexadecimal parts and add to destination_summary
# Function to extract parts safely
def extract_hex_part(hex_string, start, length):
    if pd.isna(hex_string) or not isinstance(hex_string, str) or len(hex_string) < start + length:
        return None
    return hex_string[start:start+length]

# Generate 2-character parts (assuming stripped hex is 16 chars long)
for i in range(8):
    start_index = i * 2
    col_name = f'dest_hex_p{i+1}_2'
    destination_summary[col_name] = destination_summary['destination_account_stripped'].apply(lambda x: extract_hex_part(x, start_index, 2))

# Generate 4-character parts (assuming stripped hex is 16 chars long)
for i in range(4):
    start_index = i * 4
    col_name = f'dest_hex_p{i+1}_4'
    destination_summary[col_name] = destination_summary['destination_account_stripped'].apply(lambda x: extract_hex_part(x, start_index, 4))

print(
"\nRésumé des comptes destinataires avec les parties hexadécimales extraites:")
print(destination_summary.head())

# Step 3: Comparative report for repetition
print("\n-- Rapport Comparatif sur les Parties Hexadécimales Répétées --")

hex_part_columns = [col for col in destination_summary.columns if 'dest_hex_p' in col]

for col in hex_part_columns:
    print(f"\nAnalyse pour la colonne : {col}")
    # Value counts, excluding None values
    value_counts = destination_summary[col].value_counts(dropna=True)
    # Filter for values that appear more than once
    repeating_values = value_counts[value_counts > 1]

    if not repeating_values.empty:
        print(f"  Trouvé {len(repeating_values)} parties uniques répétées. Voici les 5 premières :")
        print(repeating_values.head())
    else:
        print("  Aucune partie répétée trouvée.")

### Analyse de la corrélation entre les segments hexadécimaux et la fraude

Nous allons calculer le taux de fraude pour chaque segment hexadécimal extrait et identifier les segments les plus et les moins associés à la fraude.

In [ ]:
hex_part_columns = [col for col in destination_summary.columns if 'dest_hex_p' in col]

correlation_results = {}

for col in hex_part_columns:
    # Group by the current hex part and sum fraud_count and total_occurrences
    grouped_data = destination_summary.groupby(col).agg(
        total_fraud_for_segment=('fraud_count', 'sum'),
        total_transactions_for_segment=('total_occurrences', 'sum')
    ).reset_index()

    # Calculate the fraud rate for each segment
    # Handle potential division by zero if total_transactions_for_segment is 0
    grouped_data['fraud_rate'] = grouped_data.apply(lambda row: row['total_fraud_for_segment'] / row['total_transactions_for_segment'] if row['total_transactions_for_segment'] > 0 else 0, axis=1)

    # Store the results, sorting by fraud_rate
    correlation_results[col] = grouped_data.sort_values(by='fraud_rate', ascending=False)

# Display top and bottom results for each column
for col, result_df in correlation_results.items():
    print(f"\n--- Top 5 Hex Segments with Highest Fraud Rate for {col} ---")
    # Filter out NaNs (from segments that might be None) and only show segments with actual transactions
    display(result_df[result_df['total_transactions_for_segment'] > 0].head(5))

    # Also display some segments with 0 fraud rate to contrast, if they exist
    print(f"--- Top 5 Hex Segments with Lowest (or Zero) Fraud Rate for {col} ---")
    display(result_df[result_df['total_transactions_for_segment'] > 0].sort_values(by='fraud_rate', ascending=True).head(5))

### Visualisation de la distribution des taux de fraude par segment

In [ ]:
for col, result_df in correlation_results.items():
    plt.figure(figsize=(10, 6))
    # Filter out entries where there are no transactions (fraud_rate might be 0 but not meaningful)
    # and also handle potential NaN values if any segment was None
    meaningful_fraud_rates = result_df[result_df['total_transactions_for_segment'] > 0]['fraud_rate'].dropna()

    if not meaningful_fraud_rates.empty:
        sns.histplot(meaningful_fraud_rates, bins=30, kde=True)
        plt.title(f'Distribution du Taux de Fraude pour le Segment {col}')
        plt.xlabel('Taux de Fraude')
        plt.ylabel('Fréquence')
        plt.grid(True, linestyle='--', alpha=0.7)
        plt.show()
    else:
        print(f"Aucune donnée significative pour la visualisation du taux de fraude pour le segment {col}")

### Test Statistique : Chi-Carré d'Indépendance

Pour confirmer statistiquement l'association entre les segments hexadécimaux et la fraude, nous allons utiliser le **test du Chi-Carré d'Indépendance** (Chi-squared test of independence).

**Objectif du test :**
Ce test permet de déterminer s'il existe une relation statistiquement significative entre deux variables catégorielles. Dans notre cas, les variables sont :
1.  Le segment hexadécimal (par exemple, `dest_hex_p1_2`), qui est une variable catégorielle avec de nombreuses catégories.
2.  Le `fraud_flag`, qui est une variable binaire (0 pour non-fraude, 1 pour fraude).

**Hypothèses :**
*   **Hypothèse Nulle (H0) :** Il n'y a pas d'association entre le segment hexadécimal et le `fraud_flag`. Les taux de fraude sont les mêmes pour tous les segments (la fraude est indépendante du segment).
*   **Hypothèse Alternative (H1) :** Il existe une association entre le segment hexadécimal et le `fraud_flag`. Les taux de fraude diffèrent significativement entre les segments (la fraude est dépendante du segment).

**Interprétation du P-value :**
*   Si le **p-value est inférieur à un seuil de signification (généralement 0.05)**, nous rejetons l'hypothèse n’ulle. Cela signifie qu'il y a une preuve statistique suffisante pour affirmer qu'il existe une association significative entre le segment hexadécimal et la fraude.
*   Si le **p-value est supérieur à 0.05**, nous ne rejetons pas l'hypothèse n’ulle. Cela signifie qu'il n'y a pas de preuve statistique suffisante pour affirmer une association significative entre le segment hexadécimal et la fraude.

### Segments Hexadécimaux avec les Taux de Fraude les Plus Élevés

Après avoir effectué les tests statistiques et confirmé l'association entre les segments hexadécimaux et la fraude, nous allons maintenant mettre en évidence les segments spécifiques qui présentent les taux de fraude les plus élevés.

In [ ]:
print("\n--- Top 5 Hex Segments with Highest Fraud Rate Across All Columns ---")

for col, result_df in correlation_results.items():
    # Filter out NaNs (from segments that might be None) and only show segments with actual transactions
    top_fraud_segments = result_df[result_df['total_transactions_for_segment'] > 0].head(5)
    if not top_fraud_segments.empty:
        print(f"\nTop 5 for {col}:")
        display(top_fraud_segments[[col, 'fraud_rate']].rename(columns={col: 'Segment'}))
    else:
        print(f"\nNo top fraud segments to display for {col}.")

In [ ]:
from scipy.stats import chi2_contingency

print("--- Résultats du Test du Chi-Carré pour l'Association Hexadécimal-Fraude ---")

for col, result_df in correlation_results.items():
    # Filter out entries where there are no transactions or where the segment is None
    # to ensure meaningful statistical testing.
    meaningful_data = result_df[result_df['total_transactions_for_segment'] > 0].dropna(subset=[col])

    if not meaningful_data.empty:
        # Calculate non-fraudulent transactions for each segment
        meaningful_data['total_non_fraud_for_segment'] = meaningful_data['total_transactions_for_segment'] - meaningful_data['total_fraud_for_segment']

        # Create a contingency table (segment counts for fraud vs. non-fraud)
        # The chi2_contingency function expects a 2D array or list of lists.
        contingency_table = meaningful_data[['total_fraud_for_segment', 'total_non_fraud_for_segment']].values

        # Perform the Chi-squared test
        chi2, p_value, _, _ = chi2_contingency(contingency_table)

        print(f"\nSegment: {col}")
        print(f"  Valeur Chi-Carré : {chi2:.2f}")
        print(f"  P-value : {p_value:.4f}")

        if p_value < 0.05:
            print("  Conclusion : Rejet de l'hypothèse nulle. Il existe une association significative entre ce segment hexadécimal et le `fraud_flag`.")
        else:
            print("  Conclusion : Incapacité à rejeter l'hypothèse nulle. Pas de preuve d'une association significative entre ce segment hexadécimal et le `fraud_flag`.")
    else:
        print(f"\nSegment: {col}")
        print("  Pas de données suffisantes pour effectuer le test du Chi-Carré.")

## EDAAAA

In [ ]:
df.columns

In [ ]:
df.head(10)

### adresse et date

In [ ]:
def extract_ipv4_network(ipv4_address, prefix_length=24):
    if ipv4_address is None: # Handle None values from previous extraction
        return None
    try:
        # Split the IP address into octets
        octets = ipv4_address.split('.')
        if len(octets) != 4:
            return None

        # Determine how many octets to include for the network part
        num_octets = prefix_length // 8
        network_part = octets[:num_octets]

        # If prefix_length is not a multiple of 8, handle the last octet partially
        if prefix_length % 8 != 0:
            # This gets complex for bit-level, for simplicity, I'll stick to octet boundaries for now.
            # For /24, it's the first three octets.
            # For /16, it's the first two octets.
            pass # Current logic handles /8, /16, /24, /32 correctly for the join below

        return ".".join(network_part)
    except Exception:
        return None

# Create new columns for origin and destination IPv4 networks (e.g., /24)
df['origin_ipv4_network_24'] = df['origin_ipv4'].apply(lambda x: extract_ipv4_network(x, 24))
df['destination_ipv4_network_24'] = df['destination_ipv4'].apply(lambda x: extract_ipv4_network(x, 24))

print("Analyse des réseaux IPv4 des comptes émetteurs:")
origin_network_analysis = df.groupby(['origin_ipv4_network_24', 'fraud_flag']).size().unstack(fill_value=0)
print(origin_network_analysis.head(10))

print("\nAnalyse des réseaux IPv4 des comptes récepteurs:")
destination_network_analysis = df.groupby(['destination_ipv4_network_24', 'fraud_flag']).size().unstack(fill_value=0)
print(destination_network_analysis.head(10))

# Further analysis: Top networks for fraud and non-fraud
print("\nTop 10 réseaux d'origine avec le plus de fraudes (Flag=1):")
print(origin_network_analysis.sort_values(by=1, ascending=False).head(10))

print("\nTop 10 réseaux de destination avec le plus de fraudes (Flag=1):")
print(destination_network_analysis.sort_values(by=1, ascending=False).head(10))

print("\nTop 10 réseaux d'origine avec le plus de non-fraudes (Flag=0):")
print(origin_network_analysis.sort_values(by=0, ascending=False).head(10))

print("\nTop 10 réseaux de destination avec le plus de non-fraudes (Flag=0):")
print(destination_network_analysis.sort_values(by=0, ascending=False).head(10))

In [ ]:
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt

# 1. CRÉATION DU GRAPHE INFORMATIQUE (DIRIGÉ)
# Les nœuds représentent les machines (IP / MAC) et les arêtes le flux réseau.
G_network = nx.DiGraph()

# 2. CONSTRUCTION DU RÉSEAU À PARTIR DES IPS ET MACS
# Remplacez 'df' par le nom de votre DataFrame.
# On utilise ici vos colonnes 'origin_ipv4', 'origin_mac', etc.
for _, row in df.iterrows():
    # On ignore les lignes où les adresses IP ou MAC sont manquantes (ex: NaT ou NaN)
    if pd.isna(row['origin_ipv4']) or pd.isna(row['destination_ipv4']):
        continue

    src_ip = row['origin_ipv4']
    dst_ip = row['destination_ipv4']
    src_mac = row['origin_mac']
    dst_mac = row['destination_mac']

    # Ajout des machines (nœuds) avec leurs attributs matériels
    G_network.add_node(src_ip, mac=src_mac, type="Émetteur")
    G_network.add_node(dst_ip, mac=dst_mac, type="Récepteur")

    # Ajout du lien réseau (flux de paquets) avec le volume de données (amount) et l'opération
    G_network.add_edge(
        src_ip,
        dst_ip,
        volume_data=float(row['amount']),
        protocol=row['operation']
    )

print("--- ANALYSE DE LA TOPOLOGIE DU RÉSEAU INFORMATIQUE ---")
print(f"Nombre de machines actives détectées (IPs) : {G_network.number_of_nodes()}")
print(f"Nombre de connexions / flux réseau actifs  : {G_network.number_of_edges()}\n")


# 3. ANALYSE DU TRAFIC (MÉTRIQUES RÉSEAU)

# Trafic Sortant (Machines qui émettent le plus de flux)
out_flux = dict(G_network.out_degree())
# Trafic Entrant (Machines serveurs ou cibles qui reçoivent le plus de connexions)
in_flux = dict(G_network.in_degree())

# Détection des Routeurs / Hubs Centraux (Betweenness Centrality)
# Identifie les machines par lesquelles passent la majorité des paquets du réseau
hubs_reseau = nx.betweenness_centrality(G_network)


# 4. CRÉATION DU RAPPORT D'AUDIT DU RÉSEAU
infrastructure_report = pd.DataFrame({
    'MAC_Adresse': pd.Series(nx.get_node_attributes(G_network, 'mac')),
    'Connexions_Sortantes': pd.Series(out_flux),
    'Connexions_Entrantes': pd.Series(in_flux),
    'Indice_Centralite_Hub': pd.Series(hubs_reseau)
}).fillna(0)

print("--- TOP 5 DES MACHINES AGISSANT COMME PASSERELLES / HUBS CENTRAUX ---")
print(infrastructure_report.sort_values(by='Indice_Centralite_Hub', ascending=False).head(5))
print("\n")


# 5. VISUALISATION ANTHROPOMORPHIQUE DU RÉSEAU
plt.figure(figsize=(14, 9))

# Algorithme de disposition (Layout) pour espacer proprement les sous-réseaux
pos = nx.kamada_kawai_layout(G_network)

# Définir la taille des nœuds selon leur importance dans le trafic total
node_sizes = [(in_flux[node] + out_flux[node]) * 300 + 100 for node in G_network.nodes()]

# Dessiner les équipements (Serveurs/Clients)
nx.draw_networkx_nodes(G_network, pos, node_size=node_sizes, node_color='springgreen', alpha=0.85)

# Dessiner les câbles réseau (Flèches directionnelles du trafic)
nx.draw_networkx_edges(G_network, pos, arrowstyle='->', arrowsize=12, edge_color='royalblue', width=1.2)

# Afficher les adresses IP sur la carte
nx.draw_networkx_labels(G_network, pos, font_size=8, font_family='sans-serif', font_weight='bold')

plt.title("Cartographie de la Topologie et des Flux du Réseau Informatique", fontsize=14, fontweight='bold')
plt.axis('off')
plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd
import datetime

# 1. Fonction de nettoyage et extraction de l'hexadécimal brut
def clean_hex(account_id):
    if pd.isna(account_id):
        return ""
    # Supprime les préfixes courants 'acc_o_' ou 'acc_d_' et les espaces
    cleaned = str(account_id).replace("acc_o_", "").replace("acc_d_", "").strip()
    return cleaned

# 2. Fonctions de conversion individuelles (gèrent les chaînes de 16 caractères hex)
def hex_to_decimal(hex_str):
    try:
        return int(hex_str, 16) if hex_str else None
    except ValueError:
        return None

def hex_to_ipv4(hex_str):
    try:
        if len(hex_str) == 16:
            # Récupération des 8 derniers caractères pour l'IPv4
            hex_ip = hex_str[8:]
            return ".".join(str(int(hex_ip[i:i+2], 16)) for i in range(0, 8, 2))
    except Exception:
        pass
    return None

def hex_to_mac(hex_str):
    try:
        if len(hex_str) == 16:
            # Récupération des 12 derniers caractères pour l'adresse MAC
            hex_mac = hex_str[4:]
            return ":".join(hex_mac[i:i+2].upper() for i in range(0, 12, 2))
    except Exception:
        pass
    return None

def hex_to_timestamp(hex_str):
    try:
        if hex_str:
            val_dec = int(hex_str, 16)
            # Conversion basée sur l'hypothèse de nanosecondes Unix
            ts_sec = val_dec / 1_000_000_000
            # On limite aux dates valides gérées par Python
            return datetime.datetime.fromtimestamp(ts_sec, datetime.timezone.utc)
    except Exception:
        pass
    return pd.NaT

# --- APPLICATION SUR LE DATAFRAME ---
# Remplacer 'df' par le nom de votre variable DataFrame existante

# Étape A: Nettoyage et isolation des chaînes hexadécimales brutes
df['origin_hex_clean'] = df['origin_account'].apply(clean_hex)
df['destination_hex_clean'] = df['destination_account'].apply(clean_hex)

# Étape B: Génération des colonnes de conversion pour l'ÉMETTEUR (Origin)
df['calc_origin_decimal']   = df['origin_hex_clean'].apply(hex_to_decimal)
df['calc_origin_ipv4']      = df['origin_hex_clean'].apply(hex_to_ipv4)
df['calc_origin_mac']       = df['origin_hex_clean'].apply(hex_to_mac)
df['calc_origin_timestamp'] = df['origin_hex_clean'].apply(hex_to_timestamp)

# Étape C: Génération des colonnes de conversion pour le RÉCEPTEUR (Destination)
df['calc_destination_decimal']   = df['destination_hex_clean'].apply(hex_to_decimal)
df['calc_destination_ipv4']      = df['destination_hex_clean'].apply(hex_to_ipv4)
df['calc_destination_mac']       = df['destination_hex_clean'].apply(hex_to_mac)
df['calc_destination_timestamp'] = df['destination_hex_clean'].apply(hex_to_timestamp)

# Optionnel: Supprimer les colonnes de travail temporaires si nécessaire
df.drop(columns=['origin_hex_clean', 'destination_hex_clean'], inplace=True)

# Affichage des résultats pour vérification
df[['origin_account', 'calc_origin_ipv4', 'calc_origin_mac', 'calc_origin_timestamp']].head(3)


In [ ]:
def hex_to_decimal(hex_str):
    try:
        return int(hex_str, 16)
    except (ValueError, TypeError):
        return None

def hex_to_utf8(hex_str):
    try:
        # Ensure hex string has an even length for bytes.fromhex
        if len(hex_str) % 2 != 0:
            hex_str = '0' + hex_str
        return bytes.fromhex(hex_str).decode('utf-8', errors='replace') # Replace invalid UTF-8 characters
    except (ValueError, TypeError):
        return None

# Create origin_decimal column
df['origin_decimal'] = df['origin_account_stripped'].apply(hex_to_decimal)

# Create destination_decimal column
df['destination_decimal'] = df['destination_account_stripped'].apply(hex_to_decimal)

# Create origin_utf8 column
df['origin_utf8'] = df['origin_account_stripped'].apply(hex_to_utf8)

# Create destination_utf8 column
df['destination_utf8'] = df['destination_account_stripped'].apply(hex_to_utf8)

print("New columns 'origin_decimal', 'destination_decimal', 'origin_utf8', 'destination_utf8' created.")
print(df[['origin_account', 'origin_account_stripped', 'origin_decimal', 'origin_utf8',
          'destination_account', 'destination_account_stripped', 'destination_decimal', 'destination_utf8']].head())

In [ ]:
import datetime

def extract_ipv4(hex_str):
    if len(hex_str) < 8: # IPv4 needs at least 8 hex characters
        return None
    try:
        # Take the last 8 characters for IPv4
        hex_ip = hex_str[-8:]
        ip_parts = [str(int(hex_ip[i:i+2], 16)) for i in range(0, 8, 2)]
        return ".".join(ip_parts)
    except (ValueError, TypeError):
        return None

def extract_mac(hex_str):
    if len(hex_str) < 12: # MAC needs at least 12 hex characters
        return None
    try:
        # Take the last 12 characters for MAC address (assuming 48-bit MAC)
        hex_mac = hex_str[-12:]
        mac_parts = [hex_mac[i:i+2].upper() for i in range(0, 12, 2)]
        return ":".join(mac_parts)
    except (ValueError, TypeError):
        return None

def extract_timestamp_ns(hex_str):
    try:
        valeur_decimale = int(hex_str, 16)
        # If treated as a precise Unix timestamp in nanoseconds
        timestamp_secondes = valeur_decimale / 1_000_000_000
        # Add a check to prevent very old or future dates that might be invalid
        if 0 < timestamp_secondes < 4102444800: # Approx. Jan 1, 1970 to Jan 1, 2100
            return datetime.datetime.fromtimestamp(timestamp_secondes, datetime.timezone.utc)
        else:
            return None
    except (ValueError, TypeError, OSError): # OSError for invalid timestamp values
        return None

# Apply functions to create new columns for origin accounts
df['origin_ipv4'] = df['origin_account_stripped'].apply(extract_ipv4)
df['origin_mac'] = df['origin_account_stripped'].apply(extract_mac)
df['origin_timestamp'] = df['origin_account_stripped'].apply(extract_timestamp_ns)

# Apply functions to create new columns for destination accounts
df['destination_ipv4'] = df['destination_account_stripped'].apply(extract_ipv4)
df['destination_mac'] = df['destination_account_stripped'].apply(extract_mac)
df['destination_timestamp'] = df['destination_account_stripped'].apply(extract_timestamp_ns)

In [ ]:
# Créer une colonne indiquant la présence d'un horodatage d'origine valide
df['has_origin_timestamp'] = df['origin_timestamp'].notna()

# Créer une colonne indiquant la présence d'un horodatage de destination valide
df['has_destination_timestamp'] = df['destination_timestamp'].notna()

print("Nouvelles colonnes 'has_origin_timestamp' et 'has_destination_timestamp' créées.")
print(df[['origin_timestamp', 'has_origin_timestamp', 'destination_timestamp', 'has_destination_timestamp', 'fraud_flag']].head())

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Distribution du 'fraud_flag' par rapport à la présence d'un horodatage d'origine
plt.figure(figsize=(10, 6))
sns.countplot(x='has_origin_timestamp', hue='fraud_flag', data=df, palette='viridis')
plt.title('Distribution du Fraud Flag par Présence d\'horodatage d\'origine')
plt.xlabel('Présence d\'horodatage d\'origine (False = NaT, True = Valide)')
plt.ylabel('Count')
plt.legend(title='Fraud Flag')
plt.show()

# Distribution du 'fraud_flag' par rapport à la présence d'un horodatage de destination
plt.figure(figsize=(10, 6))
sns.countplot(x='has_destination_timestamp', hue='fraud_flag', data=df, palette='magma')
plt.title('Distribution du Fraud Flag par Présence d\'horodatage de destination')
plt.xlabel('Présence d\'horodatage de destination (False = NaT, True = Valide)')
plt.ylabel('Count')
plt.legend(title='Fraud Flag')
plt.show()

In [ ]:
all_ids_start_with_dtf = df['id'].apply(lambda x: x.startswith('dtf_')).all()

if all_ids_start_with_dtf:
    print("Oui, tous les IDs de transaction commencent par 'dtf_'.")
else:
    print("Non, tous les IDs de transaction ne commencent pas par 'dtf_'.")
    # Optionally, print an example that doesn't conform
    non_conforming_id = df[~df['id'].apply(lambda x: x.startswith('dtf_'))]['id'].iloc[0]
    print(f"Exemple d'ID qui ne commence pas par 'dtf_': {non_conforming_id}")

In [ ]:
all_origin_accounts_start_with_acco = df['origin_account'].apply(lambda x: x.startswith('acc_o_')).all()

if all_origin_accounts_start_with_acco:
    print("Oui, tous les comptes émetteurs commencent par 'acc_o_'.")
else:
    print("Non, tous les comptes émetteurs ne commencent pas par 'acc_o_'.")
    # Optionally, print an example that doesn't conform
    non_conforming_account = df[~df['origin_account'].apply(lambda x: x.startswith('acc_o_'))]['origin_account'].iloc[0]
    print(f"Exemple de compte qui ne commence pas par 'acc_o_': {non_conforming_account}")

In [ ]:
all_destination_accounts_start_with_accd = df['destination_account'].apply(lambda x: x.startswith('acc_d_')).all()

if all_destination_accounts_start_with_accd:
    print("Oui, tous les comptes récepteurs commencent par 'acc_d_'.")
else:
    print("Non, tous les comptes récepteurs ne commencent pas par 'acc_d_'.")
    # Optionally, print an example that doesn't conform
    non_conforming_account = df[~df['destination_account'].apply(lambda x: x.startswith('acc_d_'))]['destination_account'].iloc[0]
    print(f"Exemple de compte qui ne commence pas par 'acc_d_': {non_conforming_account}")

In [ ]:
df['origin_account_stripped'] = df['origin_account'].str.replace('acc_o_', '')
df['destination_account_stripped'] = df['destination_account'].str.replace('acc_d_', '')

unique_stripped_origin = set(df['origin_account_stripped'].unique())
unique_stripped_destination = set(df['destination_account_stripped'].unique())

common_stripped_accounts = unique_stripped_origin.intersection(unique_stripped_destination)

if common_stripped_accounts:
    print(f"Oui, il y a {len(common_stripped_accounts)} comptes qui sont à la fois expéditeurs et récepteurs après avoir enlevé les préfixes.")
    print(f"Voici quelques exemples : {list(common_stripped_accounts)[:5]}")
else:
    print("Non, il n'y a pas de comptes qui sont à la fois expéditeurs et récepteurs après avoir enlevé les préfixes.")

### suite

In [ ]:
for col in df.columns:
  print(f'{col}:{df[col].nunique()}')

In [ ]:
#nombre de compte uniques expediteur ou receveur
a = df['origin_account'].to_list()
b = df['destination_account'].to_list()
c = a + b
print(f'nombre de compte uniques expediteur ou receveur: {len(np.unique(c))}')

In [ ]:
#le nombre d'expediteur etant aussi recepteur
# Trouve les éléments communs aux deux colonnes
valeurs_communes = df.loc[df['origin_account'].isin(df['destination_account']), 'origin_account'].unique()

print(f'le nombre d\'expediteur etant aussi recepteur: {len(valeurs_communes)}')

creer des colonnes historiques qui indique dans l'ordre l'apparution d'un destinataire puis une autre pour l'expediteur.

connaitre le nombre d'expediteur de flag 0 et 1, de meme pour destinataire

In [ ]:
non_fraudulent_origin_accounts = df[df['fraud_flag'] == 0]['origin_account'].unique()
fraudulent_origin_accounts = df[df['fraud_flag'] == 1]['origin_account'].unique()

origin_accounts_in_both_fraud_and_non_fraud = set(non_fraudulent_origin_accounts).intersection(set(fraudulent_origin_accounts))
print(f"Number of unique origin accounts involved in both fraud (flag 1) and non-fraud (flag 0) transactions: {len(origin_accounts_in_both_fraud_and_non_fraud)}")
print(f'nombre de compte unique frauduleux: {len(fraudulent_origin_accounts)}')
print(f'nombre de compte unique non frauduleux: {len(non_fraudulent_origin_accounts)}')

In [ ]:
non_fraudulent_destination_accounts = df[df['fraud_flag'] == 0]['destination_account'].unique()
fraudulent_destination_accounts = df[df['fraud_flag'] == 1]['destination_account'].unique()

destination_accounts_in_both_fraud_and_non_fraud = set(non_fraudulent_destination_accounts).intersection(set(fraudulent_destination_accounts))
print(f"Number of unique destination accounts involved in both fraud (flag 1) and non-fraud (flag 0) transactions: {len(destination_accounts_in_both_fraud_and_non_fraud)}")
print(f'nombre de compte unique frauduleux (destination): {len(fraudulent_destination_accounts)}')
print(f'nombre de compte unique non frauduleux (destination): {len(non_fraudulent_destination_accounts)}')


In [ ]:
# Filter for fraudulent transactions (fraud_flag == 1)
fraudulent_transactions = df[df['fraud_flag'] == 1]

# Group by 'origin_account' and count the number of frauds for each account
fraud_counts_per_origin_account = fraudulent_transactions['origin_account'].value_counts()

# Get the minimum and maximum number of frauds per fraudulent origin account
min_frauds = fraud_counts_per_origin_account.min()
max_frauds = fraud_counts_per_origin_account.max()

print(f"Minimum number of frauds for a unique fraudulent origin account: {min_frauds}")
print(f"Maximum number of frauds for a unique fraudulent origin account: {max_frauds}")

In [ ]:
plt.figure(figsize=(10, 6))
sns.histplot(fraud_counts_per_origin_account, bins=40, kde=True)
plt.title('Distribution du nombre de fraudes par compte émetteur frauduleux')
plt.xlabel('Nombre de fraudes par compte émetteur')
plt.ylabel('Fréquence')
plt.show()

In [ ]:
# Filter for fraudulent transactions (fraud_flag == 1)
# fraudulent_transactions is already defined from the previous step

# Group by 'destination_account' and count the number of frauds for each account
fraud_counts_per_destination_account = fraudulent_transactions['destination_account'].value_counts()

# Get the minimum and maximum number of frauds per fraudulent destination account
min_frauds_dest = fraud_counts_per_destination_account.min()
max_frauds_dest = fraud_counts_per_destination_account.max()

print(f"Minimum number of frauds for a unique fraudulent destination account: {min_frauds_dest}")
print(f"Maximum number of frauds for a unique fraudulent destination account: {max_frauds_dest}")

In [ ]:
plt.figure(figsize=(10, 6))
sns.histplot(fraud_counts_per_destination_account, bins=30, kde=True)
plt.title('Distribution du nombre de fraudes par compte destinataire frauduleux')
plt.xlabel('Nombre de fraudes par compte destinataire')
plt.ylabel('Fréquence')
plt.show()

In [ ]:
df = df.sort_values(by='period').reset_index(drop=True)
print("DataFrame 'df' has been sorted by the 'period' column.")
df[['id', 'period', 'operation']].head()

In [ ]:
print(f"Total number of fraudulent transactions (fraud_flag = 1): {df['fraud_flag'].sum()}")
print(f"Number of unique transaction IDs with fraud_flag = 1: {df[df['fraud_flag'] == 1]['id'].nunique()}")

# Identify unique origin accounts involved in fraud
fraudulent_origin_accounts = df[df['fraud_flag'] == 1]['origin_account'].unique()
print(f"\nNumber of unique origin accounts involved in fraud: {len(fraudulent_origin_accounts)}")

# Identify unique destination accounts involved in fraud
fraudulent_destination_accounts = df[df['fraud_flag'] == 1]['destination_account'].unique()
print(f"Number of unique destination accounts involved in fraud: {len(fraudulent_destination_accounts)}")

# Check if fraudulent origin accounts also appear in non-fraudulent transactions
non_fraudulent_origin_accounts = df[df['fraud_flag'] == 0]['origin_account'].unique()
origin_fraud_and_non_fraud = set(fraudulent_origin_accounts).intersection(set(non_fraudulent_origin_accounts))
print(f"\nNumber of origin accounts involved in both fraudulent and non-fraudulent transactions: {len(origin_fraud_and_non_fraud)}")

# Check if fraudulent destination accounts also appear in non-fraudulent transactions
non_fraudulent_destination_accounts = df[df['fraud_flag'] == 0]['destination_account'].unique()
destination_fraud_and_non_fraud = set(fraudulent_destination_accounts).intersection(set(non_fraudulent_destination_accounts))
print(f"Number of destination accounts involved in both fraudulent and non-fraudulent transactions: {len(destination_fraud_and_non_fraud)}")

# Create a set of all accounts that have ever committed fraud (origin or destination)
all_fraudulent_accounts = set(fraudulent_origin_accounts).union(set(fraudulent_destination_accounts))

# Create new columns for 'origin_account_previously_fraud' and 'destination_account_previously_fraud'
df['origin_account_previously_fraud'] = df['origin_account'].isin(all_fraudulent_accounts).astype(int)
df['destination_account_previously_fraud'] = df['destination_account'].isin(all_fraudulent_accounts).astype(int)

print("\nNew columns 'origin_account_previously_fraud' and 'destination_account_previously_fraud' created.")
print("First 5 rows with new columns:")
print(df[['origin_account', 'destination_account', 'fraud_flag', 'origin_account_previously_fraud', 'destination_account_previously_fraud']].head())

In [ ]:
df['operation'].value_counts()

probleme de la fraude est la destination

emetteur honnete qui envoie vers distination fraudeuleuse est une fraude

les destinataire frauduleux ne font jamais les autres operations

-fraude 03: transfert sans arrivé sur le compte ou departs

les seules fraudes sont celles de compte recepteurs et jamais de compte emetteur

01: retrait, négatif autorisé

02: aucune variation de montant expediteur ou recepteurs

03: transfer ou prets, transfert négatif restant autorisé, les comptes destinataires ne font jamais une autre transactions fraude ou non

04: depot

05: retrait par code

op_04, op_02, op_05: toujours ensembles sur un compte destinataire non-frauduleux


In [ ]:
first_fraud_occurrence = df[df['fraud_flag'] == 1]['period'].min()
print(f"La première apparition d'une transaction frauduleuse est à la période: {first_fraud_occurrence}")

In [ ]:
first_fraud_row_index = df[df['fraud_flag'] == 1].index[0]
print(f"Le numéro de la ligne de la première apparition d'une transaction frauduleuse est: {first_fraud_row_index}")

### Analyse des Comptes Destinataires : Transition vers la Fraude

In [ ]:
# 1. Identifier les comptes destinataires qui sont non-frauduleux puis frauduleux

# Group by destination account and find the minimum and maximum fraud_flag
# and the period of their first non-fraudulent and first fraudulent transaction
account_fraud_summary = df.groupby('destination_account').agg(
    first_fraud_period=('period', lambda x: x[df.loc[x.index, 'fraud_flag'] == 1].min()),
    first_non_fraud_period=('period', lambda x: x[df.loc[x.index, 'fraud_flag'] == 0].min()),
    has_fraud=('fraud_flag', lambda x: (x == 1).any()),
    has_non_fraud=('fraud_flag', lambda x: (x == 0).any())
).reset_index()

# Accounts that were non-fraudulent first and then became fraudulent
also_become_fraudulent = account_fraud_summary[
    account_fraud_summary['has_non_fraud'] &
    account_fraud_summary['has_fraud'] &
    (account_fraud_summary['first_non_fraud_period'] < account_fraud_summary['first_fraud_period'])
]

print(f"Nombre de comptes destinataires qui étaient non-frauduleux (flag=0) au début et sont devenus frauduleux (flag=1) plus tard : {len(also_become_fraudulent)}")
if not also_become_fraudulent.empty:
    print("Exemples de tels comptes :\n", also_become_fraudulent.head())
else:
    print("Aucun compte destinataire n'a été trouvé qui est passé de non-frauduleux à frauduleux.")

### Analyse des Comptes Destinataires : Persistance de la Fraude

In [ ]:
# 2. Vérifier si un compte, une fois devenu frauduleux, le reste toujours

# Filter for accounts that have at least one fraudulent transaction as a destination
fraudulent_dest_accounts_overall = df[df['fraud_flag'] == 1]['destination_account'].unique()

always_fraudulent_once_fraud = []

for account in fraudulent_dest_accounts_overall:
    account_transactions = df[df['destination_account'] == account].sort_values(by='period')

    # Find the period of the first fraudulent transaction for this account
    first_fraud_period_for_account = account_transactions[account_transactions['fraud_flag'] == 1]['period'].min()

    # Check transactions AFTER the first fraudulent one
    subsequent_transactions = account_transactions[account_transactions['period'] > first_fraud_period_for_account]

    # If there are subsequent transactions and any of them are non-fraudulent, then it's not 'always fraudulent'
    if not subsequent_transactions.empty and (subsequent_transactions['fraud_flag'] == 0).any():
        always_fraudulent_once_fraud.append(False)
    else:
        # If no subsequent transactions, or all subsequent are fraudulent, then it's 'always fraudulent'
        always_fraudulent_once_fraud.append(True)

# Calculate the proportion or count of accounts that always remain fraudulent
if fraudulent_dest_accounts_overall.size > 0:
    num_always_fraud = sum(always_fraudulent_once_fraud)
    print(f"Sur {len(fraudulent_dest_accounts_overall)} comptes destinataires ayant commis la fraude au moins une fois :")
    print(f"- {num_always_fraud} comptes sont restés frauduleux après leur première fraude.")
    print(f"- {len(fraudulent_dest_accounts_overall) - num_always_fraud} comptes ont eu des transactions non-frauduleuses après leur première fraude.")
else:
    print("Aucun compte destinataire n'a été impliqué dans la fraude.")

Sur 5000 comptes destinataires ayant commis la fraude au moins une fois :
- 0 comptes sont restés frauduleux après leur première fraude.
- 5000 comptes ont eu des transactions non-frauduleuses après leur première fraude.

Nombre de comptes destinataires qui étaient non-frauduleux (flag=0) au début et sont devenus frauduleux (flag=1) plus tard : 2549

il y a des comptes recpeteurs au debut qui ne sont pas des fraudes et qui deviendront fraude.

est ce que dé devenu fraude il va rester ainsi?

les comptes fraudes recepteurs apparaise la premier fois avec 85 a 100 et subitement de grande somme

gros transfert d'un expediteur qui n a rien et parfois des chiffres negatif vers un compte vide. CELA AU DEBUT

dans 03 peut on faire du retrait negatif dans non fraude?

### K-Means Clustering for `op_03` Data

To explore potential separation in the `op_03` transactions, we will apply K-Means clustering. We'll use three key numerical features: `amount`, `origin_balance_after`, and `destination_balance_after`.

First, we'll extract the `op_03` transactions and prepare the features. We'll then scale these features to ensure that all features contribute equally to the distance calculation during clustering. Finally, we'll perform K-Means clustering and visualize the results in 3D.

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
import matplotlib.pyplot as plt
import seaborn as sns
from mpl_toolkits.mplot3d import Axes3D

# Filter the DataFrame for operation == 'op_03' and make a copy to ensure all feature engineering is applied
df_op03_clustering = df[df['operation'] == 'op_03'].copy()

# Select numerical features for clustering
features_for_clustering = ['amount', 'origin_balance_after', 'destination_balance_after']

# Drop rows with NaN values in selected features to avoid issues with KMeans
df_op03_clustering.dropna(subset=features_for_clustering, inplace=True)

# Ensure the DataFrame is not empty after dropping NaNs
if df_op03_clustering.empty:
    print("No data available for clustering after filtering 'op_03' and dropping NaNs.")
else:
    X_clustering = df_op03_clustering[features_for_clustering]

    # Scale the features
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X_clustering)

    # Apply K-Means clustering (starting with 2 clusters, as fraud_flag is binary)
    kmeans = KMeans(n_clusters=2, random_state=42, n_init=10) # n_init is set to suppress warning
    df_op03_clustering['cluster_label'] = kmeans.fit_predict(X_scaled)

    print("K-Means clustering completed. Visualizing results...")

    # 3D Visualization of Clusters
    fig = plt.figure(figsize=(12, 10))
    ax = fig.add_subplot(111, projection='3d')

    scatter = ax.scatter(
        df_op03_clustering['amount'],
        df_op03_clustering['origin_balance_after'],
        df_op03_clustering['destination_balance_after'],
        c=df_op03_clustering['cluster_label'],
        cmap='viridis', # Different color for each cluster
        marker='o',
        s=20,
        alpha=0.6
    )

    ax.set_xlabel('Amount')
    ax.set_ylabel('Origin Balance After')
    ax.set_zlabel('Destination Balance After')
    ax.set_title('3D K-Means Clusters for op_03 Transactions')

    # Add a color bar
    legend1 = ax.legend(*scatter.legend_elements(), title="Clusters")
    ax.add_artist(legend1)

    plt.show()

    # 3D Visualization of Fraud Flag for comparison
    fig = plt.figure(figsize=(12, 10))
    ax = fig.add_subplot(111, projection='3d')

    scatter_fraud = ax.scatter(
        df_op03_clustering['amount'],
        df_op03_clustering['origin_balance_after'],
        df_op03_clustering['destination_balance_after'],
        c=df_op03_clustering['fraud_flag'],
        cmap='coolwarm', # Different color for fraud vs non-fraud
        marker='o',
        s=20,
        alpha=0.6
    )

    ax.set_xlabel('Amount')
    ax.set_ylabel('Origin Balance After')
    ax.set_zlabel('Destination Balance After')
    ax.set_title('3D Fraud Flag Distribution for op_03 Transactions')

    # Add a color bar
    legend2 = ax.legend(*scatter_fraud.legend_elements(), title="Fraud Flag")
    ax.add_artist(legend2)

    plt.show()

In [ ]:
from sklearn.metrics import silhouette_score

if not df_op03_clustering.empty:
    # Calculate the silhouette score
    silhouette_avg = silhouette_score(X_scaled, df_op03_clustering['cluster_label'])
    print(f"The average silhouette score for the K-Means clusters is: {silhouette_avg:.4f}")
else:
    print("Cannot calculate silhouette score: No data available for clustering.")

The plots above show the 3D visualization of the `op_03` transactions using `amount`, `origin_balance_after`, and `destination_balance_after`. The first plot colors data points by their assigned K-Means cluster, and the second plot colors them by their actual `fraud_flag`.

By comparing these two plots, we can visually inspect if the clusters identified by K-Means align with the fraudulent and non-fraudulent transactions. If there's a clear separation, it suggests that these features are strong indicators for distinguishing between fraud and non-fraud.

### Visualisation de la distribution des clusters K-Means par rapport au 'fraud_flag'

Cette visualisation aide à évaluer dans quelle mesure les clusters identifiés par l'algorithme K-Means correspondent aux transactions frauduleuses et non-frauduleuses.

In [ ]:
plt.figure(figsize=(10, 6))
sns.countplot(x='cluster_label', hue='fraud_flag', data=df_op03_clustering, palette='coolwarm')
plt.title('Distribution des clusters K-Means par Fraud Flag pour op_03')
plt.xlabel('Cluster K-Means')
plt.ylabel('Nombre de Transactions')
plt.legend(title='Fraud Flag')
plt.show()

### Application d'un second algorithme de clustering (DBSCAN) sur le Cluster 1 de K-Means

Pour approfondir l'analyse, nous allons maintenant appliquer l'algorithme de clustering DBSCAN aux transactions qui ont été classées dans le `cluster_label` 1 par le précédent K-Means. DBSCAN est capable de trouver des clusters de formes arbitraires et d'identifier le bruit, ce qui peut révéler des structures que K-Means n'aurait pas capturées.

In [ ]:
from sklearn.cluster import DBSCAN
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
import matplotlib.pyplot as plt
import seaborn as sns
from mpl_toolkits.mplot3d import Axes3D

# --- Code from cell 64cd3e42 (K-Means Clustering) to ensure df_op03_clustering is defined ---
# Filter the DataFrame for operation == 'op_03' and make a copy
df_op03_clustering = df[df['operation'] == 'op_03'].copy()

# Select numerical features for clustering
features_for_clustering = ['amount', 'origin_balance_after', 'destination_balance_after']

# Drop rows with NaN values in selected features to avoid issues with KMeans
df_op03_clustering.dropna(subset=features_for_clustering, inplace=True)

# Ensure the DataFrame is not empty after dropping NaNs
if df_op03_clustering.empty:
    print("No data available for K-Means clustering after filtering 'op_03' and dropping NaNs.")
    # Exit if no data, to prevent subsequent errors
    exit()
else:
    X_clustering = df_op03_clustering[features_for_clustering]

    # Scale the features
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X_clustering)

    # Apply K-Means clustering (starting with 2 clusters, as fraud_flag is binary)
    kmeans = KMeans(n_clusters=2, random_state=42, n_init=10)
    df_op03_clustering['cluster_label'] = kmeans.fit_predict(X_scaled)

    print("K-Means clustering completed as a prerequisite for DBSCAN.")

# --- Original DBSCAN code starts here ---
# Filter data for cluster_label 1 from the previous K-Means result
df_cluster1 = df_op03_clustering[df_op03_clustering['cluster_label'] == 1].copy()

# Check if there's enough data in cluster 1 to perform further clustering
if df_cluster1.empty:
    print("No data in cluster_label 1 to perform further clustering with DBSCAN.")
else:
    # Select the same features for clustering
    features_for_clustering_dbscan = ['amount', 'origin_balance_after', 'destination_balance_after']
    X_cluster1 = df_cluster1[features_for_clustering_dbscan]

    # Scale the features (important for distance-based algorithms like DBSCAN)
    scaler_dbscan = StandardScaler()
    X_scaled_cluster1 = scaler_dbscan.fit_transform(X_cluster1)

    # Apply DBSCAN clustering
    # Parameters eps and min_samples need to be tuned based on data density
    # For demonstration, using some default values, but these would ideally be optimized.
    dbscan = DBSCAN(eps=0.5, min_samples=5)
    df_cluster1['dbscan_cluster_label'] = dbscan.fit_predict(X_scaled_cluster1)

    print(f"DBSCAN clustering applied to K-Means Cluster 1. Found {df_cluster1['dbscan_cluster_label'].nunique()} clusters (including noise -1).")

    # 3D Visualization of DBSCAN Clusters within K-Means Cluster 1
    fig = plt.figure(figsize=(12, 10))
    ax = fig.add_subplot(111, projection='3d')

    # Use a different colormap for DBSCAN clusters
    scatter = ax.scatter(
        df_cluster1['amount'],
        df_cluster1['origin_balance_after'],
        df_cluster1['destination_balance_after'],
        c=df_cluster1['dbscan_cluster_label'],
        cmap='plasma', # Another colormap for differentiation
        marker='o',
        s=20,
        alpha=0.6
    )

    ax.set_xlabel('Amount')
    ax.set_ylabel('Origin Balance After')
    ax.set_zlabel('Destination Balance After')
    ax.set_title('3D DBSCAN Clusters within K-Means Cluster 1 (op_03 Transactions)')

    # Add a color bar
    legend1 = ax.legend(*scatter.legend_elements(), title="DBSCAN Clusters")
    ax.add_artist(legend1)

    plt.show()

    # Visualize the distribution of DBSCAN clusters by fraud_flag within K-Means Cluster 1
    plt.figure(figsize=(10, 6))
    sns.countplot(x='dbscan_cluster_label', hue='fraud_flag', data=df_cluster1, palette='viridis')
    plt.title('Distribution des clusters DBSCAN par Fraud Flag (dans K-Means Cluster 1)')
    plt.xlabel('Cluster DBSCAN')
    plt.ylabel('Nombre de Transactions')
    plt.legend(title='Fraud Flag')
    plt.show()

### Corrélation entre les clusters DBSCAN et le `fraud_flag`

Nous allons calculer la corrélation de Pearson entre le `dbscan_cluster_label` et le `fraud_flag` pour évaluer dans quelle mesure les clusters identifiés par DBSCAN sont liés à la fraude. Une corrélation plus élevée (positive ou négative) indiquerait une relation plus forte.

In [ ]:
if 'df_cluster1' in locals() and not df_cluster1.empty:
    correlation_dbscan_fraud = df_cluster1[['dbscan_cluster_label', 'fraud_flag']].corr()
    print("Matrice de corrélation entre les clusters DBSCAN et le fraud_flag :")
    display(correlation_dbscan_fraud)
else:
    print("Le DataFrame df_cluster1 n'est pas défini ou est vide, impossible de calculer la corrélation.")

In [ ]:
df03 = df[df['operation'] == 'op_03'].copy()
df03['delta_origin_account'] = df03['origin_balance_after'] - df03['origin_balance_before']
df03['delta_destination_account'] = df03['destination_balance_after'] - df03['destination_balance_before']
print("DataFrame df03 created with 'op_03' transactions and new delta columns.")
df03[['operation', 'origin_balance_before', 'origin_balance_after', 'delta_origin_account', 'destination_balance_before', 'destination_balance_after', 'delta_destination_account']].head()

la diffrence des deltas est toujours simultanément 0 ou toulours simultaément different

In [ ]:
# Distribution of fraud_flag where delta_origin_account is 0
print("\nDistribution de fraud_flag lorsque delta_origin_account est different de 0:")
df03_delta_origin_zero = df03[df03['delta_origin_account'] < 0]
print(df03_delta_origin_zero['fraud_flag'].value_counts(normalize= True))
print(df03_delta_origin_zero['fraud_flag'].value_counts())

# Distribution of fraud_flag where delta_destination_account is 0
print("\nDistribution de fraud_flag lorsque delta_destination_account est 0:")
df03_delta_dest_zero = df03[df03['delta_destination_account'] == 0]
print(df03_delta_dest_zero['fraud_flag'].value_counts(normalize= True))
print(df03_delta_dest_zero['fraud_flag'].value_counts())


In [ ]:
# Get unique destination accounts where delta_destination_account is 0
dest_accounts_delta_zero = set(df03[df03['delta_destination_account'] == 0]['destination_account'].unique())

# Get unique destination accounts where delta_destination_account is not 0
dest_accounts_delta_not_zero = set(df03[df03['delta_destination_account'] != 0]['destination_account'].unique())

# Find the intersection of these two sets
intersecting_accounts = dest_accounts_delta_zero.intersection(dest_accounts_delta_not_zero)

if intersecting_accounts:
    print(f"Oui, il y a {len(intersecting_accounts)} comptes récepteurs uniques qui ont un delta = 0 et un delta != 0.")
    print("Voici quelques exemples de ces comptes :", list(intersecting_accounts)[:5])
else:
    print("Non, il n'y a pas de comptes récepteurs uniques qui ont un delta = 0 et un delta != 0.")

In [ ]:
# 1. Accounts with variable delta (already identified as 'intersecting_accounts')
variable_delta_dest_accounts = intersecting_accounts

# 2. Accounts with fixed (zero) delta: those that ONLY appear with delta_destination_account == 0 in op_03
all_dest_accounts_op03 = set(df03['destination_account'].unique())
fixed_zero_delta_dest_accounts = set(dest_accounts_delta_zero) - variable_delta_dest_accounts

print(f"\nNumber of destination accounts with variable delta: {len(variable_delta_dest_accounts)}")
print(f"Number of destination accounts with fixed (zero) delta: {len(fixed_zero_delta_dest_accounts)}")

In [ ]:
# 3. Analyze fraud for variable delta accounts
print("\n--- Analyse des comptes avec delta variable ---")
df_variable_delta = df03[df03['destination_account'].isin(list(variable_delta_dest_accounts))]
if not df_variable_delta.empty:
    print("Distribution de fraud_flag pour les comptes à delta variable:")
    print(df_variable_delta['fraud_flag'].value_counts(normalize=True))
else:
    print("Aucune transaction trouvée pour les comptes à delta variable.")

In [ ]:
# 4. Analyze fraud for fixed (zero) delta accounts
print("\n--- Analyse des comptes avec delta fixe (zéro) ---")
df_fixed_zero_delta = df03[df03['destination_account'].isin(list(fixed_zero_delta_dest_accounts))]
if not df_fixed_zero_delta.empty:
    print("Distribution de fraud_flag pour les comptes à delta fixe (zéro):")
    print(df_fixed_zero_delta['fraud_flag'].value_counts())
else:
    print("Aucune transaction trouvée pour les comptes à delta fixe (zéro).")

In [ ]:
# Identify accounts that ONLY have non-zero deltas
# These are accounts that appear in 'dest_accounts_delta_not_zero' but NOT in 'dest_accounts_delta_zero'
fixed_non_zero_delta_dest_accounts = set(dest_accounts_delta_not_zero) - set(dest_accounts_delta_zero)

print(f"\nNumber of destination accounts with fixed (non-zero) delta: {len(fixed_non_zero_delta_dest_accounts)}")

# Analyze fraud for fixed (non-zero) delta accounts
print("\n--- Analyse des comptes avec delta fixe (non-zéro) ---")
df_fixed_non_zero_delta = df03[df03['destination_account'].isin(list(fixed_non_zero_delta_dest_accounts))]

if not df_fixed_non_zero_delta.empty:
    print("Distribution de fraud_flag pour les comptes à delta fixe (non-zéro):")
    print(df_fixed_non_zero_delta['fraud_flag'].value_counts(normalize=True))
else:
    print("Aucune transaction trouvée pour les comptes à delta fixe (non-zéro).")

In [ ]:
zero_origin_delta = df03[df03['delta_origin_account'] == 0]

if not zero_origin_delta.empty:
    unique_dest_deltas = zero_origin_delta['delta_destination_account'].unique()
    if len(unique_dest_deltas) == 1 and unique_dest_deltas[0] == 0:
        print("Oui, à chaque fois que le delta de l'émetteur est 0, celui du récepteur est aussi 0 pour les transactions 'op_03'.")
    else:
        print("Non, il y a des cas où le delta de l'émetteur est 0, mais celui du récepteur n'est pas 0 (ou vice versa) pour les transactions 'op_03'.")
        print("Valeurs uniques de delta_destination_account lorsque delta_origin_account est 0:")
        print(unique_dest_deltas)
else:
    print("Aucune transaction trouvée où le delta de l'émetteur est 0 dans 'op_03'.")

In [ ]:
df.head(20)

In [ ]:
fraud_with_non_fraud_dest = df[(df['fraud_flag'] == 1) & (df['destination_account_previously_fraud'] == 0)]

num_fraud_with_non_fraud_dest = len(fraud_with_non_fraud_dest)

print(f"Number of fraudulent transactions where destination_account_previously_fraud is 0: {num_fraud_with_non_fraud_dest}")

if num_fraud_with_non_fraud_dest > 0:
    print("Yes, there are fraudulent transactions where the destination account was not previously flagged as fraudulent.")
else:
    print("No, there are no fraudulent transactions where the destination account was not previously flagged as fraudulent.")

# Display some examples if any
if num_fraud_with_non_fraud_dest > 0:
    print("\nExamples of such fraudulent transactions:")
    print(fraud_with_non_fraud_dest.head())

In [ ]:
fraudulent_destination_per_operation = df[df['fraud_flag'] == 0].groupby('operation')['destination_account'].nunique()
print("Répartition des comptes destinataires non-frauduleux uniques par opération :")
print(fraudulent_destination_per_operation)

In [ ]:
# 1. Identify truly non-fraudulent accounts (from previous steps)
#    'all_unique_accounts', 'all_fraudulent_accounts', and 'truly_non_fraudulent_accounts' should be in kernel state.

# 2. Filter the DataFrame for transactions involving these truly non-fraudulent destination accounts
df_truly_non_fraud_dest_txns = df[df['destination_account'].isin(truly_non_fraudulent_accounts)]

# 3. Group by destination account and collect all unique operation types as a set for each account
operations_set_per_truly_non_fraud_dest_account = df_truly_non_fraud_dest_txns.groupby('destination_account')['operation'].apply(lambda x: set(x.unique()))

# 4. Identify accounts that are involved in 'op_03'
accounts_involved_in_op03 = operations_set_per_truly_non_fraud_dest_account[operations_set_per_truly_non_fraud_dest_account.apply(lambda x: 'op_03' in x)]

# 5. From these accounts, count how many are involved in 'op_03' AND at least one other operation
#    This means the size of their unique operations set is greater than 1.
accounts_in_op03_and_multiple_others = accounts_involved_in_op03[accounts_involved_in_op03.apply(len) > 1]

print(f"Number of unique non-fraudulent destination accounts involved in 'op_03' AND at least one other operation: {len(accounts_in_op03_and_multiple_others)}")
print("These accounts are:")
print(accounts_in_op03_and_multiple_others)

In [ ]:
fraudulent_destination_per_operation = df[df['fraud_flag'] == 1].groupby('operation')['destination_account'].nunique()
print("Répartition des comptes destinataires frauduleux uniques par opération :")
print(fraudulent_destination_per_operation)

In [ ]:
# 1. Identify fraudulent origin accounts and their total transaction counts
# 'fraudulent_origin_accounts' is already defined from previous cells

# Filter df to include only transactions from fraudulent origin accounts
df_fraud_origin_txns = df[df['origin_account'].isin(fraudulent_origin_accounts)]
# Count total transactions for each fraudulent origin account
fraud_origin_tx_counts = df_fraud_origin_txns['origin_account'].value_counts()

# 2. Identify truly non-fraudulent origin accounts and their total transaction counts
# 'truly_non_fraudulent_accounts' is already defined from previous cells

# Filter df to include only transactions from truly non-fraudulent origin accounts
df_non_fraud_origin_txns = df[df['origin_account'].isin(truly_non_fraudulent_accounts)]
# Count total transactions for each non-fraudulent origin account
non_fraud_origin_tx_counts = df_non_fraud_origin_txns['origin_account'].value_counts()

# 3. Extract top accounts and create dataframes

# Top 3 fraudulent origin accounts by total transaction count
top3_fraud_origin_accounts = fraud_origin_tx_counts.head(3).index.tolist()

# Create fraud1e, fraud2e, fraud3e
if len(top3_fraud_origin_accounts) > 0:
    fraud1e = df[df['origin_account'] == top3_fraud_origin_accounts[0]]
    print(f"Created fraud1e for account {top3_fraud_origin_accounts[0]} with {len(fraud1e)} entries.")
else:
    print("No fraudulent origin accounts found to create fraud1e.")

if len(top3_fraud_origin_accounts) > 1:
    fraud2e = df[df['origin_account'] == top3_fraud_origin_accounts[1]]
    print(f"Created fraud2e for account {top3_fraud_origin_accounts[1]} with {len(fraud2e)} entries.")

if len(top3_fraud_origin_accounts) > 2:
    fraud3e = df[df['origin_account'] == top3_fraud_origin_accounts[2]]
    print(f"Created fraud3e for account {top3_fraud_origin_accounts[2]} with {len(fraud3e)} entries.")

# Top 3 truly non-fraudulent origin accounts by total transaction count
top3_non_fraud_origin_accounts = non_fraud_origin_tx_counts.head(3).index.tolist()

# Create nonfraud1e, nonfraud2e, nonfraud3e
if len(top3_non_fraud_origin_accounts) > 0:
    nonfraud1e = df[df['origin_account'] == top3_non_fraud_origin_accounts[0]]
    print(f"Created nonfraud1e for account {top3_non_fraud_origin_accounts[0]} with {len(nonfraud1e)} entries.")
else:
    print("No truly non-fraudulent origin accounts found to create nonfraud1e.")

if len(top3_non_fraud_origin_accounts) > 1:
    nonfraud2e = df[df['origin_account'] == top3_non_fraud_origin_accounts[1]]
    print(f"Created nonfraud2e for account {top3_non_fraud_origin_accounts[1]} with {len(nonfraud2e)} entries.")

if len(top3_non_fraud_origin_accounts) > 2:
    nonfraud3e = df[df['origin_account'] == top3_non_fraud_origin_accounts[2]]
    print(f"Created nonfraud3e for account {top3_non_fraud_origin_accounts[2]} with {len(nonfraud3e)} entries.")

In [ ]:
nonfraud3e.head(20)

In [ ]:
fraud1e.head(20)

si le amount superieur a origin_balance_before alors fraude

operation de type 04 depot

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Assuming fraud1e, fraud2e, fraud3e are already defined from the previous step
fraudulent_origin_dfs = {'fraud1e': fraud1e, 'fraud2e': fraud2e, 'fraud3e': fraud3e}

for name, df_account in fraudulent_origin_dfs.items():
    if not df_account.empty:
        # Get the unique origin account ID for the title
        account_id = df_account['origin_account'].iloc[0]

        # Mark fraudulent periods with vertical lines
        fraud_periods = df_account[df_account['fraud_flag'] == 1]['period'].unique()

        # Plot for origin_balance_after
        plt.figure(figsize=(14, 7))
        sns.lineplot(x='period', y='origin_balance_after', data=df_account, label='Solde du compte émetteur (après transaction)', marker='o', markersize=4, color='blue')
        for p in fraud_periods:
            plt.axvline(x=p, color='red', linestyle='--', linewidth=1, label='Événement de fraude' if p == fraud_periods[0] else '')
        plt.title(f'Solde après transaction pour le compte {account_id} au fil du temps avec événements de fraude')
        plt.xlabel('Période')
        plt.ylabel('Solde du compte émetteur (après transaction)')
        plt.grid(True)
        plt.legend()
        plt.tight_layout()
        plt.show()

        # Plot for origin_balance_before
        plt.figure(figsize=(14, 7))
        sns.lineplot(x='period', y='origin_balance_before', data=df_account, label='Solde du compte émetteur (avant transaction)', marker='x', markersize=4, color='orange')
        for p in fraud_periods:
            plt.axvline(x=p, color='red', linestyle='--', linewidth=1, label='Événement de fraude' if p == fraud_periods[0] else '')
        plt.title(f'Solde avant transaction pour le compte {account_id} au fil du temps avec événements de fraude')
        plt.xlabel('Période')
        plt.ylabel('Solde du compte émetteur (avant transaction)')
        plt.grid(True)
        plt.legend()
        plt.tight_layout()
        plt.show()

        # Plot for amount
        plt.figure(figsize=(14, 7))
        sns.lineplot(x='period', y='amount', data=df_account, label='Montant de la transaction', marker='s', markersize=4, color='purple')
        for p in fraud_periods:
            plt.axvline(x=p, color='red', linestyle='--', linewidth=1, label='Événement de fraude' if p == fraud_periods[0] else '')
        plt.title(f'Montant de la transaction pour le compte {account_id} au fil du temps avec événements de fraude')
        plt.xlabel('Période')
        plt.ylabel('Montant de la transaction')
        plt.grid(True)
        plt.legend()
        plt.tight_layout()
        plt.show()

    else:
        print(f"DataFrame {name} est vide, impossible de visualiser.")

### Visualisation Détaillée pour les Comptes Non-Frauduleux

Pour une meilleure clarté, nous allons maintenant séparer les visualisations du solde avant, après, et du montant des transactions pour les comptes non-frauduleux (`nonfraud1e`, `nonfraud2e`, `nonfraud3e`). Comme ces comptes n'ont jamais été impliqués dans la fraude, il n'y aura pas de marqueurs de fraude.

In [ ]:
non_fraudulent_origin_dfs = {'nonfraud1e': nonfraud1e, 'nonfraud2e': nonfraud2e, 'nonfraud3e': nonfraud3e}

for name, df_account in non_fraudulent_origin_dfs.items():
    if not df_account.empty:
        account_id = df_account['origin_account'].iloc[0]

        # Plot for origin_balance_after
        plt.figure(figsize=(14, 7))
        sns.lineplot(x='period', y='origin_balance_after', data=df_account, label='Solde du compte émetteur (après transaction)', marker='o', markersize=4, color='blue')
        plt.title(f'Solde après transaction pour le compte non-frauduleux {account_id} au fil du temps')
        plt.xlabel('Période')
        plt.ylabel('Solde du compte émetteur (après transaction)')
        plt.grid(True)
        plt.legend()
        plt.tight_layout()
        plt.show()

        # Plot for origin_balance_before
        plt.figure(figsize=(14, 7))
        sns.lineplot(x='period', y='origin_balance_before', data=df_account, label='Solde du compte émetteur (avant transaction)', marker='x', markersize=4, color='orange')
        plt.title(f'Solde avant transaction pour le compte non-frauduleux {account_id} au fil du temps')
        plt.xlabel('Période')
        plt.ylabel('Solde du compte émetteur (avant transaction)')
        plt.grid(True)
        plt.legend()
        plt.tight_layout()
        plt.show()

        # Plot for amount
        plt.figure(figsize=(14, 7))
        sns.lineplot(x='period', y='amount', data=df_account, label='Montant de la transaction', marker='s', markersize=4, color='purple')
        plt.title(f'Montant de la transaction pour le compte non-frauduleux {account_id} au fil du temps')
        plt.xlabel('Période')
        plt.ylabel('Montant de la transaction')
        plt.grid(True)
        plt.legend()
        plt.tight_layout()
        plt.show()

    else:
        print(f"DataFrame {name} est vide, impossible de visualiser.")

## test rapide

In [ ]:
df.describe(include='all')

## Data Visualizations by Fraud Flag

Let's visualize the distributions of key variables broken down by `fraud_flag` to identify patterns.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore') # Suppress warnings for better readability

# Box plot for 'period'
plt.figure(figsize=(8, 5))
sns.boxplot(x='fraud_flag', y='period', data=df)
plt.title('Period Distribution by Fraud Flag')
plt.xlabel('Fraud Flag (0 = No Fraud, 1 = Fraud)')
plt.ylabel('Period')
plt.show()

In [ ]:
# Bar plot for 'operation'
plt.figure(figsize=(10, 6))
sns.countplot(x='operation', hue='fraud_flag', data=df, palette='viridis')
plt.title('Operation Type Distribution by Fraud Flag')
plt.xlabel('Operation Type')
plt.ylabel('Count')
plt.legend(title='Fraud Flag')
plt.show()

In [ ]:
# Box plot for 'amount'
plt.figure(figsize=(8, 5))
sns.boxplot(x='fraud_flag', y='amount', data=df)
plt.title('Transaction Amount Distribution by Fraud Flag')
plt.xlabel('Fraud Flag (0 = No Fraud, 1 = Fraud)')
plt.ylabel('Amount')
plt.ylim(0, df['amount'].quantile(0.99)) # Limit y-axis to focus on the main distribution
plt.show()

In [ ]:
# Box plot for 'origin_balance_before'
plt.figure(figsize=(8, 5))
sns.boxplot(x='fraud_flag', y='origin_balance_before', data=df)
plt.title('Origin Balance Before Transaction by Fraud Flag')
plt.xlabel('Fraud Flag (0 = No Fraud, 1 = Fraud)')
plt.ylabel('Origin Balance Before')
plt.ylim(df['origin_balance_before'].quantile(0.01), df['origin_balance_before'].quantile(0.99)) # Limit y-axis for better visualization
plt.show()

In [ ]:
# Box plot for 'origin_balance_after'
plt.figure(figsize=(8, 5))
sns.boxplot(x='fraud_flag', y='origin_balance_after', data=df)
plt.title('Origin Balance After Transaction by Fraud Flag')
plt.xlabel('Fraud Flag (0 = No Fraud, 1 = Fraud)')
plt.ylabel('Origin Balance After')
plt.ylim(df['origin_balance_after'].quantile(0.01), df['origin_balance_after'].quantile(0.99)) # Limit y-axis for better visualization
plt.show()

In [ ]:
# Box plot for 'destination_balance_before'
plt.figure(figsize=(8, 5))
sns.boxplot(x='fraud_flag', y='destination_balance_before', data=df)
plt.title('Destination Balance Before Transaction by Fraud Flag')
plt.xlabel('Fraud Flag (0 = No Fraud, 1 = Fraud)')
plt.ylabel('Destination Balance Before')
plt.ylim(df['destination_balance_before'].quantile(0.01), df['destination_balance_before'].quantile(0.99)) # Limit y-axis for better visualization
plt.show()

In [ ]:
# Box plot for 'destination_balance_after'
plt.figure(figsize=(8, 5))
sns.boxplot(x='fraud_flag', y='destination_balance_after', data=df)
plt.title('Destination Balance After Transaction by Fraud Flag')
plt.xlabel('Fraud Flag (0 = No Fraud, 1 = Fraud)')
plt.ylabel('Destination Balance After')
plt.ylim(df['destination_balance_after'].quantile(0.01), df['destination_balance_after'].quantile(0.99)) # Limit y-axis for better visualization
plt.show()

## Data Visualizations for `operation = 'op_03'` by Fraud Flag

As `op_03` was identified as a critical operation type for fraudulent transactions, let's re-examine the distributions of other variables, but only for transactions where the operation is `op_03`.

In [ ]:
# Filter the DataFrame for operation == 'op_03'
df_op03 = df[df['operation'] == 'op_03'].copy()

print(f"Filtered DataFrame shape (only op_03): {df_op03.shape}")

In [ ]:
# Box plot for 'period' filtered by op_03
plt.figure(figsize=(8, 5))
sns.boxplot(x='fraud_flag', y='period', data=df_op03)
plt.title('Period Distribution for op_03 Transactions by Fraud Flag')
plt.xlabel('Fraud Flag (0 = No Fraud, 1 = Fraud)')
plt.ylabel('Period')
plt.show()

In [ ]:
# Box plot for 'amount' filtered by op_03
plt.figure(figsize=(8, 5))
sns.boxplot(x='fraud_flag', y='amount', data=df_op03)
plt.title('Transaction Amount Distribution for op_03 Transactions by Fraud Flag')
plt.xlabel('Fraud Flag (0 = No Fraud, 1 = Fraud)')
plt.ylabel('Amount')
plt.ylim(0, df_op03['amount'].quantile(0.99)) # Limit y-axis to focus on the main distribution
plt.show()

In [ ]:
# Box plot for 'origin_balance_before' filtered by op_03
plt.figure(figsize=(8, 5))
sns.boxplot(x='fraud_flag', y='origin_balance_before', data=df_op03)
plt.title('Origin Balance Before Transaction for op_03 by Fraud Flag')
plt.xlabel('Fraud Flag (0 = No Fraud, 1 = Fraud)')
plt.ylabel('Origin Balance Before')
plt.ylim(df_op03['origin_balance_before'].quantile(0.01), df_op03['origin_balance_before'].quantile(0.99)) # Limit y-axis for better visualization
plt.show()

In [ ]:
# Box plot for 'origin_balance_after' filtered by op_03
plt.figure(figsize=(8, 5))
sns.boxplot(x='fraud_flag', y='origin_balance_after', data=df_op03)
plt.title('Origin Balance After Transaction for op_03 by Fraud Flag')
plt.xlabel('Fraud Flag (0 = No Fraud, 1 = Fraud)')
plt.ylabel('Origin Balance After')
plt.ylim(df_op03['origin_balance_after'].quantile(0.01), df_op03['origin_balance_after'].quantile(0.99)) # Limit y-axis for better visualization
plt.show()

In [ ]:
# Box plot for 'destination_balance_before' filtered by op_03
plt.figure(figsize=(8, 5))
sns.boxplot(x='fraud_flag', y='destination_balance_before', data=df_op03)
plt.title('Destination Balance Before Transaction for op_03 by Fraud Flag')
plt.xlabel('Fraud Flag (0 = No Fraud, 1 = Fraud)')
plt.ylabel('Destination Balance Before')
plt.ylim(df_op03['destination_balance_before'].quantile(0.01), df_op03['destination_balance_before'].quantile(0.99)) # Limit y-axis for better visualization
plt.show()

In [ ]:
# Box plot for 'destination_balance_after' filtered by op_03
plt.figure(figsize=(8, 5))
sns.boxplot(x='fraud_flag', y='destination_balance_after', data=df_op03)
plt.title('Destination Balance After Transaction for op_03 by Fraud Flag')
plt.xlabel('Fraud Flag (0 = No Fraud, 1 = Fraud)')
plt.ylabel('Destination Balance After')
plt.ylim(df_op03['destination_balance_after'].quantile(0.01), df_op03['destination_balance_after'].quantile(0.99)) # Limit y-axis for better visualization
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Distribution of fraud_flag based on 'origin_account_previously_fraud'
plt.figure(figsize=(10, 6))
sns.countplot(x='origin_account_previously_fraud', hue='fraud_flag', data=df, palette='pastel')
plt.title('Fraud Flag Distribution by Origin Account Previously Fraud Status')
plt.xlabel('Origin Account Previously Fraud (0 = No, 1 = Yes)')
plt.ylabel('Count')
plt.show()

# Distribution of fraud_flag based on 'destination_account_previously_fraud'
plt.figure(figsize=(10, 6))
sns.countplot(x='destination_account_previously_fraud', hue='fraud_flag', data=df, palette='pastel')
plt.title('Fraud Flag Distribution by Destination Account Previously Fraud Status')
plt.xlabel('Destination Account Previously Fraud (0 = No, 1 = Yes)')
plt.ylabel('Count')
plt.show()

# Combine the two new columns into a single categorical column for joint analysis
df['fraud_history_status'] = df['origin_account_previously_fraud'].astype(str) + df['destination_account_previously_fraud'].astype(str)

# Distribution of fraud_flag for combined fraud history status
plt.figure(figsize=(12, 7))
sns.countplot(x='fraud_history_status', hue='fraud_flag', data=df, palette='viridis')
plt.title('Fraud Flag Distribution by Combined Fraud History Status (Origin-Destination)')
plt.xlabel('Fraud History Status (Origin_Fraud_Prev | Destination_Fraud_Prev)')
plt.ylabel('Count')
plt.show()

print("Distribution analysis for new features complete. The 'fraud_history_status' column has been created for combined analysis.")

## EDA

In [ ]:
print(f"Total number of fraudulent transactions (fraud_flag = 1): {df['fraud_flag'].sum()}")
print(f"Number of unique transaction IDs with fraud_flag = 1: {df[df['fraud_flag'] == 1]['id'].nunique()}")

# Identify unique origin accounts involved in fraud
fraudulent_origin_accounts = df[df['fraud_flag'] == 1]['origin_account'].unique()
print(f"\nNumber of unique origin accounts involved in fraud: {len(fraudulent_origin_accounts)}")

# Identify unique destination accounts involved in fraud
fraudulent_destination_accounts = df[df['fraud_flag'] == 1]['destination_account'].unique()
print(f"Number of unique destination accounts involved in fraud: {len(fraudulent_destination_accounts)}")

# Check if fraudulent origin accounts also appear in non-fraudulent transactions
non_fraudulent_origin_accounts = df[df['fraud_flag'] == 0]['origin_account'].unique()
origin_fraud_and_non_fraud = set(fraudulent_origin_accounts).intersection(set(non_fraudulent_origin_accounts))
print(f"\nNumber of origin accounts involved in both fraudulent and non-fraudulent transactions: {len(origin_fraud_and_non_fraud)}")

# Check if fraudulent destination accounts also appear in non-fraudulent transactions
non_fraudulent_destination_accounts = df[df['fraud_flag'] == 0]['destination_account'].unique()
destination_fraud_and_non_fraud = set(fraudulent_destination_accounts).intersection(set(non_fraudulent_destination_accounts))
print(f"Number of destination accounts involved in both fraudulent and non-fraudulent transactions: {len(destination_fraud_and_non_fraud)}")

# Create a set of all accounts that have ever committed fraud (origin or destination)
all_fraudulent_accounts = set(fraudulent_origin_accounts).union(set(fraudulent_destination_accounts))

# Create new columns for 'origin_account_previously_fraud' and 'destination_account_previously_fraud'
df['origin_account_previously_fraud'] = df['origin_account'].isin(all_fraudulent_accounts).astype(int)
df['destination_account_previously_fraud'] = df['destination_account'].isin(all_fraudulent_accounts).astype(int)

print("\nNew columns 'origin_account_previously_fraud' and 'destination_account_previously_fraud' created.")
print("First 5 rows with new columns:")
print(df[['origin_account', 'destination_account', 'fraud_flag', 'origin_account_previously_fraud', 'destination_account_previously_fraud']].head())

les comptes frauduleux sont dans le dataset dans le groupe frauduleux et non frauduleux

In [ ]:
are_all_fraudulent_destination_also_origin = set(fraudulent_destination_accounts).issubset(set(fraudulent_origin_accounts))
print(f"Are all fraudulent destination accounts also fraudulent origin accounts? {are_all_fraudulent_destination_also_origin}")

In [ ]:
fraudulent_origin_and_destination_accounts = set(fraudulent_origin_accounts).intersection(set(fraudulent_destination_accounts))
print(f"Number of accounts that are both fraudulent origin and fraudulent destination accounts: {len(fraudulent_origin_and_destination_accounts)}")

# Check if these accounts act as both origin and destination
# (This is implicitly true if they are in both fraudulent_origin_accounts and fraudulent_destination_accounts, by definition of these sets)
# The question is more about their overall transaction behavior.

# Filter transactions where either the origin or destination account is in 'fraudulent_origin_and_destination_accounts'
df_dual_role_accounts_transactions = df[
    df['origin_account'].isin(fraudulent_origin_and_destination_accounts) |
    df['destination_account'].isin(fraudulent_origin_and_destination_accounts)
].copy()

print(f"\nTotal transactions involving accounts that are both fraudulent origin and destination: {len(df_dual_role_accounts_transactions)}")

# Check if these dual-role accounts send money (appear as origin) and receive money (appear as destination)
# We've already established they do by their inclusion in both `fraudulent_origin_accounts` and `fraudulent_destination_accounts`.
# Let's confirm their role in these specific transactions.
sends_money = df_dual_role_accounts_transactions['origin_account'].isin(fraudulent_origin_and_destination_accounts).any()
receives_money = df_dual_role_accounts_transactions['destination_account'].isin(fraudulent_origin_and_destination_accounts).any()
print(f"Do these dual-role accounts send money in these transactions? {sends_money}")
print(f"Do these dual-role accounts receive money in these transactions? {receives_money}")

# Calculate the proportion of fraudulent transactions among these 'dual-role' account transactions
if not df_dual_role_accounts_transactions.empty:
    fraud_proportion_dual_role = df_dual_role_accounts_transactions['fraud_flag'].mean()
    print(f"Proportion of fraudulent transfers involving these dual-role accounts: {fraud_proportion_dual_role:.4f}")
else:
    print("No transactions found for accounts that are both fraudulent origin and destination.")

In [ ]:
# First, identify all unique accounts present in the dataset
all_unique_accounts = pd.concat([df['origin_account'], df['destination_account']]).unique()

# All accounts that have *ever* been involved in fraud (from previous steps)
# fraudulent_origin_accounts and fraudulent_destination_accounts were defined in cell LWfX9gw_2ayC
all_fraudulent_accounts = set(fraudulent_origin_accounts).union(set(fraudulent_destination_accounts))

# Identify truly non-fraudulent accounts: those that are in all_unique_accounts but NOT in all_fraudulent_accounts
truly_non_fraudulent_accounts = set(all_unique_accounts) - all_fraudulent_accounts
print(f"Total number of truly non-fraudulent accounts (never involved in fraud): {len(truly_non_fraudulent_accounts)}")

# Now, let's see if these truly non-fraudulent accounts act as both origin and destination
non_fraudulent_origin_only = df[df['origin_account'].isin(truly_non_fraudulent_accounts)]['origin_account'].unique()
non_fraudulent_destination_only = df[df['destination_account'].isin(truly_non_fraudulent_accounts)]['destination_account'].unique()

non_fraudulent_origin_and_destination = set(non_fraudulent_origin_only).intersection(set(non_fraudulent_destination_only))
print(f"Number of truly non-fraudulent accounts that act as both origin and destination: {len(non_fraudulent_origin_and_destination)}")

# Analyze transactions involving these 'dual-role' non-fraudulent accounts
df_non_fraud_dual_role_transactions = df[
    (df['origin_account'].isin(non_fraudulent_origin_and_destination)) |
    (df['destination_account'].isin(non_fraudulent_origin_and_destination))
].copy()

print(f"\nTotal transactions involving accounts that are truly non-fraudulent and act as both origin and destination: {len(df_non_fraud_dual_role_transactions)}")

# Verify if they send and receive money within these transactions
sends_money_nf = df_non_fraud_dual_role_transactions['origin_account'].isin(non_fraudulent_origin_and_destination).any()
receives_money_nf = df_non_fraud_dual_role_transactions['destination_account'].isin(non_fraudulent_origin_and_destination).any()
print(f"Do these dual-role non-fraudulent accounts send money in these transactions? {sends_money_nf}")
print(f"Do these dual-role non-fraudulent accounts receive money in these transactions? {receives_money_nf}")

# Proportion of fraudulent transfers involving these truly non-fraudulent dual-role accounts
# By definition, if an account is 'truly non-fraudulent', it should never be associated with fraud_flag=1
if not df_non_fraud_dual_role_transactions.empty:
    fraud_proportion_non_fraud_dual_role = df_non_fraud_dual_role_transactions['fraud_flag'].mean()
    print(f"Proportion of fraudulent transfers involving these truly non-fraudulent dual-role accounts: {fraud_proportion_non_fraud_dual_role:.4f}")
else:
    print("No transactions found for these truly non-fraudulent dual-role accounts.")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Group by 'period' and sum 'fraud_flag' to get the number of frauds per period
fraud_by_period = df.groupby('period')['fraud_flag'].sum().reset_index()

plt.figure(figsize=(12, 6))
sns.lineplot(x='period', y='fraud_flag', data=fraud_by_period)
plt.title('Evolution du nombre de fraudes par période')
plt.xlabel('Période')
plt.ylabel('Nombre de fraudes')
plt.grid(True)
plt.show()

### Corrélation entre les comptes classés et la fraude

Nous allons calculer la corrélation de Pearson entre les colonnes `origin_account_ranked`, `destination_account_ranked` et `fraud_flag` pour voir s'il existe une relation linéaire.

### Taux de Fraude Moyen par Période

Nous allons calculer le taux de fraude moyen pour chaque période. Cela nous permettra de voir si le pourcentage de transactions frauduleuses varie significativement d'une période à l'autre, ce qui pourrait indiquer des moments où le système de détection de fraude a été contourné ou des changements dans les tactiques des fraudeurs.

In [ ]:
# Calculate the average fraud rate per period
fraud_rate_by_period = df.groupby('period')['fraud_flag'].mean().reset_index()

plt.figure(figsize=(12, 6))
sns.lineplot(x='period', y='fraud_flag', data=fraud_rate_by_period)
plt.title('Taux de Fraude Moyen par Période')
plt.xlabel('Période')
plt.ylabel('Taux de Fraude Moyen')
plt.grid(True)
plt.show()

In [ ]:
correlation_ranked_accounts = df[['origin_account_ranked', 'destination_account_ranked', 'fraud_flag']].corr()
print("Matrice de corrélation entre les comptes classés et la fraude :")
display(correlation_ranked_accounts)

### Visualisation de la distribution des comptes classés par statut de fraude

In [ ]:
plt.figure(figsize=(12, 6))
sns.histplot(data=df, x='origin_account_ranked', hue='fraud_flag', multiple='stack', bins=50, kde=True)
plt.title('Distribution des comptes émetteurs classés par statut de fraude')
plt.xlabel('Rang du compte émetteur')
plt.ylabel('Nombre de transactions')
plt.show()

In [ ]:
plt.figure(figsize=(12, 6))
sns.histplot(data=df, x='destination_account_ranked', hue='fraud_flag', multiple='stack', bins=50, kde=True)
plt.title('Distribution des comptes destinataires classés par statut de fraude')
plt.xlabel('Rang du compte destinataire')
plt.ylabel('Nombre de transactions')
plt.show()

# soumission

In [ ]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, precision_score, recall_score, accuracy_score, confusion_matrix, average_precision_score
import datetime
import matplotlib.pyplot as plt
import seaborn as sns

# --- 1. Reload the complete original training DataFrame and filter for op_03 ---
print("1. Reloading original training data and filtering for 'op_03' operations...")
df = pd.read_csv('/content/drive/MyDrive/CSV/train.csv')
df = df[df['operation'] == 'op_03'].copy() # Filter for 'op_03' operations only

# Ensure df is sorted by period for sequential features
df = df.sort_values(by='period').reset_index(drop=True)

# --- 2. Feature Engineering Pipeline (aligned with previous steps) ---
print("2. Applying feature engineering to 'op_03' training data...")

# 2.1 Sequential transaction features (from naZiIj_ryjNw)
df['origin_transaction_sequence'] = df.groupby('origin_account').cumcount() + 1
df['destination_transaction_sequence'] = df.groupby('destination_account').cumcount() + 1
df['origin_dest_pair_sequence'] = df.groupby(['origin_account', 'destination_account']).cumcount() + 1

# 2.2 Previously fraudulent flags (from cZwG6UBZMH-W, nkv6k2TX5OPZ)
fraudulent_origin_accounts = df[df['fraud_flag'] == 1]['origin_account'].unique()
fraudulent_destination_accounts = df[df['fraud_flag'] == 1]['destination_account'].unique()
all_fraudulent_accounts = set(fraudulent_origin_accounts).union(set(fraudulent_destination_accounts))
df['origin_account_previously_fraud'] = df['origin_account'].isin(all_fraudulent_accounts).astype(int)
df['destination_account_previously_fraud'] = df['destination_account'].isin(all_fraudulent_accounts).astype(int)

# The following feature engineering steps (account stripping/ranking, hex-related, decimal/digit, fraud pattern) were deemed less informative for the final models or were implicitly dropped in previous successful model trainings.
# Keeping only the features explicitly identified as most important for efficiency and resource saving.

print("Feature engineering on 'op_03' training data complete.")

# --- 3. Train sub-model (model_ever_fraud_dest) ---
print("3. Training sub-model to predict 'destination_account_previously_fraud' on 'op_03' data...")

Y_EVER_FRAUD_TARGET = 'destination_account_previously_fraud'
y_ever_fraud = df[Y_EVER_FRAUD_TARGET]

# Features for sub-model (reduced to the most informative ones from previous analysis)
features_for_ever_fraud_prediction_model = [
    'period', 'amount', 'origin_balance_before', 'origin_balance_after',
    'destination_balance_before', 'destination_balance_after',
    'origin_transaction_sequence', 'destination_transaction_sequence', 'origin_dest_pair_sequence'
]
features_for_ever_fraud_prediction_model = [f for f in features_for_ever_fraud_prediction_model if f in df.columns]
X_ever_fraud = df[features_for_ever_fraud_prediction_model]
X_ever_fraud = X_ever_fraud.fillna(0) # Impute NaNs consistently

X_train_ef, X_test_ef, y_train_ef, y_test_ef = train_test_split(
    X_ever_fraud, y_ever_fraud, test_size=0.3, random_state=42, stratify=y_ever_fraud
)
neg_count_ef = y_train_ef.value_counts()[0]
pos_count_ef = y_train_ef.value_counts()[1]
scale_pos_weight_ef = neg_count_ef / pos_count_ef

model_ever_fraud_dest = xgb.XGBClassifier(
    objective='binary:logistic',
    eval_metric='aucpr',
    random_state=42,
    scale_pos_weight=scale_pos_weight_ef
)
model_ever_fraud_dest.fit(X_train_ef, y_train_ef)

# Generate 'predicted_dest_ever_fraud_proba' for the entire df
df['predicted_dest_ever_fraud_proba'] = model_ever_fraud_dest.predict_proba(X_ever_fraud)[:, 1]

print("Sub-model training complete and 'predicted_dest_ever_fraud_proba' added to training data.")

# Sub-model evaluation and Confusion Matrix
y_pred_ef = model_ever_fraud_dest.predict(X_test_ef)
y_pred_proba_ef = model_ever_fraud_dest.predict_proba(X_test_ef)[:, 1]

accuracy_ef = accuracy_score(y_test_ef, y_pred_ef)
precision_ef = precision_score(y_test_ef, y_pred_ef)
recall_ef = recall_score(y_test_ef, y_pred_ef)
roc_auc_ef = roc_auc_score(y_test_ef, y_pred_proba_ef)
average_precision_ef = average_precision_score(y_test_ef, y_pred_proba_ef)

print(f"\n--- Sub-Model Performance (predicting destination_account_previously_fraud) ---")
print(f"Accuracy: {accuracy_ef:.4f}")
print(f"Precision: {precision_ef:.4f}")
print(f"Recall: {recall_ef:.4f}")
print(f"ROC AUC: {roc_auc_ef:.4f}")
print(f"Average Precision (PR-AUC): {average_precision_ef:.4f}")

cm_ef = confusion_matrix(y_test_ef, y_pred_ef)
plt.figure(figsize=(8, 6))
sns.heatmap(cm_ef, annot=True, fmt='d', cmap='Blues', cbar=False,
            xticklabels=['Predicted Not Ever Fraud', 'Predicted Ever Fraud'],
            yticklabels=['Actual Not Ever Fraud', 'Actual Ever Fraud'])
plt.title('Confusion Matrix for Sub-Model (Ever Fraudulent Destination Account)')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()

# --- 4. Train main fraud detection model (model_main_fraud) ---
print("4. Training main fraud detection model on 'op_03' data...")

# Filter df for training the main model (using censoring strategy from sp-hqzMQq13Z)

all_unique_accounts_set = set(pd.concat([df['origin_account'], df['destination_account']]).unique())
all_fraudulent_accounts_set = set(fraudulent_origin_accounts).union(set(fraudulent_destination_accounts))
truly_non_fraudulent_accounts_set = all_unique_accounts_set - all_fraudulent_accounts_set

first_appearance_all = df.groupby('destination_account')['period'].min().reset_index().rename(columns={'period': 'first_appearance_period'})
last_appearance_all = df.groupby('destination_account')['period'].max().reset_index().rename(columns={'period': 'last_appearance_period'})
account_lifetimes = pd.merge(first_appearance_all, last_appearance_all, on='destination_account', how='left')
account_lifetimes['account_lifetime'] = account_lifetimes['last_appearance_period'] - account_lifetimes['first_appearance_period']
truly_non_fraud_dest_lifetimes = account_lifetimes[account_lifetimes['destination_account'].isin(truly_non_fraudulent_accounts_set)].copy()

# This value was derived from destination_fraud_metrics in previous turns. Setting a sensible default or re-deriving.
# For simplicity and to follow the 'less features' directive, I'll use a fixed value or try to infer it if possible.
# From the kernel state, mean_time_to_fraud_dest was 2.1948, so long_time_threshold_periods was 6.
long_time_threshold_periods = 6 # Derived from (2.1948 * 3) rounded up.

stable_long_time_accounts_set = set(truly_non_fraud_dest_lifetimes[truly_non_fraud_dest_lifetimes['account_lifetime'] >= long_time_threshold_periods]['destination_account'].tolist())

df_filtered_for_training = df[
    (df['fraud_flag'] == 1) |
    ((df['fraud_flag'] == 0) & (df['destination_account'].isin(stable_long_time_accounts_set)))
].copy()

y_main = df_filtered_for_training['fraud_flag']

# Main model features (reduced to the most informative ones and including the sub-model's prediction)
main_model_features = [
    'period', 'amount', 'origin_balance_before', 'origin_balance_after',
    'destination_balance_before', 'destination_balance_after',
    'origin_transaction_sequence', 'destination_transaction_sequence', 'origin_dest_pair_sequence',
    'predicted_dest_ever_fraud_proba' # This is the crucial feature from the sub-model
]
main_model_features = [f for f in main_model_features if f in df_filtered_for_training.columns]
X_main = df_filtered_for_training[main_model_features]
X_main = X_main.fillna(0) # Impute NaNs consistently

X_train_main, X_test_main, y_train_main, y_test_main = train_test_split(
    X_main, y_main, test_size=0.3, random_state=42, stratify=y_main
)
neg_count_main = y_train_main.value_counts()[0]
pos_count_main = y_train_main.value_counts()[1]
scale_pos_weight_main = neg_count_main / pos_count_main

model_main_fraud = xgb.XGBClassifier(
    objective='binary:logistic',
    eval_metric='aucpr',
    random_state=42,
    scale_pos_weight=scale_pos_weight_main
)
model_main_fraud.fit(X_train_main, y_train_main)

print("Main model training complete.")

# Main model evaluation and Confusion Matrix
y_pred_main = model_main_fraud.predict(X_test_main)
y_pred_proba_main = model_main_fraud.predict_proba(X_test_main)[:, 1]

accuracy_main = accuracy_score(y_test_main, y_pred_main)
precision_main = precision_score(y_test_main, y_pred_main)
recall_main = recall_score(y_test_main, y_pred_main)
roc_auc_main = roc_auc_score(y_test_main, y_pred_proba_main)
average_precision_main = average_precision_score(y_test_main, y_pred_proba_main)

print(f"\n--- Main Model Performance (predicting fraud_flag) ---")
print(f"Accuracy: {accuracy_main:.4f}")
print(f"Precision: {precision_main:.4f}")
print(f"Recall: {recall_main:.4f}")
print(f"ROC AUC: {roc_auc_main:.4f}")
print(f"Average Precision (PR-AUC): {average_precision_main:.4f}")

cm_main = confusion_matrix(y_test_main, y_pred_main)
plt.figure(figsize=(8, 6))
sns.heatmap(cm_main, annot=True, fmt='d', cmap='Blues', cbar=False,
            xticklabels=['Predicted Non-Fraud (0)', 'Predicted Fraud (1)'],
            yticklabels=['Actual Non-Fraud (0)', 'Actual Fraud (1)'])
plt.title('Confusion Matrix for Main Model (Fraud Flag)')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()


# --- 5. Process Test Data and Generate Predictions ---
print("5. Processing test data and generating predictions...")

test_df = pd.read_csv('/content/drive/MyDrive/CSV/test.csv')
test_df = test_df[test_df['operation'] == 'op_03'].copy() # Filter test data for 'op_03' only
submission_df = test_df[['id']].copy()

# Apply the same feature engineering steps to the test_df
# Ensure test_df is sorted by period
test_df = test_df.sort_values(by='period').reset_index(drop=True)

# Sequential transaction features
test_df['origin_transaction_sequence'] = test_df.groupby('origin_account').cumcount() + 1
test_df['destination_transaction_sequence'] = test_df.groupby('destination_account').cumcount() + 1
test_df['origin_dest_pair_sequence'] = test_df.groupby(['origin_account', 'destination_account']).cumcount() + 1

# Previously fraudulent flags (using info from training data)
test_df['origin_account_previously_fraud'] = test_df['origin_account'].isin(all_fraudulent_accounts).astype(int)
test_df['destination_account_previously_fraud'] = test_df['destination_account'].isin(all_fraudulent_accounts).astype(int)

# The following feature engineering steps (account stripping/ranking, hex-related, decimal/digit, fraud pattern) were removed for efficiency.
# This means features like 'origin_account_ranked', 'destination_account_ranked', 'calc_origin_decimal', etc.,
# as well as fraud pattern features (`time_to_first_fraud`, `mean_non_fraud_duration` etc.) will not be generated for test_df.

# Predict 'predicted_dest_ever_fraud_proba' for test_df
X_test_ever_fraud_predict = test_df[features_for_ever_fraud_prediction_model].fillna(0)
test_df['predicted_dest_ever_fraud_proba'] = model_ever_fraud_dest.predict_proba(X_test_ever_fraud_predict)[:, 1]

# Select features for the main model prediction
X_test_main_predict = test_df[main_model_features].fillna(0)

# Generate final fraud probabilities
submission_df['target'] = model_main_fraud.predict_proba(X_test_main_predict)[:, 1]

# --- 6. Save submission file ---
submission_df.to_csv('submission.csv', index=False)

print("Full pipeline executed on 'op_03' data. Submission file 'submission.csv' generated.")
print("First 5 rows of submission.csv:")
print(submission_df.head())


In [ ]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, precision_score, recall_score, accuracy_score, confusion_matrix, average_precision_score
import datetime
import matplotlib.pyplot as plt
import seaborn as sns

# --- 1. Reload the complete original training DataFrame and filter for op_03 (for model training) ---
print("1. Reloading original training data and filtering for 'op_03' operations for model training...")
df_train_full = pd.read_csv('/content/drive/MyDrive/CSV/train.csv')
df_op03_train = df_train_full[df_train_full['operation'] == 'op_03'].copy()

# Ensure df_op03_train is sorted by period for sequential features
df_op03_train = df_op03_train.sort_values(by='period').reset_index(drop=True)

# --- 2. Feature Engineering Pipeline for op_03 training data ---
print("2. Applying feature engineering to 'op_03' training data...")

# 2.1 Sequential transaction features
df_op03_train['origin_transaction_sequence'] = df_op03_train.groupby('origin_account').cumcount() + 1
df_op03_train['destination_transaction_sequence'] = df_op03_train.groupby('destination_account').cumcount() + 1
df_op03_train['origin_dest_pair_sequence'] = df_op03_train.groupby(['origin_account', 'destination_account']).cumcount() + 1

# 2.2 Previously fraudulent flags
# These need to be calculated based on the *full* original training data to reflect true 'ever fraudulent' status
fraudulent_origin_accounts_full = df_train_full[df_train_full['fraud_flag'] == 1]['origin_account'].unique()
fraudulent_destination_accounts_full = df_train_full[df_train_full['fraud_flag'] == 1]['destination_account'].unique()
all_fraudulent_accounts_full = set(fraudulent_origin_accounts_full).union(set(fraudulent_destination_accounts_full))
df_op03_train['origin_account_previously_fraud'] = df_op03_train['origin_account'].isin(all_fraudulent_accounts_full).astype(int)
df_op03_train['destination_account_previously_fraud'] = df_op03_train['destination_account'].isin(all_fraudulent_accounts_full).astype(int)

print("Feature engineering on 'op_03' training data complete.")

# --- 3. Train sub-model (model_ever_fraud_dest) ---
print("3. Training sub-model to predict 'destination_account_previously_fraud' on 'op_03' training data...")

Y_EVER_FRAUD_TARGET = 'destination_account_previously_fraud'
y_ever_fraud = df_op03_train[Y_EVER_FRAUD_TARGET]

# Features for sub-model (reduced to the most informative ones from previous analysis)
features_for_ever_fraud_prediction_model = [
    'period', 'amount', 'origin_balance_before', 'origin_balance_after',
    'destination_balance_before', 'destination_balance_after',
    'origin_transaction_sequence', 'destination_transaction_sequence', 'origin_dest_pair_sequence'
]
features_for_ever_fraud_prediction_model = [f for f in features_for_ever_fraud_prediction_model if f in df_op03_train.columns]
X_ever_fraud = df_op03_train[features_for_ever_fraud_prediction_model]
X_ever_fraud = X_ever_fraud.fillna(0) # Impute NaNs consistently

X_train_ef, X_test_ef, y_train_ef, y_test_ef = train_test_split(
    X_ever_fraud, y_ever_fraud, test_size=0.3, random_state=42, stratify=y_ever_fraud
)
neg_count_ef = y_train_ef.value_counts()[0]
pos_count_ef = y_train_ef.value_counts()[1]
scale_pos_weight_ef = neg_count_ef / pos_count_ef

model_ever_fraud_dest = xgb.XGBClassifier(
    objective='binary:logistic',
    eval_metric='aucpr',
    random_state=42,
    scale_pos_weight=scale_pos_weight_ef
)
model_ever_fraud_dest.fit(X_train_ef, y_train_ef)

# Generate 'predicted_dest_ever_fraud_proba' for the entire df_op03_train
df_op03_train['predicted_dest_ever_fraud_proba'] = model_ever_fraud_dest.predict_proba(X_ever_fraud)[:, 1]

print("Sub-model training complete and 'predicted_dest_ever_fraud_proba' added to training data.")

# Sub-model evaluation and Confusion Matrix
y_pred_ef = model_ever_fraud_dest.predict(X_test_ef)
y_pred_proba_ef = model_ever_fraud_dest.predict_proba(X_test_ef)[:, 1]

accuracy_ef = accuracy_score(y_test_ef, y_pred_ef)
precision_ef = precision_score(y_test_ef, y_pred_ef)
recall_ef = recall_score(y_test_ef, y_pred_ef)
roc_auc_ef = roc_auc_score(y_test_ef, y_pred_proba_ef)
average_precision_ef = average_precision_score(y_test_ef, y_pred_proba_ef)

print(f"\n--- Sub-Model Performance (predicting destination_account_previously_fraud) ---")
print(f"Accuracy: {accuracy_ef:.4f}")
print(f"Precision: {precision_ef:.4f}")
print(f"Recall: {recall_ef:.4f}")
print(f"ROC AUC: {roc_auc_ef:.4f}")
print(f"Average Precision (PR-AUC): {average_precision_ef:.4f}")

cm_ef = confusion_matrix(y_test_ef, y_pred_ef)
plt.figure(figsize=(8, 6))
sns.heatmap(cm_ef, annot=True, fmt='d', cmap='Blues', cbar=False,
            xticklabels=['Predicted Not Ever Fraud', 'Predicted Ever Fraud'],
            yticklabels=['Actual Not Ever Fraud', 'Actual Ever Fraud'])
plt.title('Confusion Matrix for Sub-Model (Ever Fraudulent Destination Account)')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()

# --- 4. Train main fraud detection model (model_main_fraud) ---
print("4. Training main fraud detection model on 'op_03' training data...")

# Filter df_op03_train for main model training (using censoring strategy)
# Ensure all_unique_accounts_full is derived from the *full* training dataset for generalizability
all_unique_accounts_full_set = set(pd.concat([df_train_full['origin_account'], df_train_full['destination_account']]).unique())
# all_fraudulent_accounts_full already defined above
truly_non_fraudulent_accounts_full_set = all_unique_accounts_full_set - all_fraudulent_accounts_full

first_appearance_all = df_train_full.groupby('destination_account')['period'].min().reset_index().rename(columns={'period': 'first_appearance_period'})
last_appearance_all = df_train_full.groupby('destination_account')['period'].max().reset_index().rename(columns={'period': 'last_appearance_period'})
account_lifetimes = pd.merge(first_appearance_all, last_appearance_all, on='destination_account', how='left')
account_lifetimes['account_lifetime'] = account_lifetimes['last_appearance_period'] - account_lifetimes['first_appearance_period']
truly_non_fraud_dest_lifetimes = account_lifetimes[account_lifetimes['destination_account'].isin(truly_non_fraudulent_accounts_full_set)].copy()

long_time_threshold_periods = 6 # Derived from mean_time_to_fraud_dest * 3 rounded up, as used previously.

stable_long_time_accounts_set = set(truly_non_fraud_dest_lifetimes[truly_non_fraud_dest_lifetimes['account_lifetime'] >= long_time_threshold_periods]['destination_account'].tolist())

df_filtered_for_training_main = df_op03_train[
    (df_op03_train['fraud_flag'] == 1) |
    ((df_op03_train['fraud_flag'] == 0) & (df_op03_train['destination_account'].isin(stable_long_time_accounts_set)))
].copy()

y_main = df_filtered_for_training_main['fraud_flag']

# Main model features (reduced to the most informative ones and including the sub-model's prediction)
main_model_features = [
    'period', 'amount', 'origin_balance_before', 'origin_balance_after',
    'destination_balance_before', 'destination_balance_after',
    'origin_transaction_sequence', 'destination_transaction_sequence', 'origin_dest_pair_sequence',
    'predicted_dest_ever_fraud_proba' # This is the crucial feature from the sub-model
]
main_model_features = [f for f in main_model_features if f in df_filtered_for_training_main.columns]
X_main = df_filtered_for_training_main[main_model_features]
X_main = X_main.fillna(0) # Impute NaNs consistently

X_train_main, X_test_main, y_train_main, y_test_main = train_test_split(
    X_main, y_main, test_size=0.3, random_state=42, stratify=y_main
)
neg_count_main = y_train_main.value_counts()[0]
pos_count_main = y_train_main.value_counts()[1]
scale_pos_weight_main = neg_count_main / pos_count_main

model_main_fraud = xgb.XGBClassifier(
    objective='binary:logistic',
    eval_metric='aucpr',
    random_state=42,
    scale_pos_weight=scale_pos_weight_main
)
model_main_fraud.fit(X_train_main, y_train_main)

print("Main model training complete.")

# Main model evaluation and Confusion Matrix
y_pred_main = model_main_fraud.predict(X_test_main)
y_pred_proba_main = model_main_fraud.predict_proba(X_test_main)[:, 1]

accuracy_main = accuracy_score(y_test_main, y_pred_main)
precision_main = precision_score(y_test_main, y_pred_main)
recall_main = recall_score(y_test_main, y_pred_main)
roc_auc_main = roc_auc_score(y_test_main, y_pred_proba_main)
average_precision_main = average_precision_score(y_test_main, y_pred_proba_main)

print(f"\n--- Main Model Performance (predicting fraud_flag) ---")
print(f"Accuracy: {accuracy_main:.4f}")
print(f"Precision: {precision_main:.4f}")
print(f"Recall: {recall_main:.4f}")
print(f"ROC AUC: {roc_auc_main:.4f}")
print(f"Average Precision (PR-AUC): {average_precision_main:.4f}")

cm_main = confusion_matrix(y_test_main, y_pred_main)
plt.figure(figsize=(8, 6))
sns.heatmap(cm_main, annot=True, fmt='d', cmap='Blues', cbar=False,
            xticklabels=['Predicted Non-Fraud (0)', 'Predicted Fraud (1)'],
            yticklabels=['Actual Non-Fraud (0)', 'Actual Fraud (1)'])
plt.title('Confusion Matrix for Main Model (Fraud Flag)')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()


# --- 5. Process Test Data and Generate Predictions for 'submission2.csv' ---
print("5. Processing test data and generating predictions for 'submission2.csv'...")

test_df_full = pd.read_csv('/content/drive/MyDrive/CSV/test.csv')
submission_df_2 = test_df_full[['id']].copy()

# Initialize all targets to 0.0 (for non-op_03 transactions)
submission_df_2['target'] = 0.0

# Separate op_03 transactions for full pipeline processing
test_df_op03 = test_df_full[test_df_full['operation'] == 'op_03'].copy()

# Apply the same feature engineering steps to the test_df_op03
test_df_op03 = test_df_op03.sort_values(by='period').reset_index(drop=True)

# Sequential transaction features
test_df_op03['origin_transaction_sequence'] = test_df_op03.groupby('origin_account').cumcount() + 1
test_df_op03['destination_transaction_sequence'] = test_df_op03.groupby('destination_account').cumcount() + 1
test_df_op03['origin_dest_pair_sequence'] = test_df_op03.groupby(['origin_account', 'destination_account']).cumcount() + 1

# Previously fraudulent flags (using info from training data, all_fraudulent_accounts_full)
test_df_op03['origin_account_previously_fraud'] = test_df_op03['origin_account'].isin(all_fraudulent_accounts_full).astype(int)
test_df_op03['destination_account_previously_fraud'] = test_df_op03['destination_account'].isin(all_fraudulent_accounts_full).astype(int)

# Predict 'predicted_dest_ever_fraud_proba' for test_df_op03
X_test_ever_fraud_predict_op03 = test_df_op03[features_for_ever_fraud_prediction_model].fillna(0)
test_df_op03['predicted_dest_ever_fraud_proba'] = model_ever_fraud_dest.predict_proba(X_test_ever_fraud_predict_op03)[:, 1]

# Select features for the main model prediction
X_test_main_predict_op03 = test_df_op03[main_model_features].fillna(0)

# Generate final fraud probabilities for op_03 transactions
final_probabilities_op03 = model_main_fraud.predict_proba(X_test_main_predict_op03)[:, 1]

# Update the 'target' column in submission_df_2 for op_03 transactions
submission_df_2.loc[submission_df_2['id'].isin(test_df_op03['id']), 'target'] = final_probabilities_op03

# --- 6. Save submission file ---
submission_df_2.to_csv('submission2.csv', index=False)

print("Full pipeline executed with custom non-op_03 handling. Submission file 'submission2.csv' generated.")
print("First 5 rows of submission2.csv:")
print(submission_df_2.head())